# TRACE — Trajectory-Referenced Audit of Cycle-level Evidence
**Reproducibility notebook for the MethodsX article.** Running all cells regenerates every number, table and figure of the Method details and Method validation chapters (Figs. 1–10, Tables 2–5, graphical abstract) and the Supplementary material (Tables S1–S4, Figs. S1–S4), plus `outputs/results_numbers.md`, which lists every quoted number with its source file.

* **Data 1 – GMP LiFePO₄ dataset** (Mendeley Data, doi:10.17632/k675pxrd83.2). Upload `GMP_LFP_40Ah_3000Cycle.csv` in Step 1 to audit the published file. If you skip the upload, the file is regenerated exactly from the published generating equations (Data in Brief 69 (2026) 113252, seed 42).
* **Data 2 – NASA PCoE cells B0005/6/7/18**, downloaded automatically.
* **Data 3 – 374 public cells (XJTU, TJU, HUST, MIT)**, cloned automatically from github.com/wang-fujin/PINN4SOH at commit cc3cc50.
* Set `QUICK = True` for a ~5-minute smoke test with fewer Monte-Carlo replicates. Leave it `False` to reproduce the paper exactly (~60–90 min on a free Colab CPU).

In [ ]:
QUICK = False   # True = fast smoke test; False = exact paper numbers
import os; os.environ['TRACE_QUICK'] = '1' if QUICK else '0'
!pip -q install pyreadr==0.5.9 pymannkendall==1.4.3 python-docx tabulate shap statsmodels scikit-learn
!mkdir -p data outputs figures

## Step 1 – (optional) upload the published GMP CSV

In [ ]:
try:
    from google.colab import files
    up = files.upload()            # choose GMP_LFP_40Ah_3000Cycle.csv, or press Cancel to regenerate it
    for name in up:
        os.replace(name, 'data/GMP.csv'); print('saved data/GMP.csv')
except Exception as e:
    print('No upload (', e, ') - the dataset will be regenerated from the published equations.')

## Step 2 – the TRACE method (single module)

In [ ]:
%%writefile trace_audit.py
"""
TRACE: Trajectory-Referenced Audit of Cycle-level Evidence
A pre-modelling information audit for cycle-indexed battery prognostics datasets.

Stages
  S1  Provenance / identity screen  -> determinism index eta, exact identities
  S2  Index-surrogate ceiling        -> skill attainable from the cycle index alone
  S3  Association tests (time-shift) -> marginal and index-conditional, Holm-corrected
  S4  Sensitivity calibration        -> injection power curves, minimum detectable effect

Dependencies: numpy, scipy, pandas (scikit-learn optional, not required).
"""
from __future__ import annotations
import numpy as np
import pandas as pd
from scipy import stats

from scipy.optimize import curve_fit

# ----------------------------------------------------------------------------
# helpers
# ----------------------------------------------------------------------------

def _z(a):
    a = np.asarray(a, float)
    s = a.std()
    return (a - a.mean()) / s if s > 0 else a * 0.0


_PROJ_CACHE = {}


def _bspline_basis(k, n_knots):
    from scipy.interpolate import BSpline
    deg = 3
    lo, hi = k.min(), k.max()
    inner = np.quantile(k, np.linspace(0, 1, n_knots + 2)[1:-1])
    t = np.r_[[lo] * (deg + 1), inner, [hi] * (deg + 1)]
    return BSpline.design_matrix(np.clip(k, lo, hi), t, deg).toarray()


def _crossfit_operator(k, n_knots):
    """Linear operator M such that M @ x is the two-fold interleaved cross-fitted
    cubic regression-spline trend of x on k (cached per index vector)."""
    key = (len(k), float(k[0]), float(k[-1]), n_knots, float(np.sum(k)))
    if key in _PROJ_CACHE:
        return _PROJ_CACHE[key]
    B = _bspline_basis(k, n_knots)
    n = len(k); idx = np.arange(n); M = np.zeros((n, n))
    for part in (0, 1):
        tr = idx % 2 != part; te = ~tr
        P = np.linalg.pinv(B[tr])            # least-squares coefficients map
        M[np.ix_(te, tr)] = B[te] @ P
    _PROJ_CACHE[key] = M
    return M


def crossfit_trend(k, x, n_knots=None):
    """Two-fold interleaved cross-fitted cubic regression spline of x on index k.
    Points at even positions are predicted from a spline fitted to odd positions and
    vice versa, so the trend cannot absorb observation noise.
    Default: min(12, n//15) interior quantile knots. Returns (trend, residual)."""
    k = np.asarray(k, float); x = np.asarray(x, float)
    if n_knots is None:
        n_knots = int(max(3, min(12, len(k) // 15)))
    M = _crossfit_operator(k, n_knots)
    fit = M @ x
    return fit, x - fit


# ----------------------------------------------------------------------------
# S1  provenance / identity screen
# ----------------------------------------------------------------------------

def determinism_index(k, x):
    """eta = RMS(cross-fitted residual) / SD(x).
    eta ~ 1: no smooth dependence on the index (stationary column);
    eta << 1 (< 1e-3): column is a deterministic smooth function of the index."""
    x = np.asarray(x, float)
    sd = x.std()
    if sd == 0:
        return np.nan
    _, r = crossfit_trend(k, x)
    return float(np.sqrt(np.mean(r ** 2)) / sd)


def identity_screen(df: pd.DataFrame, index_col: str, tol=1e-9):
    """Flag constant columns, columns affine in another column (|r| >= 1-tol),
    and columns strictly monotone in the index."""
    out = []
    cols = [c for c in df.columns]
    k = df[index_col].values
    for c in cols:
        x = df[c].values.astype(float)
        rec = dict(column=c)
        rec['constant'] = bool(np.nanstd(x) == 0)
        if rec['constant']:
            rec.update(affine_to='', strictly_monotone_in_index=False,
                       spearman_index=np.nan, eta=np.nan)
            out.append(rec); continue
        partners = []
        for c2 in cols:
            if c2 == c or np.std(df[c2].values) == 0:
                continue
            r = np.corrcoef(x, df[c2].values.astype(float))[0, 1]
            if abs(r) >= 1 - tol:
                partners.append(c2)
        rec['affine_to'] = ';'.join(partners)
        dx = np.diff(x[np.argsort(k)])
        rec['strictly_monotone_in_index'] = bool(np.all(dx > 0) or np.all(dx < 0))
        rec['spearman_index'] = float(stats.spearmanr(k, x)[0])
        rec['eta'] = determinism_index(k, x) if c != index_col else 0.0
        out.append(rec)
    return pd.DataFrame(out)


def trend_r2(k, x):
    """Cross-fitted trend R^2 = 1 - eta^2 (can be slightly negative under no trend)."""
    e = determinism_index(k, x)
    return np.nan if np.isnan(e) else 1 - e ** 2


def _lb_q(R, lags):
    """Ljung-Box Q for each column of R."""
    R = R - R.mean(0); n = R.shape[0]; d = np.sum(R ** 2, 0)
    return n * (n + 2) * sum(np.sum(R[h:] * R[:-h], 0) ** 2 / d ** 2 / (n - h) for h in range(1, lags + 1))


def lb_gate(k, x, lags=10, B=999, seed=0):
    """Exchangeability gate: Ljung-Box statistic of the spline-detrended residual, with its null
    distribution obtained by passing i.i.d. N(0,1) series through the same detrending projection.
    The asymptotic chi-square reference ignores the degrees of freedom removed by the spline and
    rejects white noise far too often on short records (34% at n = 168); this calibration restores
    the nominal size under i.i.d. errors. Returns the Monte Carlo p-value."""
    k = np.asarray(k, float); x = np.asarray(x, float)
    Bf = _bspline_basis(k, int(max(3, min(12, len(k) // 15))))
    Q, _ = np.linalg.qr(Bf)
    proj = lambda V: V - Q @ (Q.T @ V)
    q0 = _lb_q(proj(x[:, None]), lags)[0]
    Z = np.random.default_rng(seed + 7919).normal(size=(len(k), B))
    qn = _lb_q(proj(Z), lags)
    return float((1 + np.sum(qn >= q0)) / (B + 1))


def trend_test(k, x, B=999, seed=0, lb_lags=10):
    """S1b: permutation test of H0 'no dependence of x on the cycle index'.
    Statistic: cross-fitted trend R^2. Null: random permutations of x (valid when
    the observations are exchangeable). Exchangeability is checked by a Ljung-Box
    test on the detrended residual; if it fails, the result is descriptive only."""
    k = np.asarray(k, float); x = np.asarray(x, float)
    M = _crossfit_operator(k, int(max(3, min(12, len(k) // 15))))
    def stat(v):
        r = v - M @ v
        return 1 - np.mean(r ** 2) / np.var(v)
    s0 = stat(x)
    rng = np.random.default_rng(seed)
    P = np.stack([rng.permutation(x) for _ in range(B)], 1)
    R = P - M @ P
    null = 1 - np.mean(R ** 2, 0) / np.var(P, 0)
    p = (1 + np.sum(null >= s0)) / (B + 1)
    # exchangeability gate on the full-sample detrended residual (interleaved cross-fitting itself
    # induces lag-1 structure), calibrated by Monte Carlo for the spline projection (lb_gate)
    lb_p = lb_gate(k, x, lags=lb_lags, seed=seed)
    return dict(R2_trend=float(s0), p_trend=float(p), null_q95=float(np.quantile(null, .95)),
                null_q99=float(np.quantile(null, .99)), LB_p=lb_p, exchangeable=lb_p > 0.05)


# ----------------------------------------------------------------------------
# S2  index-surrogate ceiling
# ----------------------------------------------------------------------------

def _powerlaw(k, a, b, c):
    return a - b * np.power(k, c)


def index_ceiling(k, y, train_frac=0.7):
    """Chronological split. Fits y = a - b*k^c on the first train_frac of cycles
    using the index only and extrapolates to the rest. Returns test R2, RMSE.
    High R2 means the target is (near-)fully determined by the index: any
    feature-based model must be judged against this ceiling."""
    k = np.asarray(k, float); y = np.asarray(y, float)
    n = len(k); m = int(round(train_frac * n))
    kt, yt = k[:m], y[:m]
    try:
        p0 = (yt[0], (yt[0] - yt[-1]) / max(kt[-1] ** 1.0, 1), 1.0)
        p, _ = curve_fit(_powerlaw, kt, yt, p0=p0, maxfev=20000,
                         bounds=([-np.inf, -np.inf, 0.2], [np.inf, np.inf, 4.0]))
        yhat = _powerlaw(k[m:], *p)
    except Exception:
        coef = np.polyfit(kt, yt, 2)
        yhat = np.polyval(coef, k[m:])
    ytest = y[m:]
    ss_res = np.sum((ytest - yhat) ** 2)
    ss_tot = np.sum((ytest - ytest.mean()) ** 2)
    return dict(R2_index=float(1 - ss_res / ss_tot) if ss_tot > 0 else np.nan,
                RMSE_index=float(np.sqrt(np.mean((ytest - yhat) ** 2))))


# ----------------------------------------------------------------------------
# S3  association tests: conservative truncated time-shift test
# ----------------------------------------------------------------------------

def _absspearman(a, b):
    return abs(stats.spearmanr(a, b)[0])


def _shift_corr(x, y, r):
    """|Pearson r| between x[r+d : n-r+d] and y[r : n-r] for every d in [-r, r],
    computed in O(n log n) with FFT cross-correlation and cumulative sums."""
    from scipy.signal import correlate
    n = len(x); L = n - 2 * r
    yw = y[r:n - r]
    yc = yw - yw.mean(); syy = np.sum(yc ** 2)
    sxy = correlate(x, yc, mode='valid', method='fft')        # length 2r+1
    cs = np.concatenate([[0.0], np.cumsum(x)])
    cs2 = np.concatenate([[0.0], np.cumsum(x * x)])
    sx = cs[L:] - cs[:-L]; sxx = cs2[L:] - cs2[:-L]
    vx = sxx - sx ** 2 / L
    with np.errstate(invalid='ignore', divide='ignore'):
        S = np.abs(sxy) / np.sqrt(vx * syy)
    return np.nan_to_num(S)


TTS_DIFF_DEFAULT = True   # TRACE v2: level + rate representation of the target residual


def tts_test(x, y, r=None, rank=True, lags=0, diff=None):
    """Truncated time-shift test (Harris 2020; Yuan & Shou 2024), conservative.
    x is the series assumed stationary; y is held fixed. For each shift d in [-r, r] the
    statistic is computed between x[R+d : n-R+d] and the fixed window y[R : n-R], R = r + lags.
    Basic statistic: |rank correlation| at lag 0 (lags=0, diff=False; TRACE v1).
    TRACE v2 statistic: the maximum |rank correlation| over lags l in [-lags, lags] and over two
    representations of the target, its level residual and its first difference (rate). The same
    statistic is computed at every shift, so the test remains valid for any stationary x.
    p = #{d : S_d >= S_0} / (2r+1); minimum attainable p = 1/(2r+1). Returns (S0, p)."""
    if diff is None:
        diff = TTS_DIFF_DEFAULT
    x = np.asarray(x, float); y = np.asarray(y, float)
    n = len(x)
    if r is None:
        r = n // 4
    R = r + lags
    targets = [y] + ([np.r_[0.0, np.diff(y)]] if diff else [])
    if rank:
        x = stats.rankdata(x); targets = [stats.rankdata(t) for t in targets]
    best = None
    for t in targets:
        S = _shift_corr(x, t, R)                     # shifts -R..R
        if lags:
            S = np.max(np.stack([S[lags + l: lags + l + 2 * r + 1] for l in range(-lags, lags + 1)]), 0)
        best = S if best is None else np.maximum(best, S)
    S0 = best[r]
    if lags:
        # exclusion zone: shifts with |d| <= 2*lags share alignments with d = 0 and are left out of the null
        d = np.arange(-r, r + 1); null = best[np.abs(d) > 2 * lags]
        p = (1 + np.sum(null >= S0 - 1e-12)) / (len(null) + 1)
    else:
        p = np.sum(best >= S0 - 1e-12) / (2 * r + 1)
    return float(S0), float(p)


def holm(p):
    p = np.asarray(p, float); m = len(p)
    o = np.argsort(p); adj = np.empty(m); run = 0.0
    for i, j in enumerate(o):
        run = max(run, min(1.0, (m - i) * p[j]))
        adj[j] = run
    return adj


def association_tests(k, y, X: pd.DataFrame, r=None, conditional=True):
    """Marginal (x vs y) and index-conditional (resid_x vs resid_y) TTS tests."""
    rows = []
    _, ry = crossfit_trend(k, y)
    ry_ok = np.std(ry) > 1e-6 * np.std(y)
    for c in X.columns:
        x = X[c].values.astype(float)
        S0, p = tts_test(x, y, r)
        rec = dict(feature=c, S_marg=S0, p_marg=p)
        if conditional and ry_ok:
            _, rx = crossfit_trend(k, x)
            S0c, pc = tts_test(rx, ry, r)
            rec.update(S_cond=S0c, p_cond=pc)
        else:
            rec.update(S_cond=np.nan, p_cond=np.nan)
        rows.append(rec)
    out = pd.DataFrame(rows)
    out['p_marg_holm'] = holm(out.p_marg.values)
    if out.p_cond.notna().all():
        out['p_cond_holm'] = holm(out.p_cond.values)
    else:
        out['p_cond_holm'] = np.nan
    out.attrs['conditional_testable'] = bool(ry_ok)
    return out


# ----------------------------------------------------------------------------
# S4  sensitivity calibration by signal injection (self-surrogate)
# ----------------------------------------------------------------------------

def inject(xnull, target, rho):
    """Return x_inj whose population correlation with target is rho when xnull
    is independent of target; preserves the mean/SD of xnull."""
    zx, zt = _z(xnull), _z(target)
    zi = np.sqrt(1 - rho ** 2) * zx + rho * zt
    return xnull.mean() + xnull.std() * zi


def self_surrogate_power(x, y, rhos, n_rep=200, alpha=0.05, r=None, seed=0,
                         target=None, n_tests=1):
    """Power of the (Holm-adjusted, n_tests-family) TTS test to detect an injected
    coupling of strength rho. Null realisations are random circular shifts of the
    observed x (self-surrogates), which preserve its autocorrelation."""
    rng = np.random.default_rng(seed)
    x = np.asarray(x, float); y = np.asarray(y, float)
    t = y if target is None else target
    n = len(x)
    a_eff = alpha / n_tests  # Bonferroni bound on the Holm step for the worst rank
    res = []
    for rho in rhos:
        hits = 0
        for _ in range(n_rep):
            xs = np.roll(x, rng.integers(n // 10, n - n // 10))
            xi = inject(xs, t, rho)
            _, p = tts_test(xi, y, r)
            hits += p <= a_eff
        res.append(hits / n_rep)
    return np.array(res)


def mde(rhos, power, target=0.8):
    """Interpolated smallest coupling with power >= target (MDE80)."""
    rhos = np.asarray(rhos); power = np.asarray(power)
    above = np.where(power >= target)[0]
    if len(above) == 0:
        return np.nan
    i = above[0]
    if i == 0:
        return float(rhos[0])
    # linear interpolation between grid points
    r0, r1, p0, p1 = rhos[i - 1], rhos[i], power[i - 1], power[i]
    return float(r0 + (target - p0) * (r1 - r0) / (p1 - p0)) if p1 > p0 else float(r1)


# ----------------------------------------------------------------------------
# verdicts
# ----------------------------------------------------------------------------

def verdict(row, mde_val, rho_rel=0.1, alpha=0.05, derived=False):
    if derived:
        return 'D'  # derived / deterministic: exclude as feature
    if row['p_marg_holm'] <= alpha or (not np.isnan(row.get('p_cond_holm', np.nan))
                                       and row['p_cond_holm'] <= alpha):
        return 'I'  # informative
    if not np.isnan(mde_val) and mde_val <= rho_rel:
        return 'N'  # null with adequate power
    return 'U'      # underpowered / inconclusive


# ----------------------------------------------------------------------------
# full audit card
# ----------------------------------------------------------------------------

def proxy_check(k, y, x):
    """Residual correlation between feature and target after removing the index
    trend. |r_res| >= 0.98 means the feature reproduces the target's own
    cycle-to-cycle fluctuations: a likely co-computed proxy (verify provenance)."""
    _, ry = crossfit_trend(k, y); _, rx = crossfit_trend(k, x)
    if np.std(ry) < 1e-6 * np.std(y):
        return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])


def restatement_index(x, y):
    """zeta = RMS(y - a - b x) / SD(y): how closely a linear map of the feature restates the target on
    the raw scale. zeta < 0.01 means the feature reproduces the target to within 1% of its spread."""
    x = np.asarray(x, float); y = np.asarray(y, float)
    if np.std(x) == 0 or np.std(y) == 0:
        return np.nan
    A = np.column_stack([np.ones_like(x), x]); res = y - A @ np.linalg.lstsq(A, y, rcond=None)[0]
    return float(np.sqrt(np.mean(res ** 2)) / np.std(y))


def run_audit(df, index_col, target_col, feature_cols, r=None, B=999,
              eta_det=1e-3, proxy_thr=0.98, zeta_thr=0.01, alpha=0.05, seed=0, version=2,
              mde=False, mde_rhos=(0, .1, .2, .3, .4, .5, .6, .7, .8, .9), mde_reps=200):
    """TRACE audit card (stages S1-S4).
    S1 identity / determinism (eta) / proxy (|r_res|, zeta) screen; S2 index-only ceiling (attrs['ceiling']);
    S3 gated permutation trend test and truncated time-shift beyond-index test, Holm within the tested family;
    S4 (mde=True) self-surrogate MDE80 of the beyond-index test for every T and N feature (column 'MDE80').
    The truncation radius r is fixed before classification, so the resolution rule uses m = number of candidate
    features (an upper bound on the tested family); Holm then uses the tested family.
    version=2 (default): level + rate beyond-index statistic and restatement index zeta. version=1: original TRACE."""
    lags, use_diff = (0, True) if version >= 2 else (0, False)
    k = df[index_col].values.astype(float); y = df[target_col].values.astype(float)
    n = len(k)
    m_tests = len(feature_cols)
    if r is None:
        # resolution rule: the smallest Holm-adjusted p, m/(2r+1), must not exceed alpha
        r_floor = int(np.ceil((m_tests / alpha - 1) / 2))
        r = int(min(max(n // 4, r_floor), np.floor(0.4 * n) - lags))
    eta_y = determinism_index(k, y)
    ceil = index_ceiling(k, y)
    rows = []
    for c in feature_cols:
        x = df[c].values.astype(float)
        rec = dict(feature=c)
        if np.std(x) == 0:
            rec.update(cls='D', reason='constant'); rows.append(rec); continue
        rx_aff = abs(np.corrcoef(x, k)[0, 1]) >= 1 - 1e-9 or abs(np.corrcoef(x, y)[0, 1]) >= 1 - 1e-9
        eta = determinism_index(k, x)
        tt = trend_test(k, x, B=B, seed=seed)
        rp = proxy_check(k, y, x)
        zt = restatement_index(x, y) if version >= 2 else np.nan
        rec.update(eta=eta, R2_trend=tt['R2_trend'], p_trend=tt['p_trend'],
                   exch=tt['exchangeable'], r_resid=rp, zeta=zt)
        if eta_y < eta_det:
            rec.update(S_cond=np.nan, p_cond=np.nan)
        else:
            _, ry = crossfit_trend(k, y); _, rx = crossfit_trend(k, x)
            S, p = tts_test(rx, ry, r, lags=lags, diff=use_diff)
            rec.update(S_cond=S, p_cond=p)
        if rx_aff:
            rec['reason'] = 'affine in index or target'
        elif eta < eta_det:
            rec['reason'] = 'deterministic in index'
        elif not np.isnan(rp) and abs(rp) >= proxy_thr:
            rec['reason'] = 'target proxy (residual |r|>=%.2f)' % proxy_thr
        elif not np.isnan(zt) and zt < zeta_thr:
            rec['reason'] = 'target proxy (restatement zeta<%.3f)' % zeta_thr
        else:
            rec['reason'] = ''
        rows.append(rec)
    out = pd.DataFrame(rows)
    tested = out['reason'].fillna('') == ''
    out['p_cond_holm'] = np.nan
    if tested.any() and out.loc[tested, 'p_cond'].notna().all():
        out.loc[tested, 'p_cond_holm'] = holm(out.loc[tested, 'p_cond'].values)
    out['p_trend_holm'] = np.nan
    ex = tested & out['exch'].fillna(False).astype(bool)
    if ex.any():
        out.loc[ex, 'p_trend_holm'] = holm(out.loc[ex, 'p_trend'].values)
    cls = []
    for _, rw in out.iterrows():
        if rw.get('reason', ''):
            cls.append('D' if 'proxy' not in rw['reason'] else 'P'); continue
        if not np.isnan(rw['p_cond_holm']) and rw['p_cond_holm'] <= alpha:
            cls.append('I'); continue
        trending = (rw['exch'] and rw['p_trend_holm'] <= alpha) or (not rw['exch'] and rw['R2_trend'] > 0.1)
        if trending:
            cls.append('T'); continue
        cls.append('N' if eta_y >= eta_det else 'N*')
    out['class'] = cls
    m_eff = int(tested.sum())
    out['MDE80'] = np.nan
    if mde and eta_y >= eta_det and m_eff > 0:
        _, ry = crossfit_trend(k, y)
        for i in out.index[out['class'].isin(['T', 'N'])]:
            _, rx = crossfit_trend(k, df[out.at[i, 'feature']].values.astype(float))
            pw = self_surrogate_power(rx, ry, mde_rhos, n_rep=mde_reps, alpha=alpha, r=r, seed=seed, n_tests=m_eff)
            out.at[i, 'MDE80'] = mde_fn(mde_rhos, pw)
    out.attrs.update(eta_target=eta_y, ceiling=ceil, r=r, conditional_testable=eta_y >= eta_det,
                     m_tested=m_eff, min_holm_p=m_eff / (2 * r + 1 - (4 * lags if lags else 0)))
    return out


mde_fn = mde   # alias used inside run_audit, whose argument 'mde' is a switch


## Step 3 – data loading, generator verification and NASA feature extraction

In [ ]:
%%writefile 00_data.py
"""Step 0 - data. GMP: use data/GMP.csv if present (download from Mendeley, doi:10.17632/k675pxrd83.2),
otherwise regenerate it from the published generating equations (Data in Brief 69 (2026) 113252, Eqs. 1-8, seed 42)
and, if the CSV is present, verify that the equations reproduce it. NASA: download data_NASA.rda and build cycle features."""
import os, urllib.request, numpy as np, pandas as pd
os.makedirs('data', exist_ok=True); os.makedirs('outputs', exist_ok=True); os.makedirs('figures', exist_ok=True)

def generate_gmp(seed=42):
    np.random.seed(seed); N = 3000; n = np.arange(1, N + 1)
    cap = 40.0 - 12.0 * (n / N) ** 1.35
    V = np.random.normal(25.60, 0.12, N); I = np.random.uniform(-40.0, 40.0, N)
    Tt = np.random.normal(28.0, 1.8, N); C = np.random.normal(3.200, 0.008, (N, 8))
    d = pd.DataFrame({'PackID': 1, 'CycleNumber': n, 'Capacity(Ah)': cap, 'SOH(%)': cap / 40 * 100,
                      'RUL(cycles)': N - n, 'Voltage(V)': V, 'Current(A)': I, 'DoD(%)': n / N * 100, 'Temperature(C)': Tt})
    for i in range(8): d[f'Cell{i+1}_V'] = C[:, i]
    return d

gen = generate_gmp()
if os.path.exists('data/GMP.csv'):
    pub = pd.read_csv('data/GMP.csv')
    diff = max(float(np.max(np.abs(pub[c].values - gen[c].values))) for c in gen.columns)
    print(f'GMP: published CSV found; max |published - regenerated| = {diff:.2e}')
else:
    gen.to_csv('data/GMP.csv', index=False)
    print('GMP: published CSV not found -> regenerated from the published equations (identical to Mendeley V2 up to CSV rounding)')

if not os.path.exists('data/nasa_cycle_features.csv'):
    url = 'https://raw.githubusercontent.com/cjrincon/cnnlstm/master/data/data_NASA.rda'
    if not os.path.exists('data/data_NASA.rda'):
        urllib.request.urlretrieve(url, 'data/data_NASA.rda')
    import pyreadr
    d = pyreadr.read_r('data/data_NASA.rda')['data_NASA']; rows = []
    for (b, c), g in d.groupby(['battery', 'cycle']):
        g = g.sort_values('time'); t = g.time.values; v = g.voltage_measured.values
        T = g.temperature_measured.values; I = g.current_measured.values
        below = np.where(v <= 2.7)[0]; icut = below[0] if len(below) else len(v) - 1; seg = slice(1, icut + 1)
        rows.append(dict(battery=int(b), cycle=int(c), capacity=float(g.capacity.iloc[0]),
                         ambient_T=float(g.ambient_temperature.iloc[0]), t_discharge=float(t[icut]),
                         V_mean=float(np.mean(v[seg])), V_600s=float(np.interp(600, t, v)),
                         T_max=float(np.max(T[:icut + 1])), t_Tmax=float(t[np.argmax(T[:icut + 1])]),
                         T_start=float(T[0]), I_mean=float(np.mean(I[seg]))))
    pd.DataFrame(rows).sort_values(['battery', 'cycle']).to_csv('data/nasa_cycle_features.csv', index=False)
f = pd.read_csv('data/nasa_cycle_features.csv'); print('NASA cycles per cell:', f.groupby('battery').size().to_dict())


In [ ]:
!python 00_data.py

## Step 4 – TRACE audit of GMP and NASA (V1, V4; Table 5)

In [ ]:
%%writefile 01_audits.py
import numpy as np, pandas as pd, sys, json
sys.path.insert(0,'.'); import trace_audit as T
from scipy import stats
pd.set_option('display.width',250); pd.set_option('display.max_columns',30)
g=pd.read_csv('data/GMP.csv')
cells=[f'Cell{i}_V' for i in range(1,9)]
g['Imbalance(V)']=g[cells].max(1)-g[cells].min(1)
feats=['SOH(%)','RUL(cycles)','DoD(%)','PackID','Voltage(V)','Current(A)','Temperature(C)']+cells+['Imbalance(V)']
a=T.run_audit(g,'CycleNumber','Capacity(Ah)',feats,B=999)
print(a.round(4).to_string()); print(a.attrs)
a.to_csv('outputs/out_audit_gmp.csv',index=False); json.dump({k:(v if not isinstance(v,(np.floating,np.bool_)) else float(v)) for k,v in a.attrs.items()},open('outputs/out_audit_gmp_attrs.json','w'),default=float)
f=pd.read_csv('data/nasa_cycle_features.csv')
nf=['ambient_T','I_mean','t_discharge','t_Tmax','V_mean','V_600s','T_max','T_start']
allc=[]
for b,h in f.groupby('battery'):
    h=h.reset_index(drop=True)
    a=T.run_audit(h,'cycle','capacity',nf,B=999)
    a.insert(0,'battery',f'B{b:04d}'); a['eta_target']=a.attrs['eta_target']; a['R2_index']=a.attrs['ceiling']['R2_index']; a['r']=a.attrs['r']
    allc.append(a); print(a.round(4).to_string())
A=pd.concat(allc); A.to_csv('outputs/out_audit_nasa.csv',index=False)
# (no pooling of p-values across cells: the four records are audited and reported separately)


In [ ]:
!python 01_audits.py

## Step 5 – shift test vs naive test under autocorrelation (V2, Fig. 3b)

In [ ]:
%%writefile 02_typeI.py
import numpy as np, pandas as pd, sys, time
sys.path.insert(0,'.'); import trace_audit as T
from scipy import stats
N=3000; n=np.arange(1,N+1); cap=40-12*(n/N)**1.35
import os; QUICK=os.environ.get('TRACE_QUICK')=='1'
R=100 if QUICK else 1000; r=750; t0=time.time()
# (a) regenerated GMP nulls: 12 operating channels per replicate (V,I,T,8 cells, imbalance)
fp=np.zeros(12); fw=0; fp_naive=np.zeros(12); fw_naive=0
for s in range(R):
    rng=np.random.RandomState(1000+s)
    V=rng.normal(25.6,.12,N); I=rng.uniform(-40,40,N); Tt=rng.normal(28,1.8,N); C=rng.normal(3.2,.008,(N,8))
    X=np.column_stack([V,I,Tt,C,C.max(1)-C.min(1)])
    p=np.array([T.tts_test(X[:,j],cap,r)[1] for j in range(12)])
    pn=np.array([stats.spearmanr(X[:,j],cap)[1] for j in range(12)])
    fp+=p<=0.05; fp_naive+=pn<=0.05
    fw+=np.any(T.holm(p)<=0.05); fw_naive+=np.any(T.holm(pn)<=0.05)
print('GMP-null per-channel FPR TTS',np.round(fp/R,3),'mean',fp.mean()/R,'FWER(Holm)',fw/R)
print('GMP-null per-channel FPR naive',np.round(fp_naive/R,3),'mean',fp_naive.mean()/R,'FWER',fw_naive/R)
# (b) autocorrelated nulls: AR(1) with phi, independent of capacity
rows=[]
for phi in [0.0,0.9,0.99,0.995]:
    a=b=0
    for s in range(R):
        rng=np.random.default_rng(5000+s)
        e=rng.normal(size=N+2000); x=np.zeros_like(e)
        for i in range(1,len(e)): x[i]=phi*x[i-1]+e[i]
        x=x[2000:]
        a+=T.tts_test(x,cap,r)[1]<=0.05
        b+=stats.spearmanr(x,cap)[1]<=0.05
    rows.append(dict(phi=phi,FPR_TTS=a/R,FPR_naive=b/R))
    print(rows[-1],round(time.time()-t0))
pd.DataFrame(rows).to_csv('outputs/out_typeI_ar.csv',index=False)
pd.DataFrame(dict(channel=['V','I','T']+[f'C{i}' for i in range(1,9)]+['Imb'],FPR_TTS=fp/R,FPR_naive=fp_naive/R)).to_csv('outputs/out_typeI_gmp.csv',index=False)
open('outputs/out_typeI_fwer.txt','w').write(f'{fw/R},{fw_naive/R}')


In [ ]:
!python 02_typeI.py

## Step 6 – false-positive control and power / MDE₈₀ (V2, V3; Figs. 3–4)

In [ ]:
%%writefile 03_validation.py
"""Step 3 - V2 (false positives of the trend test and the beyond-index test) and V3 (power / MDE80)."""
import numpy as np, pandas as pd, sys, time, json, os
sys.path.insert(0, '.'); import trace_audit as T
from scipy import stats
QUICK = os.environ.get('TRACE_QUICK') == '1'
t0 = time.time(); res = {}
N = 3000; k = np.arange(1, N + 1.); cap = 40 - 12 * (k / N) ** 1.35
# fast factorised two-fold cross-fitted spline operator (identical to T._crossfit_operator(k, 12))
Bb = T._bspline_basis(k, 12); idx = np.arange(N); PARTS = []
for part in (0, 1):
    tr = idx % 2 != part; te = ~tr; PARTS.append((tr, te, np.linalg.pinv(Bb[tr]), Bb[te]))
def M(P):
    out = np.empty_like(P)
    for tr, te, L, Bt in PARTS: out[te] = Bt @ (L @ P[tr])
    return out
def r2(v): r = v - M(v); return 1 - np.mean(r ** 2) / np.var(v)

# ---- V2a trend-test FPR on regenerated GMP null datasets (permutation, B = 399)
R = 100 if QUICK else 1000; fp = np.zeros(12); fw = 0
for s in range(R):
    rng = np.random.RandomState(1000 + s)
    V = rng.normal(25.6, .12, N); I = rng.uniform(-40, 40, N); Tt = rng.normal(28, 1.8, N); C = rng.normal(3.2, .008, (N, 8))
    X = np.column_stack([V, I, Tt, C, C.max(1) - C.min(1)]); ps = []
    for j in range(12):
        x = X[:, j]; s0 = r2(x)
        P = np.stack([rng.permutation(x) for _ in range(399)], 1); Rr = P - M(P)
        nul = 1 - np.mean(Rr ** 2, 0) / np.var(P, 0); ps.append((1 + np.sum(nul >= s0)) / 400)
    ps = np.array(ps); fp += ps <= 0.05; fw += np.any(T.holm(ps) <= 0.05)
res['V2a_trend_FPR'] = list(fp / R); res['V2a_trend_FWER'] = fw / R
print('V2a', np.round(fp / R, 3), fw / R, round(time.time() - t0), flush=True)

# ---- V3a trend-injection power on GMP: true generator vs self-surrogate
g = pd.read_csv('data/GMP.csv'); v_obs = g['Voltage(V)'].values; rng = np.random.default_rng(3)
P = np.stack([rng.permutation(v_obs) for _ in range(4000)], 1); nul = 1 - np.mean((P - M(P)) ** 2, 0) / np.var(P, 0)
ct = np.quantile(nul, 0.95); cf = np.quantile(nul, 1 - 0.05 / 12)
rows = []; RR = 100 if QUICK else 500
for kind in ['generator', 'self-surrogate']:
    for rho in [0, 0.02, 0.04, 0.06, 0.08, 0.10, 0.12, 0.14, 0.16, 0.18, 0.20]:
        h1 = h2 = 0
        for s in range(RR):
            x = np.random.RandomState(20000 + s).normal(25.6, .12, N) if kind == 'generator' else rng.permutation(v_obs)
            st = r2(T.inject(x, cap, rho)); h1 += st > ct; h2 += st > cf
        rows.append(dict(kind=kind, rho=rho, power_per_test=h1 / RR, power_family12=h2 / RR))
pw = pd.DataFrame(rows); pw.to_csv('outputs/out_power_trend_gmp.csv', index=False)
for kind in ['generator', 'self-surrogate']:
    q = pw[pw.kind == kind]; res[f'MDE80_trend_{kind}_test'] = T.mde(q.rho, q.power_per_test); res[f'MDE80_trend_{kind}_fam'] = T.mde(q.rho, q.power_family12)
print({k_: v for k_, v in res.items() if 'MDE' in k_}, round(time.time() - t0), flush=True)

# ---- V2b beyond-index FPR on NASA targets (AR(1) + random smooth trend nulls) vs naive test
f = pd.read_csv('data/nasa_cycle_features.csv'); rows = []; RN = 200 if QUICK else 1000
def r_rule(n, m=8): return int(min(max(n // 4, int(np.ceil((m / 0.05 - 1) / 2))), np.floor(0.4 * n)))
for b, h in f.groupby('battery'):
    kk = h.cycle.values.astype(float); y = h.capacity.values; n = len(kk); r = r_rule(n)
    _, ry = T.crossfit_trend(kk, y)
    for phi in [0.0, 0.5, 0.9, 0.97]:
        a = bn = 0
        for s in range(RN):
            rg = np.random.default_rng(40000 + s); e = rg.normal(size=n + 500); x = np.zeros_like(e)
            for i in range(1, len(e)): x[i] = phi * x[i - 1] + e[i]
            x = x[500:] + rg.normal(0, 2) * np.sin(np.pi * kk / n * rg.uniform(0.5, 2) + rg.uniform(0, 6))
            _, rx = T.crossfit_trend(kk, x)
            a += T.tts_test(rx, ry, r)[1] <= 0.05; bn += stats.spearmanr(rx, ry)[1] <= 0.05
        rows.append(dict(battery=b, phi=phi, FPR_TTS=a / RN, FPR_naive=bn / RN))
pd.DataFrame(rows).to_csv('outputs/out_typeI_cond_nasa.csv', index=False); print('V2b done', round(time.time() - t0), flush=True)

# ---- V3b beyond-index power on NASA (self-surrogate: circular shifts of the start-temperature residual)
rows = []; rhos = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]; RP = 100 if QUICK else 400
for b, h in f.groupby('battery'):
    kk = h.cycle.values.astype(float); y = h.capacity.values; n = len(kk); r = r_rule(n)
    _, ry = T.crossfit_trend(kk, y); _, rx = T.crossfit_trend(kk, h.T_start.values); rg = np.random.default_rng(7)
    for rho in rhos:
        h1 = h2 = 0
        for s in range(RP):
            xi = T.inject(np.roll(rx, rg.integers(n // 10, n - n // 10)), ry, rho)
            p = T.tts_test(xi, ry, r)[1]; h1 += p <= 0.05; h2 += p <= 0.05 / 5
        rows.append(dict(battery=b, rho=rho, power_per_test=h1 / RP, power_family5=h2 / RP))
pc = pd.DataFrame(rows); pc.to_csv('outputs/out_power_cond_nasa.csv', index=False)
for b, q in pc.groupby('battery'):
    res[f'MDE80_cond_B{b}_test'] = T.mde(q.rho, q.power_per_test); res[f'MDE80_cond_B{b}_fam'] = T.mde(q.rho, q.power_family5)
json.dump(res, open('outputs/out_validation.json', 'w'), indent=1, default=float)
print({k_: (round(v, 3) if isinstance(v, float) else v) for k_, v in res.items() if 'MDE' in k_}, round(time.time() - t0))


In [ ]:
!python 03_validation.py

## Step 7 – diagnostics quoted in the text (Ljung–Box size, Mann–Kendall size)

In [ ]:
%%writefile 04_ljungbox_mk.py
"""Step 4 - diagnostics quoted in the text: size of the Monte Carlo-calibrated Ljung-Box gate (and of the chi-square reference) on detrended iid noise; Mann-Kendall (Hamed-Rao) size under AR(1)."""
import numpy as np, sys, json, os
sys.path.insert(0, '.'); import trace_audit as T
from statsmodels.stats.diagnostic import acorr_ljungbox
import pymannkendall as mk
QUICK = os.environ.get('TRACE_QUICK') == '1'
k = np.arange(1, 3001.); Bf = T._bspline_basis(k, 12); out = {}
R = 100 if QUICK else 400; f = []
for i in range(R):
    x = np.random.default_rng(i).normal(size=3000); r = x - Bf @ np.linalg.lstsq(Bf, x, rcond=None)[0]
    f.append(acorr_ljungbox(r, lags=[10], return_df=True)['lb_pvalue'].iloc[0] <= 0.05)
out['LB_size_lag10_chi2_reference'] = float(np.mean(f))
out['LB_size_lag10'] = float(np.mean([T.lb_gate(k, np.random.default_rng(i).normal(size=3000), B=299, seed=i) <= 0.05 for i in range(R)]))
R = 60 if QUICK else 300
for phi in [0.9, 0.99]:
    c = 0
    for s in range(R):
        rng = np.random.default_rng(7000 + s); e = rng.normal(size=5000); x = np.zeros_like(e)
        for i in range(1, len(e)): x[i] = phi * x[i - 1] + e[i]
        c += mk.hamed_rao_modification_test(x[2000:]).p <= 0.05
    out[f'MK_HR_size_phi{phi}'] = c / R
json.dump(out, open('outputs/out_diagnostics.json', 'w'), indent=1); print(out)


In [ ]:
!python 04_ljungbox_mk.py

## Step 8 – robustness to knots and truncation radius (V5)

In [ ]:
%%writefile 05_robustness.py
import numpy as np, pandas as pd, sys
sys.path.insert(0,'.'); import trace_audit as T
f=pd.read_csv('data/nasa_cycle_features.csv'); nf=['ambient_T','I_mean','t_discharge','t_Tmax','V_mean','V_600s','T_max','T_start']
base=pd.read_csv('outputs/out_audit_nasa.csv').set_index(['battery','feature'])['class']
rows=[]
orig=T.crossfit_trend
for kn in [6,9,12]:
    for rfrac in ['default',0.25,0.33]:
        T.crossfit_trend=lambda k,x,n_knots=None,_kn=kn: orig(k,x,_kn)
        _old=T.determinism_index
        for b,h in f.groupby('battery'):
            h=h.reset_index(drop=True); n=len(h)
            r=None if rfrac=='default' else int(rfrac*n)
            a=T.run_audit(h,'cycle','capacity',nf,B=499,r=r)
            for _,w in a.iterrows():
                rows.append(dict(knots=kn,r=rfrac,battery=f'B{b:04d}',feature=w.feature,cls=w['class'],base=base[(f'B{b:04d}',w.feature)]))
T.crossfit_trend=orig
d=pd.DataFrame(rows); d['agree']=d.cls==d.base
print(d.groupby(['knots','r']).agree.mean().unstack())
print(d[~d.agree].groupby(['battery','feature']).cls.apply(lambda s: ','.join(s)).to_string())
d.to_csv('outputs/out_robust_nasa.csv',index=False)


In [ ]:
!python 05_robustness.py

## Step 9 – NASA leave-one-cell-out and SHAP consequences (V8, Fig. 9)

In [ ]:
%%writefile 06_consequences.py
"""E1 generalizability (leave-one-cell-out) and E2 explainability (SHAP) consequences of TRACE classes."""
import numpy as np, pandas as pd, json, sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0,'.'); import trace_audit as T
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import shap
f=pd.read_csv('data/nasa_cycle_features.csv')
FS={'Index only':['cycle'],
    'All features':['cycle','t_discharge','t_Tmax','V_mean','V_600s','T_max','T_start','I_mean'],
    'Proxy only (P)':['t_discharge'],
    'Trend-only (T)':['cycle','T_max','T_start','I_mean'],
    'TRACE-admitted (I)':['cycle','V_600s','V_mean']}
rows=[]; shap_rows=[]
for name,cols in FS.items():
    for held in sorted(f.battery.unique()):
        tr=f[f.battery!=held]; te=f[f.battery==held]
        for mname,mk in [('GBR',lambda: GradientBoostingRegressor(n_estimators=300,max_depth=3,learning_rate=0.05,subsample=0.8,random_state=0)),
                         ('Ridge',lambda: make_pipeline(StandardScaler(),RidgeCV(alphas=np.logspace(-3,3,13))))]:
            m=mk().fit(tr[cols],tr.capacity); p=m.predict(te[cols])
            rmse=float(np.sqrt(np.mean((p-te.capacity)**2))); r2=float(1-np.sum((p-te.capacity)**2)/np.sum((te.capacity-te.capacity.mean())**2))
            rows.append(dict(feature_set=name,model=mname,held_out=f'B{held:04d}',RMSE=rmse,R2=r2))
            if mname=='GBR' and name in ('All features','TRACE-admitted (I)'):
                sv=shap.TreeExplainer(m).shap_values(te[cols])
                for j,c in enumerate(cols): shap_rows.append(dict(feature_set=name,held_out=f'B{held:04d}',feature=c,mean_abs_shap=float(np.mean(np.abs(sv[:,j])))))
R=pd.DataFrame(rows); R.to_csv('outputs/out_loco.csv',index=False)
S=pd.DataFrame(shap_rows); S.to_csv('outputs/out_shap_nasa.csv',index=False)
print(R.pivot_table(index=['feature_set'],columns=['model'],values='RMSE',aggfunc='mean').round(4))
print(R.pivot_table(index=['feature_set','held_out'],columns='model',values='R2').round(3))
print(S.groupby(['feature_set','feature']).mean_abs_shap.mean().round(4))
# ---------- GMP: SHAP null calibration with self-surrogates ----------
g=pd.read_csv('data/GMP.csv'); cells=[f'Cell{i}_V' for i in range(1,9)]
ops=['Voltage(V)','Current(A)','Temperature(C)']+cells
X=g[['CycleNumber']+ops].copy(); y=g['Capacity(Ah)'].values
rng=np.random.default_rng(0)
for c in ops: X['NULL_'+c]=rng.permutation(X[c].values)   # self-surrogate null copies
idx=rng.permutation(len(X)); tr,te=idx[:2400],idx[2400:]       # random split, as commonly (mis)used
m=GradientBoostingRegressor(n_estimators=300,max_depth=3,learning_rate=0.05,subsample=0.8,random_state=0).fit(X.iloc[tr],y[tr])
p=m.predict(X.iloc[te]); r2_rand=float(1-np.sum((p-y[te])**2)/np.sum((y[te]-y[te].mean())**2))
sv=shap.TreeExplainer(m).shap_values(X.iloc[te]); ma=np.mean(np.abs(sv),0)
G=pd.DataFrame(dict(feature=X.columns,mean_abs_shap=ma)); G.to_csv('outputs/out_shap_gmp.csv',index=False)
null_band=G[G.feature.str.startswith('NULL_')].mean_abs_shap
real=G[G.feature.isin(ops)].mean_abs_shap
print('GMP random-split R2',r2_rand,'cycle shap',float(G[G.feature=='CycleNumber'].mean_abs_shap.iloc[0]))
print('real ops mean|SHAP| range',real.min(),real.max(),'null copies range',null_band.min(),null_band.max())
from scipy.stats import mannwhitneyu
mw=mannwhitneyu(real,null_band); print('MW p',mw.pvalue)
# chronological split with operating features only (no index)
m2=GradientBoostingRegressor(n_estimators=300,max_depth=3,learning_rate=0.05,random_state=0).fit(g[ops].iloc[:2100],y[:2100])
p2=m2.predict(g[ops].iloc[2100:]); r2_chrono=float(1-np.sum((p2-y[2100:])**2)/np.sum((y[2100:]-y[2100:].mean())**2))
print('GMP chrono ops-only R2',r2_chrono)
json.dump(dict(gmp_r2_random_split=r2_rand,gmp_r2_chrono_ops_only=r2_chrono,
               gmp_shap_cycle=float(G[G.feature=='CycleNumber'].mean_abs_shap.iloc[0]),
               gmp_shap_ops_min=float(real.min()),gmp_shap_ops_max=float(real.max()),
               gmp_shap_null_min=float(null_band.min()),gmp_shap_null_max=float(null_band.max()),
               gmp_shap_mw_p=float(mw.pvalue)),open('outputs/out_consequence.json','w'),indent=1)


In [ ]:
!python 06_consequences.py

## Step 10 – reliability landscape: MDE₈₀ surface, null P–P, shift landscape (Fig. 5)

In [ ]:
%%writefile 07_landscape.py
"""Reliability landscape: MDE80 of the beyond-index test over record length n and feature autocorrelation phi;
null p-value calibration; shift-statistic landscape on B0005."""
import numpy as np, pandas as pd, sys, json, time
sys.path.insert(0,'.'); import trace_audit as T
from scipy import stats
def ar(n,phi,rng):
    e=rng.normal(size=n+300); x=np.zeros_like(e)
    for i in range(1,len(e)): x[i]=phi*x[i-1]+e[i]
    return x[300:]
t0=time.time(); rows=[]
ns=[100,150,250,500,1000,2000]; phis=[0.0,0.5,0.8,0.95]; rhos=np.array([0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9]); R=200; m=5; a=0.05/m
for n in ns:
    r=int(min(max(n//4,int(np.ceil((m/0.05-1)/2))),np.floor(0.4*n)))
    for phi in phis:
        pw=[]
        for rho in rhos:
            h=0
            for s in range(R):
                rng=np.random.default_rng(hash((n,phi,s))%2**32)
                ey=ar(n,0.7,rng); ex=ar(n,phi,rng)       # target residual AR(0.7) as in NASA capacity residuals
                h+=T.tts_test(T.inject(ex,ey,rho),ey,r)[1]<=a
            pw.append(h/R)
        rows.append(dict(n=n,phi=phi,r=r,MDE80=T.mde(rhos,pw),**{f'p{x:.1f}':y for x,y in zip(rhos,pw)}))
    print(n,round(time.time()-t0),flush=True)
L=pd.DataFrame(rows); L.to_csv('outputs/out_mde_surface.csv',index=False); print(L[['n','phi','r','MDE80']].to_string())
# null p-value calibration (P-P): beyond-index TTS vs naive, n=168, phi=0.9
pt,pn=[],[]
for s in range(2000):
    rng=np.random.default_rng(90000+s); ey=ar(168,0.7,rng); ex=ar(168,0.9,rng)
    pt.append(T.tts_test(ex,ey,67)[1]); pn.append(stats.spearmanr(ex,ey)[1])
pd.DataFrame(dict(p_tts=pt,p_naive=pn)).to_csv('outputs/out_null_pvalues.csv',index=False)
print('KS uniform (TTS is conservative/discrete):',np.mean(np.array(pt)<=0.05),np.mean(np.array(pn)<=0.05))
# shift landscape for B0005
f=pd.read_csv('data/nasa_cycle_features.csv'); h=f[f.battery==5]; k=h.cycle.values.astype(float)
_,ry=T.crossfit_trend(k,h.capacity.values); r=67; out={}
for c in ['V_600s','V_mean','T_max','T_start','I_mean']:
    _,rx=T.crossfit_trend(k,h[c].values)
    out[c]=list(T._shift_corr(stats.rankdata(rx),stats.rankdata(ry),r))
json.dump(out,open('outputs/out_shift_landscape_B0005.json','w'))


In [ ]:
!python 07_landscape.py

## Step 11 – external validation: 374 public cells (XJTU, TJU, HUST, MIT; cloned from PINN4SOH at a pinned commit) and cross-cell LOCO (V6, Fig. 7)

In [ ]:
%%writefile 11_external.py
"""Step 11 - V8: external validation on 387 measured cells from four public datasets (XJTU, TJU, HUST, MIT;
processed release github.com/wang-fujin/PINN4SOH, commit cc3cc50), each audited as its own single trajectory.
(a) Audit every cell (16 charge features, target = measured discharge capacity); store statistics and classes.
(b) Consequence for generalizability: within each dataset, leave-one-cell-out GBR with
    index only / index + features that TRACE classes I in the majority of training cells / index + T-majority
    features / index + all non-leaking features. Shows that T features, which TRACE cannot credit on one
    trajectory, may still transfer across cells (the identifiability boundary), with cell-level bootstrap CIs."""
import os, sys, glob, subprocess, warnings, numpy as np, pandas as pd
from joblib import Parallel, delayed
warnings.filterwarnings('ignore'); sys.path.insert(0, '.'); import trace_audit as T
QUICK = os.environ.get('TRACE_QUICK') == '1'
REPO, COMMIT = 'https://github.com/wang-fujin/PINN4SOH.git', 'cc3cc5053caf38f04e0665f7f88cb109144d035e'
F = ['voltage mean', 'voltage std', 'voltage kurtosis', 'voltage skewness', 'CC Q', 'CC charge time', 'voltage slope',
     'voltage entropy', 'current mean', 'current std', 'current kurtosis', 'current skewness', 'CV Q', 'CV charge time',
     'current slope', 'current entropy']


def load():
    tgt = 'data/PINN4SOH'
    if not os.path.isdir(tgt):
        subprocess.run(['git', 'clone', '-q', REPO, tgt], check=True); subprocess.run(['git', '-C', tgt, 'checkout', '-q', COMMIT], check=True)
    B = tgt + '/data/'; fs = glob.glob(B + 'XJTU data/*.csv') + glob.glob(B + 'HUST data/*.csv') + glob.glob(B + 'MIT data/*/*.csv') + glob.glob(B + 'TJU data/*/*.csv')
    out = []
    for f in sorted(fs):
        d = pd.read_csv(f).replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)
        if len(d) < 100: continue
        ds = 'XJTU' if 'XJTU' in f else 'HUST' if 'HUST' in f else 'MIT' if 'MIT' in f else 'TJU'
        d = d.assign(cell=f.split('data/', 2)[-1], dataset=ds, k=np.arange(1, len(d) + 1))
        out.append(d)
    return pd.concat(out, ignore_index=True)


def audit(cell, g):
    card = T.run_audit(g[['k', 'capacity'] + F].reset_index(drop=True), 'k', 'capacity', F, B=199, seed=0)
    card['cell'] = cell; card['dataset'] = g.dataset.iloc[0]; card['n'] = len(g)
    card['eta_target'] = card.attrs['eta_target']; card['r'] = card.attrs['r']; card['p_floor_holm'] = card.attrs['min_holm_p']
    card['R2_index'] = card.attrs['ceiling']['R2_index']
    return card


if __name__ == '__main__':
    D = load(); cells = list(D.groupby('cell'))
    if QUICK: cells = cells[::8]
    print('cells', len(cells), 'datasets', D.dataset.value_counts().to_dict(), flush=True)
    A = pd.concat(Parallel(n_jobs=2)(delayed(audit)(c, g) for c, g in cells), ignore_index=True)
    A.to_csv('outputs/out_ext_audit.csv', index=False)
    C = A.drop_duplicates('cell')
    print('cells audited', len(C), 'median n', C.n.median(), 'eta_target range', C.eta_target.min(), C.eta_target.max(),
          'cells with deterministic target', int((C.eta_target < 1e-3).sum()))
    tab = A.pivot_table(index='feature', columns='class', values='cell', aggfunc='count', fill_value=0)
    tab = (tab.T / tab.sum(1)).T.round(3); tab.to_csv('outputs/out_ext_class_share.csv'); print(tab)
    by_ds = A.groupby(['dataset', 'class']).size().unstack(fill_value=0); by_ds = (by_ds.T / by_ds.sum(1)).T.round(3)
    by_ds.to_csv('outputs/out_ext_class_by_dataset.csv'); print(by_ds)
    # (b) leave-one-cell-out within each dataset
    from sklearn.ensemble import HistGradientBoostingRegressor
    cls = A.set_index(['cell', 'feature'])['class']
    rows = []
    for ds, dd in D[D.cell.isin([c for c, _ in cells])].groupby('dataset'):
        cl = dd.cell.unique()
        for te in cl:
            tr_cells = [c for c in cl if c != te]
            maj = cls.loc[tr_cells].groupby(level='feature').agg(lambda s: s.value_counts().idxmax())
            sets = {'index only': [], 'index + I': [f for f in F if maj.get(f) == 'I'],
                    'index + T': [f for f in F if maj.get(f) == 'T'], 'index + all non-leaking': [f for f in F if maj.get(f) in ('I', 'T', 'N')]}
            tr = dd[dd.cell.isin(tr_cells)].iloc[::3]; ts = dd[dd.cell == te]
            for nm, fs in sets.items():
                X = ['k'] + fs
                p = HistGradientBoostingRegressor(max_iter=150, random_state=0).fit(tr[X], tr.capacity).predict(ts[X])
                y = ts.capacity.values
                rows.append(dict(dataset=ds, cell=te, set=nm, n_feat=len(fs), rmse=float(np.sqrt(np.mean((p - y) ** 2))),
                                 r2=float(1 - np.sum((p - y) ** 2) / np.sum((y - y.mean()) ** 2)), nominal=float(y[:5].mean())))
        print(ds, 'LOCO done', flush=True)
    L = pd.DataFrame(rows); L['rmse_pct'] = 100 * L.rmse / L.nominal; L.to_csv('outputs/out_ext_loco.csv', index=False)
    rng = np.random.default_rng(0); summ = []
    for (ds, s), g in L.groupby(['dataset', 'set']):
        v = g.rmse_pct.values; bs = [np.median(rng.choice(v, len(v))) for _ in range(2000)]
        summ.append(dict(dataset=ds, set=s, cells=len(v), median_rmse_pct=float(np.median(v)), lo=float(np.percentile(bs, 2.5)), hi=float(np.percentile(bs, 97.5)),
                         mean_n_feat=float(g.n_feat.mean())))
    S = pd.DataFrame(summ); S.to_csv('outputs/out_ext_loco_summary.csv', index=False); print(S.round(3).to_string(index=False))


In [ ]:
!python 11_external.py

## Step 12 – benchmark of 10 planted feature types against 6 established tests on 12 measured targets (V7, Fig. 8)

In [ ]:
%%writefile 12_benchmark_fair.py
"""Step 12 - V7 (revised): benchmark of TRACE against established association tests for non-stationary,
autocorrelated series, with ground truth that is not defined through TRACE's own residual.
Targets: measured capacity of the four NASA cells and of 8 public cells (2 per dataset: XJTU, TJU, HUST, MIT).
Planted feature types (truth):
  info_res   (I)  smooth random trend + coupling to the spline residual of capacity (original V7 design; reported for continuity)
  info_lag   (I)  smooth random trend + coupling to the one-cycle-lagged Savitzky-Golay high-pass of capacity (different detrender, lagged)
  info_rate  (I)  smooth random trend + coupling to the first difference of capacity (rate-type feature)
  trend_hi   (T)  cubic polynomial fit of capacity on cycle + AR(0.95) noise: a trend-carrying health indicator
  null_ar    (N)  AR(0.9) noise
  null_trend (N)  random smooth trend + AR(0.9) noise (confounded null)
  proxy      (P)  capacity + noise of 0.2% of its SD
  label      (D)  remaining-cycles countdown n - k
Decision rules (credit = feature declared informative):
  M1 Pearson screen |r| >= 0.3 and iid p < 0.05;  M2 SHAP share >= 5% (random split);
  M3 detrended Spearman, iid p, Holm;  M5 pre-whitened cross-correlation (first differences, AR(p) by AIC, Holm) [Box-Jenkins];
  M6 DCCA coefficient (box n/10) with 299 phase-randomised (Fourier) surrogates of the feature, Holm [Podobnik-Stanley; Theiler];
  M7 detrended regression with Newey-West (HAC) standard errors, Holm;  M4 TRACE (credited = class I; leakage = class P or D).
Part (b): strength sweep of info_smooth (coupling c) against null_trend for TRACE, M5, M6, M7 on the NASA cells.
Part (c): proxy-noise sweep: TRACE class of capacity + noise at 0.2-50% of the capacity SD."""
import os, sys, warnings, json, numpy as np, pandas as pd
from scipy import stats
from scipy.signal import savgol_filter, lfilter
import statsmodels.api as sm
from statsmodels.tsa.ar_model import ar_select_order
from joblib import Parallel, delayed
warnings.filterwarnings('ignore'); sys.path.insert(0, '.'); import trace_audit as T
QUICK = os.environ.get('TRACE_QUICK') == '1'
R_NASA, R_EXT, R_SWEEP = (8, 4, 6) if QUICK else (50, 20, 25)
TYPES = ['info_res', 'info_rate', 'info_smooth', 'info_nonlin', 'info_lag', 'trend_hi', 'null_ar', 'null_trend', 'proxy', 'label']
TRUTH = dict(info_res='I', info_rate='I', info_smooth='I', info_nonlin='I', info_lag='I', trend_hi='T', null_ar='N', null_trend='N', proxy='P', label='D')
DESIGN = ['info_res', 'info_rate']          # types seen while revising the statistic (v2 rate representation)
HELDOUT = ['info_smooth', 'info_nonlin', 'info_lag']   # never used to design or tune TRACE
# two audit families of five features (m = 5, as in the NASA audits), each with informative, negative and leakage members
FAM = [['info_res', 'info_rate', 'trend_hi', 'null_ar', 'proxy'], ['info_smooth', 'info_nonlin', 'info_lag', 'null_trend', 'label']]


def ar(n, phi, rg):
    e = rg.normal(size=n + 300); x = np.zeros_like(e)
    for i in range(1, len(e)): x[i] = phi * x[i - 1] + e[i]
    x = x[300:]; return (x - x.mean()) / x.std()


z = lambda a: (a - a.mean()) / (a.std() + 1e-12)


def plant(k, y, rg, c_lag=0.8):
    n = len(k); _, ry = T.crossfit_trend(k, y)
    smooth = lambda: np.sin(np.pi * k / n * rg.uniform(0.5, 2) + rg.uniform(0, 6))
    w = min(21, (n // 8) * 2 + 1); hp = y - savgol_filter(y, w, 2); lag = np.r_[hp[0], hp[:-1]]
    dy = np.r_[0.0, np.diff(y)]
    return pd.DataFrame(dict(
        info_res=2 * smooth() + 0.8 * z(ry) + 0.6 * rg.normal(size=n),
        info_lag=2 * smooth() + 0.8 * z(lag) + 0.6 * rg.normal(size=n),
        info_rate=2 * smooth() + 0.8 * z(dy) + 0.6 * rg.normal(size=n),
        info_smooth=2 * smooth() + c_lag * z(np.convolve(hp, np.ones(5) / 5, mode='same')) + 0.6 * rg.normal(size=n),
        info_nonlin=2 * smooth() + 0.8 * z(np.tanh(2 * z(hp))) + 0.6 * rg.normal(size=n),
        trend_hi=z(np.polyval(np.polyfit(k, y, 3), k)) + 0.6 * ar(n, 0.95, rg),
        null_ar=ar(n, 0.9, rg), null_trend=2 * smooth() + ar(n, 0.9, rg),
        proxy=y + rg.normal(0, 0.002 * y.std(), n), label=(n - k).astype(float)))


def prewhiten_p(x, y):
    dx, dy = np.diff(x), np.diff(y)
    try:
        sel = ar_select_order(dx, maxlag=min(8, len(dx) // 10), ic='aic', trend='c'); lags = sel.ar_lags or []
    except Exception:
        lags = []
    if lags:
        res = sm.tsa.AutoReg(dx, lags=lags, trend='c').fit(); phi = np.zeros(max(lags) + 1); phi[0] = 1
        for l, cf in zip(lags, res.params[1:]): phi[l] = -cf
        dx, dy = lfilter(phi, [1], dx)[max(lags):], lfilter(phi, [1], dy)[max(lags):]
    return stats.pearsonr(dx, dy)[1]


def dcca_rho(x, y, s):
    X, Y = np.cumsum(x - x.mean()), np.cumsum(y - y.mean()); n = len(x); t = np.arange(s + 1)
    fxx = fyy = fxy = 0.0
    for i in range(0, n - s, max(1, s // 2)):
        xs, ys = X[i:i + s + 1], Y[i:i + s + 1]
        ex = xs - np.polyval(np.polyfit(t, xs, 1), t); ey = ys - np.polyval(np.polyfit(t, ys, 1), t)
        fxx += np.sum(ex * ex); fyy += np.sum(ey * ey); fxy += np.sum(ex * ey)
    return fxy / np.sqrt(fxx * fyy + 1e-30)


def ft_surrogate(x, rg):
    F = np.fft.rfft(x - x.mean()); ph = np.exp(1j * rg.uniform(0, 2 * np.pi, len(F))); ph[0] = 1
    if len(x) % 2 == 0: ph[-1] = 1
    return np.fft.irfft(F * ph, n=len(x)) + x.mean()


def dcca_p(x, y, rg, S=299):
    s = max(8, len(x) // 10); r0 = abs(dcca_rho(x, y, s))
    null = [abs(dcca_rho(ft_surrogate(x, rg), y, s)) for _ in range(S)]
    return (1 + sum(v >= r0 for v in null)) / (S + 1)


def hac_p(rx, ry):
    L = int(np.floor(4 * (len(rx) / 100) ** (2 / 9)))
    return float(sm.OLS(ry, sm.add_constant(rx)).fit(cov_type='HAC', cov_kwds=dict(maxlags=L)).pvalues[1])


def run_rep(cell, k, y, seed, with_shap=True, c_lag=0.8, types=TYPES):
    rg = np.random.default_rng(seed); X = plant(k, y, rg, c_lag)[types]; n = len(k); out = []
    _, ry = T.crossfit_trend(k, y)
    m1 = {c: abs(stats.pearsonr(X[c], y)[0]) >= 0.3 and stats.pearsonr(X[c], y)[1] < 0.05 for c in types}
    if with_shap:
        import shap
        from sklearn.ensemble import GradientBoostingRegressor
        idx = rg.permutation(n); tr = idx[:int(0.8 * n)]
        mdl = GradientBoostingRegressor(n_estimators=150, max_depth=3, learning_rate=0.05, random_state=0).fit(X.iloc[tr], y[tr])
        sv = np.mean(np.abs(shap.TreeExplainer(mdl).shap_values(X)), 0); sh = sv / sv.sum(); m2 = {c: sh[i] >= 0.05 for i, c in enumerate(types)}
    else:
        m2 = {c: np.nan for c in types}
    p3, p5, p6, p7 = {}, {}, {}, {}
    for c in types:
        x = X[c].values; _, rx = T.crossfit_trend(k, x)
        ok = np.std(rx) > 1e-9 * (np.std(x) + 1e-12)
        p3[c] = stats.spearmanr(rx, ry)[1] if ok else 1.0
        p5[c] = prewhiten_p(x, y) if np.std(np.diff(x)) > 0 else 1.0
        p6[c] = dcca_p(x, y, rg) if ok else 1.0
        p7[c] = hac_p(rx, ry) if ok else 1.0
    hol = lambda d: dict(zip(types, T.holm(np.array([d[c] for c in types]))))
    h3, h5, h6, h7 = hol(p3), hol(p5), hol(p6), hol(p7)
    card = T.run_audit(pd.concat([pd.DataFrame(dict(k=k, y=y)), X.reset_index(drop=True)], axis=1), 'k', 'y', types, B=199, seed=seed)
    cls = dict(zip(card.feature, card['class']))
    card1 = T.run_audit(pd.concat([pd.DataFrame(dict(k=k, y=y)), X.reset_index(drop=True)], axis=1), 'k', 'y', types, B=199, seed=seed, version=1)
    cls1 = dict(zip(card1.feature, card1['class']))
    for c in types:
        out.append(dict(target=cell, seed=seed, c_lag=c_lag, feature=c, truth=TRUTH[c], M1_corr=m1[c], M2_shap=m2[c], M3_detrended=h3[c] < .05,
                        M5_prewhiten=h5[c] < .05, M6_dcca=h6[c] < .05, M7_hac=h7[c] < .05, M4_TRACE=cls[c] == 'I', M4v1_TRACE=cls1[c] == 'I', TRACE_v1_class=cls1[c],
                        TRACE_class=cls[c], TRACE_leak=cls[c] in ('P', 'D')))
    return out


def targets():
    f = pd.read_csv('data/nasa_cycle_features.csv'); out = [(f'NASA B{int(b):04d}', h.cycle.values.astype(float), h.capacity.values, R_NASA) for b, h in f.groupby('battery')]
    E = pd.read_csv('outputs/out_ext_audit.csv').drop_duplicates('cell')
    import importlib.util
    spec = importlib.util.spec_from_file_location('ext', '11_external.py'); ext = importlib.util.module_from_spec(spec); spec.loader.exec_module(ext)
    D = ext.load()
    for ds in ['XJTU', 'TJU', 'HUST', 'MIT']:
        cs = E[E.dataset == ds].sort_values('n').cell.values; pick = [cs[len(cs) // 3], cs[2 * len(cs) // 3]]
        for c in pick:
            g = D[D.cell == c]; out.append((f'{ds} {c.split("/")[-1][:-4]}', g.k.values.astype(float), g.capacity.values, R_EXT))
    return out


if __name__ == '__main__':
    TG = targets()
    jobs = [(nm, k, y, 1000 * i + s, True, 0.8, fam) for i, (nm, k, y, R) in enumerate(TG) for s in range(R) for fam in FAM]
    rows = [r for rr in Parallel(n_jobs=2)(delayed(run_rep)(*j) for j in jobs) for r in rr]
    Dd = pd.DataFrame(rows); Dd.to_csv('outputs/out_bench2_raw.csv', index=False)
    M = ['M1_corr', 'M2_shap', 'M3_detrended', 'M5_prewhiten', 'M6_dcca', 'M7_hac', 'M4v1_TRACE', 'M4_TRACE']
    tab = Dd.groupby('feature')[M].mean().loc[TYPES]; tab['TRACE_leak_flag'] = Dd.groupby('feature').TRACE_leak.mean().loc[TYPES]
    tab.to_csv('outputs/out_bench2_table.csv'); print(tab.round(3))
    summ = {}
    pos = Dd.truth == 'I'
    for m in M:
        cr = Dd[m].astype(float)
        tp = cr[pos].sum(); fp = cr[~pos].sum()
        summ[m] = dict(recall=float(cr[pos].mean()), precision=float(tp / (tp + fp)) if tp + fp else 0.0,
                       false_credit=float(cr[~pos].mean()), false_credit_trend_null=float(cr[Dd.truth.isin(['T', 'N'])].mean()),
                       credit_leak=float(cr[Dd.truth.isin(['P', 'D'])].mean()),
                       recall_design=float(cr[Dd.feature.isin(DESIGN)].mean()), recall_heldout=float(cr[Dd.feature.isin(HELDOUT)].mean()),
                       recall_heldout_nolag=float(cr[Dd.feature.isin(['info_smooth', 'info_nonlin'])].mean()))
    summ['TRACE_leak_detection'] = float(Dd[Dd.truth.isin(['P', 'D'])].TRACE_leak.mean())
    summ['TRACE_v1_leak_detection'] = float(Dd[Dd.truth.isin(['P', 'D'])].TRACE_v1_class.isin(['P', 'D']).mean())
    summ['targets'] = [t[0] for t in TG]; summ['replicates'] = len(jobs) // 2
    json.dump(summ, open('outputs/out_bench2_summary.json', 'w'), indent=1); print(json.dumps({k: v for k, v in summ.items() if k != 'targets'}, indent=1))
    # (b) strength sweep on NASA targets
    nasa = [t for t in TG if t[0].startswith('NASA')]
    jobs = [(nm, k, y, 50000 + 1000 * i + 37 * j + s, False, c, ['info_smooth', 'null_trend']) for i, (nm, k, y, _) in enumerate(nasa)
            for j, c in enumerate([0.0, 0.2, 0.4, 0.6, 0.8, 1.2]) for s in range(R_SWEEP)]
    Sw = pd.DataFrame([r for rr in Parallel(n_jobs=2)(delayed(run_rep)(*j) for j in jobs) for r in rr])
    Sw.to_csv('outputs/out_bench2_sweep.csv', index=False)
    print(Sw.groupby(['c_lag', 'feature'])[['M5_prewhiten', 'M6_dcca', 'M7_hac', 'M4v1_TRACE', 'M4_TRACE']].mean().round(3))
    # (c) proxy-noise sweep
    pr = []
    for nm, k, y, _ in TG:
        for lev in [0.002, 0.01, 0.05, 0.1, 0.2, 0.5]:
            for s in range(10 if not QUICK else 3):
                rg = np.random.default_rng(s); x = y + rg.normal(0, lev * y.std(), len(y))
                card = T.run_audit(pd.DataFrame(dict(k=k, y=y, x=x)), 'k', 'y', ['x'], B=99, seed=s)
                pr.append(dict(target=nm, noise=lev, cls=card['class'][0], r_res=card.r_resid[0]))
    P = pd.DataFrame(pr); P.to_csv('outputs/out_bench2_proxy.csv', index=False)
    print(P.groupby('noise').cls.value_counts(normalize=True).unstack(fill_value=0).round(3))


In [ ]:
!python 12_benchmark_fair.py

## Step 13 – p-value floor, threshold and ζ sweeps, Ljung–Box gate size (V5, Supplementary Tables S2, S4)

In [ ]:
%%writefile 13_sensitivity.py
"""Step 13 - V5 (extended): resolution and threshold sensitivity.
(a) p-value floor: for each NASA cell, the smallest attainable Holm-adjusted time-shift p-value m/(2r+1), and which
    class-I verdicts sit at that floor. Sensitivity test with finer resolution: circular-shift p-value over all
    n - 1 non-zero circular shifts of the feature residual (full overlap; resolution 1/n), Holm-adjusted.
(b) Threshold sweep: reclassify the stored audit statistics of the 4 NASA cells and the 374 public cells
    under eta_det in {1e-4, 3e-4, 1e-3, 3e-3, 1e-2} and proxy threshold in {0.90, 0.95, 0.98, 0.99, 0.995};
    report the share of class assignments that change from the defaults (1e-3, 0.98).
(c) Ljung-Box exchangeability gate: rejection rate under AR(1) nulls (n = 168 and 3000; phi 0-0.99), i.e. how often
    the trend test is downgraded to descriptive."""
import sys, numpy as np, pandas as pd, warnings, os
from scipy import stats
warnings.filterwarnings('ignore'); sys.path.insert(0, '.'); import trace_audit as T
QUICK = os.environ.get('TRACE_QUICK') == '1'
f = pd.read_csv('data/nasa_cycle_features.csv'); A = pd.read_csv('outputs/out_audit_nasa.csv')
FE = [c for c in A.feature.unique()]

# (a)
rows = []
for b, h in f.groupby('battery'):
    k = h.cycle.values.astype(float); y = h.capacity.values; a = A[A.battery == f'B{int(b):04d}']
    tested = a[a['class'].isin(['I', 'T', 'N'])]; m = len(tested); r = int(a.r.iloc[0]); n = len(k)
    _, ry = T.crossfit_trend(k, y); pc = {}
    for fe in tested.feature:
        _, rx = T.crossfit_trend(k, h[fe].values.astype(float)); xr, yr = stats.rankdata(rx), stats.rankdata(ry)
        s0 = abs(np.corrcoef(xr, yr)[0, 1]); null = np.array([abs(np.corrcoef(np.roll(xr, d), yr)[0, 1]) for d in range(1, n)])
        pc[fe] = (1 + np.sum(null >= s0 - 1e-12)) / n
    hc = dict(zip(pc, T.holm(np.array(list(pc.values())))))
    for _, rw in tested.iterrows():
        rows.append(dict(battery=b, feature=rw.feature, n=n, r=r, m=m, p_floor_holm=m / (2 * r + 1), cls=rw['class'], p_tts_holm=rw.p_cond_holm,
                         at_floor=bool(abs(rw.p_cond_holm - m / (2 * r + 1)) < 1e-9), p_circ_holm=hc[rw.feature], cls_circ='I' if hc[rw.feature] <= .05 else '-'))
P = pd.DataFrame(rows); P.to_csv('outputs/out_pfloor_nasa.csv', index=False)
print(P[P.cls == 'I'][['battery', 'feature', 'p_floor_holm', 'p_tts_holm', 'at_floor', 'p_circ_holm']].round(4).to_string(index=False))
print('class-I verdicts at floor:', int(P[P.cls == 'I'].at_floor.sum()), 'of', int((P.cls == 'I').sum()),
      '| confirmed by circular-shift test (Holm <= 0.05):', int((P[P.cls == 'I'].p_circ_holm <= .05).sum()))

# (b)
E = pd.read_csv('outputs/out_ext_audit.csv'); N_ = A.copy(); N_['cell'] = N_.battery.astype(str); N_['eta_target'] = N_.eta_target
both = pd.concat([E.assign(src='public'), N_.assign(src='NASA')], ignore_index=True)


def reclass(d, eta_det, pthr, zthr=0.01):
    out = []
    for _, rw in d.iterrows():
        reason = str(rw.get('reason', '')) if pd.notna(rw.get('reason', np.nan)) else ''
        if reason in ('constant', 'affine in index or target'): out.append('D'); continue
        if pd.notna(rw.eta) and rw.eta < eta_det: out.append('D'); continue
        if pd.notna(rw.r_resid) and abs(rw.r_resid) >= pthr: out.append('P'); continue
        if pd.notna(rw.get('zeta', np.nan)) and rw.zeta < zthr: out.append('P'); continue
        if rw['class'] in ('D', 'P'):   # was leakage under defaults but not under the new threshold: test result needed
            out.append('I' if (pd.notna(rw.p_cond) and rw.p_cond * max(1, rw.get('m_eff', 5)) <= .05) else 'T'); continue
        out.append(rw['class'])
    return out


base = both['class'].values; sw = []
for eta in [1e-4, 3e-4, 1e-3, 3e-3, 1e-2]:
    for pt in [0.90, 0.95, 0.98, 0.99, 0.995]:
        c = np.array(reclass(both, eta, pt))
        sw.append(dict(eta_det=eta, proxy_thr=pt, changed=float(np.mean(c != base)),
                       changed_public=float(np.mean(c[both.src == 'public'] != base[both.src == 'public'])),
                       changed_nasa=float(np.mean(c[both.src == 'NASA'] != base[both.src == 'NASA'])),
                       n_P=int((c == 'P').sum()), n_D=int((c == 'D').sum())))
S = pd.DataFrame(sw); S.to_csv('outputs/out_threshold_sweep.csv', index=False); print(S.round(4).to_string(index=False))
zs = []
for z in [0.0025, 0.005, 0.01, 0.02, 0.05]:
    c = np.array(reclass(both, 1e-3, 0.98, z))
    zs.append(dict(zeta_thr=z, changed=float(np.mean(c != base)), n_P=int((c == 'P').sum()),
                   n_P_public=int((c[both.src == 'public'] == 'P').sum()), n_P_nasa=int((c[both.src == 'NASA'] == 'P').sum())))
Z = pd.DataFrame(zs); Z.to_csv('outputs/out_zeta_sweep.csv', index=False); print(Z.round(4).to_string(index=False))
te = pd.concat([E.drop_duplicates('cell')[['cell', 'eta_target']].assign(src='public'), N_.drop_duplicates('cell')[['cell', 'eta_target']].assign(src='NASA')])
M_ = pd.DataFrame([dict(eta_det=e, cells_target_deterministic=int((te.eta_target < e).sum()), of=len(te)) for e in [1e-4, 3e-4, 1e-3, 3e-3, 1e-2]])
M_.to_csv('outputs/out_target_eta_margin.csv', index=False); print(M_.to_string(index=False))
print('public cell target eta: min %.4f, 1st pct %.4f' % (E.drop_duplicates('cell').eta_target.min(), E.drop_duplicates('cell').eta_target.quantile(.01)))
print('feature |r_res| max among non-proxy public features: %.3f' % E[E['class'] != 'P'].r_resid.abs().max())

# (c) size of the exchangeability gate: Monte Carlo-calibrated (TRACE) vs the asymptotic chi-square reference
from statsmodels.stats.diagnostic import acorr_ljungbox
rng = np.random.default_rng(0); lb = []
for n in [168, 3000]:
    k = np.arange(1, n + 1, dtype=float); Bf = T._bspline_basis(k, int(max(3, min(12, n // 15))))
    for phi in [0.0, 0.3, 0.6, 0.9, 0.99]:
        rej = rej_asy = 0; R = 100 if QUICK else (1000 if n < 1000 else 300)
        for i_ in range(R):
            e = rng.normal(size=n + 200); x = np.zeros_like(e)
            for i in range(1, len(e)): x[i] = phi * x[i - 1] + e[i]
            x = x[200:]; res = x - Bf @ np.linalg.lstsq(Bf, x, rcond=None)[0]
            rej_asy += acorr_ljungbox(res, lags=[10], return_df=True)['lb_pvalue'].iloc[0] < 0.05
            rej += T.lb_gate(k, x, lags=10, B=499, seed=i_) < 0.05
        lb.append(dict(n=n, phi=phi, gate_rejects=rej / R, chi2_reference_rejects=rej_asy / R))
L = pd.DataFrame(lb); L.to_csv('outputs/out_lb_gate.csv', index=False); print(L.to_string(index=False))


In [ ]:
!python 13_sensitivity.py

## Step 14 – coordinates of every measured feature in the 3-D audit space (Fig. 10a)

In [ ]:
%%writefile 08_audit_space.py
"""Step 8 - coordinates of every MEASURED audited feature in TRACE's 3-D audit space
(log10 eta, |r_res|, beyond-index statistic S0) with its class, for Fig. 9a:
GMP columns, NASA features and the 16 charge features of the 374 public cells (no planted features).
Run after 01_audits.py and 11_external.py."""
import numpy as np, pandas as pd
rows = []
ag = pd.read_csv('outputs/out_audit_gmp.csv')
for _, w in ag.iterrows():
    if w.feature == 'PackID' or pd.isna(w.get('eta', np.nan)): continue
    rows.append(dict(source='GMP', feature=w.feature, eta=w.eta, r_res=abs(w.r_resid) if pd.notna(w.r_resid) else 0.0,
                     S0=w.S_cond if pd.notna(w.S_cond) else 0.0, cls=w['class']))
an = pd.read_csv('outputs/out_audit_nasa.csv')
for _, w in an.iterrows():
    if pd.isna(w.eta): continue
    rows.append(dict(source='NASA', feature=f'{w.battery}:{w.feature}', eta=w.eta, r_res=abs(w.r_resid), S0=w.S_cond, cls=w['class']))
E = pd.read_csv('outputs/out_ext_audit.csv')
for _, w in E.iterrows():
    if pd.isna(w.eta): continue
    rows.append(dict(source=w.dataset, feature=f'{w.cell}:{w.feature}', eta=w.eta, r_res=abs(w.r_resid), S0=w.S_cond, cls=w['class']))
D = pd.DataFrame(rows); D.to_csv('outputs/out_audit_space.csv', index=False)
print(len(D), 'measured features'); print(D.groupby('cls')[['eta', 'r_res', 'S0']].median().round(3)); print(D.cls.value_counts())


In [ ]:
!python 08_audit_space.py

## Step 15 – all data figures (Figs. 2–10)

In [ ]:
%%writefile 10_figures.py
"""All article figures. Readability rules: large bold text, no dark text on blue fills, boxes sized to their content."""
import numpy as np, pandas as pd, json, sys, os, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Rectangle
sys.path.insert(0, '.'); import trace_audit as T
os.makedirs('figures', exist_ok=True)

BLUE, ORANGE, GREEN, AMBER, PINK, DKGREEN = '#1f5fbf', '#d9541e', '#138a5e', '#c98a00', '#c2477d', '#006400'
C = [BLUE, ORANGE, GREEN, AMBER, PINK, DKGREEN]
INK, MUTED, GRID = '#111111', '#555555', '#dddddd'
NAVY = '#0b2a5b'
CLS = {'D': ('#bdbdb5', INK), 'P': ('#f5ae8c', INK), 'I': ('#a9c6f0', NAVY), 'T': ('#f3d58c', INK),
       'N': ('#ededea', INK), 'N*': ('#ededea', INK)}          # fill, text colour (dark bold text only; no white text)
from matplotlib.colors import LinearSegmentedColormap
LIGHT = LinearSegmentedColormap.from_list('lightYlOrBr', plt.get_cmap('YlOrBr')(np.linspace(0.0, 0.55, 256)))
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 12, 'font.weight': 'bold',
                     'axes.titleweight': 'bold', 'axes.labelweight': 'bold', 'axes.titlesize': 12.5,
                     'axes.labelsize': 12, 'xtick.labelsize': 11, 'ytick.labelsize': 11, 'legend.fontsize': 10.5,
                     'axes.edgecolor': INK, 'axes.labelcolor': INK, 'xtick.color': INK, 'ytick.color': INK,
                     'axes.linewidth': 1.1, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.7, 'lines.linewidth': 2.4,
                     'legend.frameon': False, 'savefig.dpi': 300, 'savefig.bbox': 'tight', 'savefig.pad_inches': 0.04})
import textwrap
def title(a, t):
    """Panel label only: the caption carries the figure title and the panel descriptions (Elsevier artwork rules)."""
    lab = t.split(')')[0] + ')' if t.startswith('(') else ''
    a.set_title(lab, loc='left', fontsize=15, weight='bold', pad=6)


def legible(fig, print_w=6.7, min_pt=7.0):
    """Raise every text element to >= min_pt points at the printed width (17 cm full-page width)."""
    import matplotlib.text as mtext
    need = min_pt * fig.get_figwidth() / print_w
    for t in fig.findobj(mtext.Text):
        if t.get_text() and t.get_fontsize() < need:
            t.set_fontsize(need)

g = pd.read_csv('data/GMP.csv'); f = pd.read_csv('data/nasa_cycle_features.csv')
b5 = f[f.battery == 5]; kk = b5.cycle.values.astype(float)
V = json.load(open('outputs/out_validation.json'))

pt = pd.read_csv('outputs/out_power_trend_gmp.csv')
L = json.load(open('outputs/out_shift_landscape_B0005.json'))
G = pd.read_csv('outputs/out_shap_gmp.csv'); ops = G[~G.feature.str.startswith('NULL_') & (G.feature != 'CycleNumber')]; nb = G[G.feature.str.startswith('NULL_')]
# =====================================================================  Fig. 2 provenance
fig, axs = plt.subplots(1, 3, figsize=(13, 4.0), gridspec_kw=dict(wspace=0.38))
axs[0].plot(g.CycleNumber, g['Capacity(Ah)'], color=BLUE, lw=3); axs[0].set_xlabel('Cycle'); axs[0].set_ylabel('Capacity (Ah)')
title(axs[0], '(a) GMP capacity follows a fixed power law')
axs[1].plot(b5.cycle, b5.capacity, color=ORANGE, lw=2.2, label='measured')
trn, _ = T.crossfit_trend(kk, b5.capacity.values); axs[1].plot(b5.cycle, trn, color=INK, lw=1.8, ls='--', label='cross-fitted trend')
axs[1].set_xlabel('Cycle'); axs[1].set_ylabel('Capacity (Ah)'); axs[1].legend(loc='upper right'); title(axs[1], '(b) NASA B0005 capacity scatters around its trend')
k = g.CycleNumber.values.astype(float); cols = ['Capacity(Ah)', 'Voltage(V)', 'Temperature(C)', 'Cell1_V']
eg = [T.determinism_index(k, g[c].values) for c in cols]
en = [T.determinism_index(kk, b5[c].values) for c in ['capacity', 'V_600s', 'T_max', 'T_start']]
xx = np.arange(4); axs[2].bar(xx - 0.2, eg, 0.38, color=BLUE, label='GMP'); axs[2].bar(xx + 0.2, en, 0.38, color=ORANGE, label='NASA B0005')
axs[2].set_yscale('log'); axs[2].axhline(1e-3, color=INK, lw=1.6, ls='--'); axs[2].text(-0.45, 1.5e-3, 'η = 10⁻³', fontsize=11, weight='bold')
axs[2].set_xticks(xx); axs[2].set_xticklabels(['cap.', 'volt.', 'temp.', 'cell/T₀']); axs[2].set_ylabel('determinism index η')
axs[2].legend(loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=2); title(axs[2], '(c) η separates derived from measured columns')
legible(fig); fig.savefig('figures/fig2_provenance.png'); plt.close(fig)

# =====================================================================  Fig. 3 false-positive control
ar = pd.read_csv('outputs/out_typeI_ar.csv'); cn = pd.read_csv('outputs/out_typeI_cond_nasa.csv')
fig, axs = plt.subplots(1, 3, figsize=(13, 4.0), gridspec_kw=dict(wspace=0.32))
ch = ['V', 'I', 'T'] + [f'C{i}' for i in range(1, 9)] + ['Imb']
axs[0].bar(np.arange(12), V['V2a_trend_FPR'], color=BLUE, width=0.72); axs[0].axhline(0.05, color=INK, lw=1.6, ls='--')
axs[0].set_ylim(0, 0.1); axs[0].set_xticks(np.arange(12)); axs[0].set_xticklabels(ch, rotation=90, fontsize=10.5)
axs[0].set_ylabel('false-positive rate'); title(axs[0], '(a) The trend test stays near 5% on GMP nulls')
for a, df, t in [(axs[1], ar, '(b) The shift test stays near 5% as autocorrelation grows (n = 3000)'), (axs[2], cn.groupby('phi')[['FPR_TTS', 'FPR_naive']].mean().reset_index(), '(c) The beyond-index test stays near 5% on NASA residuals')]:
    a.plot(df.phi.astype(str), df.FPR_naive, 's-', color=ORANGE, ms=8, label='naive Spearman')
    a.plot(df.phi.astype(str), df.FPR_TTS, 'o-', color=BLUE, ms=8, label='time-shift (TRACE)')
    a.axhline(0.05, color=INK, lw=1.6, ls='--'); a.set_ylim(0, 1); a.set_xlabel('AR(1) coefficient φ'); title(a, t)
axs[1].legend(loc='center right')
legible(fig); fig.savefig('figures/fig3_typeI.png'); plt.close(fig)

# =====================================================================  Fig. 4 power
pc = pd.read_csv('outputs/out_power_cond_nasa.csv')
fig, axs = plt.subplots(1, 2, figsize=(12, 4.2), gridspec_kw=dict(wspace=0.25))
for i, kd in enumerate(['generator', 'self-surrogate']):
    q = pt[pt.kind == kd]; axs[0].plot(q.rho, q.power_family12, ['o-', 's--'][i], color=C[i], ms=7, label=['true generator', 'self-surrogate'][i])
axs[0].axhline(0.8, color=INK, lw=1.6, ls=':'); axs[0].set_xlabel('injected trend coupling ρ'); axs[0].set_ylabel('power (family of 12)')
axs[0].set_xticks([0, 0.05, 0.1, 0.15, 0.2]); axs[0].legend(loc='upper left'); title(axs[0], '(a) Self-surrogate power matches true-generator power on GMP')
for i, (b, q) in enumerate(pc.groupby('battery')):
    axs[1].plot(q.rho, q.power_family5, 'o-', color=C[i], ms=6, label=f'B{b:04d}')
axs[1].axhline(0.8, color=INK, lw=1.6, ls=':'); axs[1].set_xlabel('injected beyond-index coupling ρ'); axs[1].set_ylabel('power (family of 5)')
axs[1].legend(loc='upper left'); title(axs[1], '(b) NASA power rises with beyond-index coupling in every cell')
legible(fig); fig.savefig('figures/fig4_power.png'); plt.close(fig)

# =====================================================================  Fig. 6 NASA evidence (classes are given in Table 5 only)
fig, ax = plt.subplots(figsize=(8.5, 3.9))
_, ry = T.crossfit_trend(kk, b5.capacity.values); _, rv = T.crossfit_trend(kk, b5.V_600s.values)
ax.plot(kk, ry / ry.std(), color=ORANGE, lw=2.2, label='capacity residual')
ax.plot(kk, rv / rv.std(), color=BLUE, lw=2.0, ls='--', label='V₆₀₀ residual')
ax.set_xlabel('Cycle'); ax.set_ylabel('standardised residual'); ax.legend(loc='upper right', ncol=2)
legible(fig); fig.savefig('figures/fig6_nasa.png'); plt.close(fig)

# =====================================================================  Fig. 5 reliability landscape
fig = plt.figure(figsize=(13, 9.2)); gs = fig.add_gridspec(2, 3, hspace=0.5, wspace=0.42)
Ls = pd.read_csv('outputs/out_mde_surface.csv'); ns = sorted(Ls.n.unique()); ph = sorted(Ls.phi.unique())
Mm = Ls.pivot(index='phi', columns='n', values='MDE80').loc[ph, ns].values
a = fig.add_subplot(gs[0, 0:2]); a.grid(False)
im = a.imshow(np.where(np.isnan(Mm), 1.0, Mm), cmap=LIGHT, vmin=0.1, vmax=1.0, aspect='auto', origin='lower')
for i in range(len(ph)):
    for j in range(len(ns)):
        v = Mm[i, j]; val = 1.0 if np.isnan(v) else v
        a.text(j, i, 'n/r' if np.isnan(v) else f'{v:.2f}', ha='center', va='center', fontsize=12.5, weight='bold', color=INK)
a.set_xticks(range(len(ns))); a.set_xticklabels(ns); a.set_yticks(range(len(ph))); a.set_yticklabels(ph)
a.set_xlabel('record length n (cycles)'); a.set_ylabel('feature autocorrelation φ')
title(a, '(a) MDE₈₀ of the beyond-index test falls as records lengthen (m = 5; n/r = not resolvable)')
cb = fig.colorbar(im, ax=a, fraction=0.035, pad=0.015); cb.ax.tick_params(labelsize=10)
nul = pd.read_csv('outputs/out_null_pvalues.csv'); u = np.linspace(0, 1, 201)
a = fig.add_subplot(gs[0, 2])
a.plot(u, [np.mean(nul.p_naive <= z) for z in u], color=ORANGE, label='naive Spearman')
a.plot(u, [np.mean(nul.p_tts <= z) for z in u], color=BLUE, label='time-shift')
a.plot([0, 1], [0, 1], color=INK, lw=1.4, ls=':'); a.set_xlabel('nominal α'); a.set_ylabel('rejection rate under H₀')
a.legend(loc='lower right'); title(a, '(b) Null p-values follow the diagonal (n = 168, φ = 0.9)')
a = fig.add_subplot(gs[1, 0:2]); d = np.arange(-67, 68)
lab = {'V_600s': 'V₆₀₀ (I)', 'V_mean': 'V̄ (I)', 'T_start': 'T₀ (I)', 'T_max': 'T_max (I)', 'I_mean': 'Ī (T)'}
for i, c in enumerate(['I_mean', 'T_max', 'T_start', 'V_mean', 'V_600s']):
    a.plot(d, L[c], color=[PINK, AMBER, GREEN, ORANGE, BLUE][i], lw=[1.6, 1.6, 1.6, 2.0, 2.6][i], label=lab[c])
a.axvline(0, color=INK, lw=1.4, ls='--'); a.set_xlabel('time shift d (cycles)'); a.set_ylabel('|ρ_S| of residuals')
a.legend(ncol=5, loc='upper center', bbox_to_anchor=(0.5, -0.2)); title(a, '(c) The observed alignment is compared with every truncated shift (NASA B0005, r = 67)')
a = fig.add_subplot(gs[1, 2])
for i, p_ in enumerate(ph):
    q = Ls[Ls.phi == p_]; a.plot(q.n, q.MDE80, 'o-', color=C[i], ms=7, label=f'φ = {p_}')
a.set_xscale('log'); a.set_xlabel('n (cycles)'); a.set_ylabel('MDE₈₀'); a.set_ylim(0, 0.8)
a.axvspan(90, 130, color='#bdbdb5', alpha=0.6, lw=0); a.text(93, 0.04, 'n/r', fontsize=11, weight='bold')
a.legend(loc='upper right'); title(a, '(d) Power increases with record length n')
legible(fig); fig.savefig('figures/fig5_reliability_landscape.png'); plt.close(fig)

# =====================================================================  Fig. 7 consequences: XAI and generalizability
fig = plt.figure(figsize=(13, 9.4)); gs = fig.add_gridspec(2, 3, hspace=0.5, wspace=0.55)
Rall = pd.read_csv('outputs/out_loco.csv'); order = ['Index only', 'Trend-only (T)', 'TRACE-admitted (I)', 'All features', 'Proxy only (P)']
Mg = Rall[Rall.model == 'GBR'].pivot_table(index='feature_set', columns='held_out', values='R2').loc[order]
a = fig.add_subplot(gs[0, 0:2]); a.grid(False)
im = a.imshow(np.clip(Mg.values, 0, 1), cmap=LIGHT, vmin=0, vmax=1, aspect='auto')
for i in range(Mg.shape[0]):
    for j in range(Mg.shape[1]):
        v = Mg.values[i, j]; a.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=13, weight='bold', color=INK)
a.set_xticks(range(4)); a.set_xticklabels(Mg.columns); a.set_yticks(range(len(order))); a.set_yticklabels(order)
a.add_patch(Rectangle((-0.5, 2.5), 4, 2, fill=False, ec=BLUE, lw=3, ls='--'))
a.text(3.6, 3.5, 'leak-\ninflated', fontsize=11.5, weight='bold', color=BLUE, va='center')
title(a, '(a) Leave-one-cell-out R² shows that admitted features transfer best without leakage')
mR = Rall.groupby(['feature_set', 'model']).RMSE.mean().unstack().loc[order]
a = fig.add_subplot(gs[0, 2]); y = np.arange(len(order))
a.barh(y - 0.2, mR.GBR * 1000, 0.38, color=BLUE, label='GBR'); a.barh(y + 0.2, mR.Ridge * 1000, 0.38, color=AMBER, label='Ridge')
a.set_yticks(y); a.set_yticklabels(['k', 'T', 'I', 'all', 'P']); a.invert_yaxis(); a.set_xlabel('mean RMSE (mAh)')
a.legend(loc='lower right'); title(a, '(b) With gradient boosting, admitted features give the lowest non-leaking RMSE')
S = pd.read_csv('outputs/out_shap_nasa.csv')
for gi, (fs, t) in enumerate([('All features', '(c) With all features, proxies dominate SHAP'), ('TRACE-admitted (I)', '(d) After TRACE, V_600s dominates SHAP')]):
    a = fig.add_subplot(gs[1, gi]); s_ = S[S.feature_set == fs].groupby('feature').mean_abs_shap.mean().sort_values()
    cc = [ORANGE if c in ('t_discharge', 't_Tmax') else BLUE if c in ('V_600s', 'V_mean') else '#8c8c86' for c in s_.index]
    a.barh(range(len(s_)), s_.values * 1000, color=cc, height=0.7); a.set_yticks(range(len(s_))); a.set_yticklabels(s_.index)
    a.set_xlabel('mean |SHAP| (mAh)'); title(a, t)
a = fig.add_subplot(gs[1, 2]); rg = np.random.default_rng(3)
a.scatter(rg.uniform(-.18, .18, len(ops)), ops.mean_abs_shap * 1e3, s=60, color=BLUE, label='real channels')
a.scatter(1 + rg.uniform(-.18, .18, len(nb)), nb.mean_abs_shap * 1e3, s=60, color=ORANGE, marker='s', label='shuffled copies')
a.set_xticks([0, 1]); a.set_xticklabels(['real', 'null']); a.set_xlim(-.55, 1.55); a.set_ylabel('mean |SHAP| (mAh)')
cj = json.load(open('outputs/out_consequence.json')); title(a, f"(e) GMP channel SHAP is indistinguishable from the shuffled null (p = {cj['gmp_shap_mw_p']:.2f})")
legible(fig); fig.savefig('figures/fig9_xai_generalization.png'); plt.close(fig)

# =====================================================================  Fig. 8 fair benchmark
tab = pd.read_csv('outputs/out_bench2_table.csv', index_col=0); sm = json.load(open('outputs/out_bench2_summary.json'))
meth = ['M1_corr', 'M2_shap', 'M3_detrended', 'M5_prewhiten', 'M6_dcca', 'M7_hac', 'M4v1_TRACE', 'M4_TRACE']
mlab = ['Correlation screen', 'SHAP screen', 'Detrended corr. (iid p)', 'Pre-whitened CCF', 'DCCA + surrogates', 'HAC regression', 'TRACE v1', 'TRACE v2']
types = ['info_res', 'info_rate', 'info_smooth', 'info_nonlin', 'info_lag', 'trend_hi', 'null_ar', 'null_trend', 'proxy', 'label']
tlab = ['residual*', 'rate*', 'smoothed', 'nonlinear', 'lag-1', 'trend HI', 'AR null', 'trend null', 'proxy', 'label']
fig = plt.figure(figsize=(14, 10.2)); gs = fig.add_gridspec(2, 2, height_ratios=[1.15, 1], hspace=0.52, wspace=0.34)
a = fig.add_subplot(gs[0, :]); a.grid(False); M_ = tab.loc[types, meth].T.values.astype(float)
a.imshow(M_, cmap=LIGHT, vmin=0, vmax=1, aspect='auto')
for i in range(M_.shape[0]):
    for j in range(M_.shape[1]): a.text(j, i, f'{M_[i, j]:.2f}', ha='center', va='center', fontsize=10.5, weight='bold', color=INK)
a.set_xticks(range(len(types))); a.set_xticklabels(tlab, fontsize=11, rotation=30, ha='right'); a.set_yticks(range(len(meth))); a.set_yticklabels(mlab, fontsize=11)
for xv in [4.5, 7.5]: a.axvline(xv, color=INK, lw=2.5)
tr_ = a.get_xaxis_transform()
for xx_, tt_ in [(2.0, 'informative (ideal 1)'), (6.0, 'not informative\n(ideal 0)'), (8.5, 'leakage\n(ideal 0)')]:
    a.text(xx_, -0.33, tt_, ha='center', va='top', fontsize=11.5, weight='bold', transform=tr_)
title(a, '(a) Share of 360 replicates in which each rule credits a planted feature (12 measured targets; * = design types)')
a = fig.add_subplot(gs[1, 0]); keys = [('precision', 'precision'), ('recall_heldout_nolag', 'recall, held-out'), ('false_credit_trend_null', 'false credit, trend/null')]
x = np.arange(len(meth)); w = 0.27
for i, (kk_, lb_) in enumerate(keys):
    a.bar(x + (i - 1) * w, [sm[m][kk_] for m in meth], w, color=[BLUE, GREEN, ORANGE][i], label=lb_)
a.set_xticks(x); a.set_xticklabels(['Corr', 'SHAP', 'Detr', 'PreW', 'DCCA', 'HAC', 'v1', 'v2'], fontsize=10.5, rotation=45, ha='right'); a.set_ylim(0, 1.05)
a.legend(loc='upper center', bbox_to_anchor=(0.5, -0.25), ncol=1, fontsize=10)
title(a, '(b) Precision, held-out recall and false credit of each rule')
a = fig.add_subplot(gs[1, 1]); Sw = pd.read_csv('outputs/out_bench2_sweep.csv')
for m, col, lab_ in [('M7_hac', AMBER, 'HAC'), ('M5_prewhiten', PINK, 'Pre-whitened'), ('M4v1_TRACE', '#8c8c86', 'TRACE v1'), ('M4_TRACE', BLUE, 'TRACE v2')]:
    pw = Sw[Sw.feature == 'info_smooth'].groupby('c_lag')[m].mean(); fpr = Sw[Sw.feature == 'null_trend'].groupby('c_lag')[m].mean().mean()
    a.plot(pw.index, pw.values, color=col, marker='o', lw=2.6, label=f'{lab_} (null credit {fpr:.2f})')
a.axhline(0.05, color=INK, ls='--', lw=1.4); a.set_xlabel('coupling strength c (held-out smoothed type)'); a.set_ylabel('share credited'); a.set_ylim(0, 1.02)
a.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=2, fontsize=9.8); title(a, '(c) Detection rises with coupling strength (NASA targets, m = 2)')
legible(fig); fig.savefig('figures/fig8_benchmark.png'); plt.close(fig)

# =====================================================================  Fig. 10 external validation on 374 public cells
E = pd.read_csv('outputs/out_ext_audit.csv'); Lx = pd.read_csv('outputs/out_ext_loco_summary.csv')
fig = plt.figure(figsize=(14, 5.6)); gs = fig.add_gridspec(1, 2, width_ratios=[1.25, 1], wspace=0.42)
a = fig.add_subplot(gs[0, 0]); a.grid(False)
sh = E.pivot_table(index='feature', columns='class', values='cell', aggfunc='count', fill_value=0)
for c_ in ['I', 'T', 'N', 'P', 'D']:
    if c_ not in sh: sh[c_] = 0
sh = (sh[['I', 'T', 'N', 'P']].T / sh.sum(1)).T.sort_values('I', ascending=False)
im = a.imshow(sh.values, cmap=LIGHT, vmin=0, vmax=1, aspect='auto')
for i in range(sh.shape[0]):
    for j in range(sh.shape[1]): a.text(j, i, f'{sh.values[i, j]:.2f}', ha='center', va='center', fontsize=10, weight='bold', color=INK)
a.set_xticks(range(4)); a.set_xticklabels(['I', 'T', 'N', 'P']); a.set_yticks(range(len(sh))); a.set_yticklabels(sh.index, fontsize=10.5)
title(a, f'(a) Class shares of 16 charge features audited cell by cell ({E.cell.nunique()} cells)')
a = fig.add_subplot(gs[0, 1]); ds = ['XJTU', 'TJU', 'HUST', 'MIT']; sets = ['index only', 'index + I', 'index + T', 'index + all non-leaking']
cols_ = ['#8c8c86', BLUE, AMBER, GREEN]; x = np.arange(len(ds)); w = 0.2
for i, (s_, c_) in enumerate(zip(sets, cols_)):
    q = Lx[Lx.set == s_].set_index('dataset').loc[ds]
    a.bar(x + (i - 1.5) * w, q.median_rmse_pct, w, color=c_, label=s_, yerr=[q.median_rmse_pct - q.lo, q.hi - q.median_rmse_pct], capsize=2, error_kw=dict(lw=1))
a.set_xticks(x); a.set_xticklabels(ds); a.set_ylabel('median LOCO RMSE (% of initial capacity)')
a.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=2, fontsize=10)
title(a, '(b) Across cells, features TRACE cannot credit on one trajectory (T) carry most transferable information')
legible(fig); fig.savefig('figures/fig7_external.png'); plt.close(fig)
print('figures done')

# =====================================================================  Fig. 9 3-D audit space and 3-D power surface
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from matplotlib import cm
A3 = pd.read_csv('outputs/out_audit_space.csv')
A3['leta'] = np.log10(A3.eta.clip(lower=1e-6))
fig = plt.figure(figsize=(16, 7.2))
a = fig.add_axes([0.0, 0.0, 0.46, 0.92], projection='3d')
style = {'D': ('#6e6e66', 'X', 'D derived'), 'P': (ORANGE, 's', 'P proxy'), 'I': (BLUE, 'o', 'I informative'),
         'T': (AMBER, '^', 'T trend-only'), 'N': ('#9a9a92', 'v', 'N null'), 'N*': ('#2b2b2b', 'D', 'N* untestable')}
for cl, (col, mk, lab) in style.items():
    q = A3[A3.cls == cl]
    if len(q) == 0: continue
    big = q.source.isin(['GMP', 'NASA'])
    a.scatter(q.leta[~big], q.r_res[~big], q.S0[~big], c=col, marker=mk, s=22, alpha=0.55, depthshade=False, edgecolors='none')
    a.scatter(q.leta[big], q.r_res[big], q.S0[big], c=col, marker=mk, s=95, alpha=1.0, depthshade=False, edgecolors='black', linewidths=0.8, label=lab)
a.set_xlabel('log₁₀ η', labelpad=10); a.set_ylabel('|r_res|', labelpad=10); a.set_zlabel('S₀ (beyond-index)', labelpad=10)
a.set_xlim(-6, 0.3); a.set_ylim(0, 1); a.set_zlim(0, 1); a.view_init(elev=22, azim=-58)
a.tick_params(labelsize=10)
a.legend(loc='upper left', bbox_to_anchor=(-0.05, 1.02), fontsize=10.5, ncol=2, markerscale=0.9)
a.set_title('(a)', loc='left', fontsize=15, weight='bold', pad=0)
# (b) power surface
Ls = pd.read_csv('outputs/out_mde_surface.csv'); rh = np.round(np.arange(0, 1.0, 0.1), 1)
q = Ls[Ls.phi == 0.8].sort_values('n'); ns_ = q.n.values
Z = q[[f'p{r_:.1f}' for r_ in rh]].values
XX, YY = np.meshgrid(rh, np.log10(ns_))
b_ = fig.add_axes([0.50, 0.0, 0.42, 0.92], projection='3d')
surf = b_.plot_surface(XX, YY, Z, cmap='YlOrBr', vmin=0, vmax=1, edgecolor='#333333', linewidth=0.4, antialiased=True, alpha=0.95)
_cs = plt.figure().add_subplot().contour(XX, YY, Z, levels=[0.8]); plt.close()
for _seg in _cs.allsegs[0]:
    b_.plot(_seg[:, 0], _seg[:, 1], np.full(len(_seg), 0.82), color=BLUE, lw=5, zorder=10)
b_.set_xlabel('coupling ρ', labelpad=10); b_.set_ylabel('record length n', labelpad=10); b_.set_zlabel('power', labelpad=8)
b_.set_yticks(np.log10(ns_)); b_.set_yticklabels([str(int(v)) for v in ns_], fontsize=9.5); b_.set_zlim(0, 1)
b_.view_init(elev=26, azim=-130); b_.tick_params(labelsize=10)
b_.text2D(0.03, 0.9, 'blue line: 80 % power contour\n(MDE₈₀ as a function of n)', transform=b_.transAxes, fontsize=10.5, weight='bold', color=BLUE)
b_.set_title('(b)', loc='left', fontsize=15, weight='bold', pad=0)
cax = fig.add_axes([0.93, 0.25, 0.012, 0.45]); cb = fig.colorbar(surf, cax=cax); cb.set_label('power', fontsize=11, weight='bold')
legible(fig); fig.savefig('figures/fig10_audit_space_3d.png'); plt.close(fig)
print('fig9 done')


In [ ]:
!python 10_figures.py

## Step 16 – methodology figure (Fig. 1)

In [ ]:
%%writefile extra_methodology_figure.py
"""Fig. 1 methodology figure: twelve numbered step panels with icon-style graphics (no axes) and no connector lines.
Drawn schematic; reads pipeline outputs only for the icon shapes."""
import numpy as np, pandas as pd, json, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Rectangle, Circle, Ellipse, Polygon
BLUE, ORANGE, GREEN, AMBER, PINK = '#1f5fbf', '#d9541e', '#138a5e', '#c98a00', '#c2477d'
TINT = {'#1f5fbf': '#a9c6f0', '#d9541e': '#f5ae8c', '#138a5e': '#9fd6b8', '#c98a00': '#f3d58c', '#c2477d': '#eeb3cc'}
INK = '#111111'
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.weight': 'bold', 'savefig.dpi': 300})
g = pd.read_csv('data/GMP.csv'); f = pd.read_csv('data/nasa_cycle_features.csv')
L = json.load(open('outputs/out_shift_landscape_B0005.json'))
pt = pd.read_csv('outputs/out_power_trend_gmp.csv'); nul = pd.read_csv('outputs/out_null_pvalues.csv')

W, H = 160, 100
fig = plt.figure(figsize=(16, 10.0)); ax = fig.add_axes([0, 0, 1, 1]); ax.set_xlim(0, W); ax.set_ylim(0, H); ax.axis('off')
PW, PH = 37.5, 30; GX, GY = 2.0, 2.4; X0, Y0 = 1.5, 1.2
def pos(i):  # i = 0..11, row-major from top
    r, c = divmod(i, 4); return X0 + c * (PW + GX), Y0 + (2 - r) * (PH + GY)
def icon_ax(x, y, w, h):
    a = fig.add_axes([x / W, y / H, w / W, h / H]); a.axis('off'); return a
def panel(i, title, tags, fill):
    x, y = pos(i)
    ax.add_patch(FancyBboxPatch((x, y), PW, PH, boxstyle='round,pad=0.2,rounding_size=1.6', fc='white', ec=INK, lw=1.8))
    ax.add_patch(Circle((x + 3.6, y + PH - 3.6), 2.7, fc=TINT.get(fill, fill), ec=fill, lw=2.5))
    ax.text(x + 3.6, y + PH - 3.6, str(i + 1), ha='center', va='center', color=INK, fontsize=15, weight='bold')
    ax.text(x + 7.4, y + PH - 3.6, title, ha='left', va='center', fontsize=13.2, weight='bold', color=INK)
    for j, t in enumerate(tags):  # tag chips at the bottom, sized to text
        ax.text(x + PW / 2, y + 2.4 + (len(tags) - 1 - j) * 4.1, t, ha='center', va='center', fontsize=10.6, weight='bold', color=INK,
                bbox=dict(boxstyle='round,pad=0.25', fc='#eef0e6', ec='none'))
    return x, y
STAGE = {0: BLUE, 1: BLUE, 2: BLUE, 3: GREEN, 4: GREEN, 5: GREEN, 6: GREEN, 7: ORANGE, 8: ORANGE, 9: ORANGE, 10: ORANGE, 11: ORANGE}

# 1 GMP pack
x, y = panel(0, 'Industrial LFP pack', ['GMP pack, 25.6 V, 40 Ah', '3000 cycles × 17 columns'], STAGE[0])
ax.add_patch(FancyBboxPatch((x + 7, y + 10.8), 21, 11, boxstyle='round,pad=0.2,rounding_size=1', fc='white', ec=INK, lw=2.2))
ax.add_patch(Rectangle((x + 28.4, y + 14.0), 2, 4.6, fc=INK))
for k, c in enumerate([GREEN, GREEN, GREEN, AMBER]): ax.add_patch(Rectangle((x + 8.3 + k * 4.9, y + 12.0), 4.1, 8.6, fc=c))
# 2 generator check
x, y = panel(1, 'Provenance check', ['published Eqs. 1–8, seed 42', 'CSV reproduced to 5×10⁻⁹'], STAGE[1])
a = icon_ax(x + 6, y + 10.0, PW - 12, 13); n = g.CycleNumber.values
a.plot(n, g['Capacity(Ah)'], color=BLUE, lw=6); a.plot(n, 40 - 12 * (n / 3000) ** 1.35, color=ORANGE, lw=2.5, ls='--')
# 3 NASA cells
x, y = panel(2, 'Measured control', ['NASA B0005/6/7/18', '+ 374 public cells (4 labs)'], STAGE[2])
for k, c in enumerate([BLUE, ORANGE, GREEN, AMBER]):
    cx = x + 8 + k * 6.4
    ax.add_patch(FancyBboxPatch((cx, y + 10.0), 4.4, 12.6, boxstyle='round,pad=0.1,rounding_size=0.6', fc=TINT[c], ec=c, lw=2.5))
    ax.add_patch(Rectangle((cx + 1.4, y + 22.7), 1.6, 1.0, fc=c))
    ax.text(cx + 2.2, y + 16.3, ['5', '6', '7', '18'][k], ha='center', va='center', color=INK, fontsize=12, weight='bold', rotation=90)
# 4 cycle table
x, y = panel(3, 'Cycle-level table', ['index k, target y, features x', 'one row per cycle'], STAGE[3])
for r_ in range(4):
    for c_ in range(5):
        ax.add_patch(Rectangle((x + 7 + c_ * 4.8, y + 9.8 + r_ * 3.4), 4.6, 3.2, fc=['#d7e4f7', '#fde0cf', '#e0f0e6', '#e0f0e6', '#e0f0e6'][c_] if r_ < 3 else ['#a9c6f0', '#f5ae8c', '#9fd6b8', '#9fd6b8', '#9fd6b8'][c_], ec='white', lw=1.2))
for c_, t in enumerate(['k', 'y', 'x₁', 'x₂', 'xₚ']): ax.text(x + 9.3 + c_ * 4.8, y + 21.4, t, ha='center', va='center', color=INK, fontsize=12, weight='bold')
# 5 S1
x, y = panel(4, 'S1 Provenance', ['determinism index η', 'proxy check: r_res, ζ'], STAGE[4])
a = icon_ax(x + 7, y + 9.8, PW - 14, 13)
a.bar(range(6), np.log10([7.5e-5, 1, 1, 0.067, 0.095, 0.78]) + 5, color=[BLUE] * 3 + [ORANGE] * 3, width=0.78)
# 6 S2
x, y = panel(5, 'S2 Index ceiling', ['y predicted from k alone', 'chronological 70/30 split'], STAGE[5])
a = icon_ax(x + 6, y + 10.0, PW - 12, 13)
a.plot(n[:2100], g['Capacity(Ah)'][:2100], color=BLUE, lw=6); a.plot(n[2100:], g['Capacity(Ah)'][2100:], color=ORANGE, lw=6)
# 7 S3
x, y = panel(6, 'S3 Association', ['gated trend test', 'shift test: level, rate'], STAGE[6])
a = icon_ax(x + 5, y + 9.8, PW - 10, 13); d = np.arange(-67, 68)
a.fill_between(d, 0, L['T_max'], color=AMBER, alpha=0.8, lw=0); a.plot(d, L['V_600s'], color=BLUE, lw=2.4)
# 8 S4
x, y = panel(7, 'S4 Sensitivity', ['self-surrogate injection', 'power curve → MDE₈₀'], STAGE[7])
a = icon_ax(x + 6, y + 10.0, PW - 12, 13)
for k, kd in enumerate(['generator', 'self-surrogate']):
    q = pt[pt.kind == kd]; a.plot(q.rho, q.power_family12, color=[BLUE, ORANGE][k], lw=[6, 3][k], ls=['-', '--'][k])
# 9 audit card
x, y = panel(8, 'Audit card', ['class per feature', 'permitted claims'], STAGE[8])
CLS = [('D', '#bdbdb5', INK), ('P', '#f5ae8c', INK), ('I', '#a9c6f0', '#0b2a5b'), ('T', '#f3d58c', INK), ('N', '#ededea', INK), ('N*', '#ededea', INK)]
for k, (t, fc, tc) in enumerate(CLS):
    cx = x + 7.5 + (k % 3) * 8; cy = y + 16.2 - (k // 3) * 6.6
    ax.add_patch(Rectangle((cx, cy), 7, 6.0, fc=fc, ec='white', lw=1.5)); ax.text(cx + 3.5, cy + 3.0, t, ha='center', va='center', color=tc, fontsize=15, weight='bold')
# 10 reliability
x, y = panel(9, 'Reliability', ['null P–P calibration', 'FPR 3–6 % vs naive ≤ 92 %'], STAGE[9])
a = icon_ax(x + 8, y + 9.8, PW - 16, 13); u = np.linspace(0, 1, 101)
a.plot(u, [np.mean(nul.p_naive <= z) for z in u], color=ORANGE, lw=5); a.plot(u, [np.mean(nul.p_tts <= z) for z in u], color=BLUE, lw=5)
# 11 explainability + generalizability
x, y = panel(10, 'XAI & generalization', ['SHAP vs null band', 'LOCO on 374 cells'], STAGE[10])
a = icon_ax(x + 6, y + 9.8, PW - 12, 13)
a.barh([3, 2, 1, 0], [79.5, 72.0, 8.3, 7.6], color=[ORANGE, ORANGE, '#8c8c86', BLUE], height=0.75)
# 12 benchmark
x, y = panel(11, 'Benchmark', ['10 planted types, 6 tests', 'false credit 1–3 %'], STAGE[11])
a = icon_ax(x + 8, y + 9.8, PW - 16, 13)
a.bar(range(4), [0.69, 0.15, 0.135, 0.017], color=['#8c8c86', AMBER, PINK, BLUE], width=0.75); a.set_ylim(0, 0.75)
import matplotlib.text as _mt
_need = 7.0 * fig.get_figwidth() / 6.7
for _t in fig.findobj(_mt.Text):
    if _t.get_text() and _t.get_fontsize() < _need: _t.set_fontsize(_need)
fig.savefig('figures/fig1_methodology.png', dpi=300); plt.close(fig)
print('methodology figure done')


In [ ]:
!python extra_methodology_figure.py

## Step 17 – graphical abstract (built from the result figures)

In [ ]:
%%writefile extra_graphical_abstract.py
"""Graphical abstract in the layout and colours of the original four-zone design, redrawn at its display size
(13 cm wide) so that every text element is >= 7 pt when shown at 13 cm (MethodsX Guide for Authors).
Zones: 1 Data (measured capacity with its cross-fitted trend), 2 TRACE audit (stages S1-S4),
3 Audit card (class grid for the NASA cells), 4 Verified outcomes (benchmark: false credit of each rule).
All plotted values are read from data/ and outputs/. Saved as PNG and TIFF at 600 dpi."""
import numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Rectangle
import trace_audit as T

INK, NAVY = '#111111', '#14306b'
GREEN, BLUE, PINK, ORANGE, AMBER, GREY = '#138a5e', '#1f5fbf', '#c2477d', '#d9541e', '#c98a00', '#8c8c86'
CLS = {'D': '#bdbdb5', 'P': '#f5ae8c', 'I': '#a9c6f0', 'T': '#f3d58c', 'N': '#d9d9d4'}
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 7, 'axes.linewidth': 0.6,
                     'xtick.major.width': 0.5, 'ytick.major.width': 0.5, 'xtick.major.size': 2, 'ytick.major.size': 2})
WMM, HMM = 130, 68                                        # canvas in mm = display size (13 cm wide)
fig = plt.figure(figsize=(WMM / 25.4, HMM / 25.4)); ax = fig.add_axes([0, 0, 1, 1])
ax.set_xlim(0, WMM); ax.set_ylim(0, HMM); ax.axis('off')
inset = lambda x, y, w, h: fig.add_axes([x / WMM, y / HMM, w / WMM, h / HMM])

def zone(x, w, num, title, col, y=7, h=49):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0,rounding_size=1.8', fc='white', ec=col, lw=1.3))
    ax.text(x + w / 2, y + h - 3.2, f'{num}  {title}', ha='center', va='center', fontsize=8, weight='bold', color=INK)

def chip(xc, y, t, fs=7):
    ax.text(xc, y, t, ha='center', va='center', fontsize=fs, color=INK,
            bbox=dict(boxstyle='round,pad=0.25', fc='#f1efe6', ec='none'))

def arrow(x0, x1, y=31.5):
    ax.add_patch(FancyArrowPatch((x0, y), (x1, y), arrowstyle='-|>', mutation_scale=8, lw=1.2, color=INK))

# title and banner (old layout)
ax.text(WMM / 2, 65, 'TRACE: what can one battery ageing record support before modelling?', ha='center', va='center',
        fontsize=8.5, weight='bold', color=INK)
ax.text(WMM / 2, 59.6, 'Pre-modelling audit: restatements flagged, false credit 1–3 %, valid under autocorrelation',
        ha='center', va='center', fontsize=7, color=INK, bbox=dict(boxstyle='round,pad=0.35', fc='#fbe7b5', ec=AMBER, lw=0.9))

# 1 Data -------------------------------------------------------------------------------------------
X1, W1 = 1, 27; zone(X1, W1, '1', 'Data', GREEN)
d = pd.read_csv('data/nasa_cycle_features.csv'); d = d[d.battery == 5]
k = d.cycle.values.astype(float); y = d.capacity.values; ct, _ = T.crossfit_trend(k, y)
a = inset(X1 + 8, 22.5, W1 - 10, 25)
a.plot(k, y, color=ORANGE, lw=0.9, label='measured'); a.plot(k, ct, color=INK, lw=0.8, ls='--', label='trend')
a.set_xticks([0, 80, 160]); a.set_yticks([1.4, 1.6, 1.8]); a.tick_params(labelsize=7, pad=1)
a.set_xlabel('cycle k', fontsize=7, labelpad=0.5); a.set_title('capacity (Ah)', fontsize=7, pad=2)
for s in ['top', 'right']: a.spines[s].set_visible(False)
chip(X1 + W1 / 2, 13.6, 'GMP + 4 NASA'); chip(X1 + W1 / 2, 9.8, '+ 374 public cells')
arrow(X1 + W1 + 0.3, X1 + W1 + 2.6)

# 2 TRACE audit ------------------------------------------------------------------------------------
X2, W2 = 31, 30; zone(X2, W2, '2', 'TRACE audit', BLUE)
for i, (s, t) in enumerate([('S1', 'provenance'), ('S2', 'index ceiling'), ('S3', 'time-shift test'), ('S4', 'power: MDE₈₀')]):
    yc = 45 - i * 7.6
    ax.add_patch(FancyBboxPatch((X2 + 2, yc - 3), W2 - 4, 6, boxstyle='round,pad=0,rounding_size=1', fc='#eaf1fb', ec=BLUE, lw=0.8))
    ax.add_patch(FancyBboxPatch((X2 + 2, yc - 3), 7, 6, boxstyle='round,pad=0,rounding_size=1', fc=BLUE, ec=BLUE, lw=0.8))
    ax.text(X2 + 5.5, yc, s, ha='center', va='center', fontsize=7.5, weight='bold', color='white')
    ax.text(X2 + 10.5, yc, t, ha='left', va='center', fontsize=7, color=INK)
chip(X2 + W2 / 2, 11.5, 'false positives ≤ 6 %')
arrow(X2 + W2 + 0.3, X2 + W2 + 2.6)

# 3 Audit card -------------------------------------------------------------------------------------
X3, W3 = 64.5, 32; zone(X3, W3, '3', 'Audit card', PINK)
A = pd.read_csv('outputs/out_audit_nasa.csv')
feats = [('ambient_T', 'amb. T'), ('t_discharge', 't_dis'), ('V_600s', 'V₆₀₀'), ('I_mean', 'I mean')]
bats = ['B0005', 'B0006', 'B0007', 'B0018']; cw, chh, gx, gy = 4.4, 4.7, X3 + 11, 45
for j, b in enumerate(bats):
    ax.text(gx + j * cw + cw / 2, gy + 1.0, b[1:].lstrip('0'), ha='center', va='bottom', fontsize=7, color=INK)
for i, (f, lab) in enumerate(feats):
    yy = gy - (i + 1) * chh
    ax.text(gx - 0.8, yy + chh / 2, lab, ha='right', va='center', fontsize=7, color=INK)
    for j, b in enumerate(bats):
        c = A[(A.feature == f) & (A.battery == b)]['class'].iloc[0]
        ax.add_patch(Rectangle((gx + j * cw, yy), cw - 0.4, chh - 0.4, fc=CLS[c], ec='white'))
        ax.text(gx + j * cw + cw / 2 - 0.2, yy + chh / 2 - 0.2, c, ha='center', va='center', fontsize=7.5, weight='bold', color=INK)
ax.text(X3 + W3 / 2, 21, 'D derived · P proxy', ha='center', va='center', fontsize=7, color=INK)
ax.text(X3 + W3 / 2, 17.4, 'I beyond k · T trend', ha='center', va='center', fontsize=7, color=INK)
chip(X3 + W3 / 2, 11.5, 'one class per feature')
arrow(X3 + W3 + 0.3, X3 + W3 + 2.6)

# 4 Verified outcomes: one benchmark chart ---------------------------------------------------------
X4, W4 = 99, 30; zone(X4, W4, '4', 'Benchmark', ORANGE)
B = pd.read_csv('outputs/out_bench2_table.csv', index_col=0)
rules = [('M1_corr', 'Corr.'), ('M6_dcca', 'DCCA'), ('M7_hac', 'HAC'), ('M5_prewhiten', 'Pre-wh.'), ('M4_TRACE', 'TRACE')]
fc = [B.loc[['trend_hi', 'null_ar', 'null_trend'], m].mean() * 100 for m, _ in rules]
a = inset(X4 + 10, 23.5, W4 - 13, 22)
cols = [GREY, PINK, AMBER, GREY, BLUE]
a.barh(range(len(rules)), fc, color=cols, height=0.7); a.invert_yaxis()
a.set_yticks(range(len(rules))); a.set_yticklabels([r[1] for r in rules], fontsize=7)
for i, v in enumerate(fc): a.text(v + 2, i, f'{v:.0f}', va='center', fontsize=7, color=INK)
a.set_xlim(0, 90); a.set_xticks([0, 40, 80]); a.tick_params(labelsize=7, pad=1)
a.set_xlabel('false credit (%)', fontsize=7, labelpad=0.5)
for s in ['top', 'right']: a.spines[s].set_visible(False)
chip(X4 + W4 / 2, 13.6, 'proxies flagged:'); chip(X4 + W4 / 2, 9.8, 'only by TRACE')

ax.text(WMM / 2, 3.2, 'what one record can support  ·  bounded negatives  ·  reproducible Python / Colab code',
        ha='center', va='center', fontsize=7, weight='bold', color=NAVY)
fig.savefig('figures/graphical_abstract.png', dpi=600)
fig.savefig('figures/graphical_abstract.tif', dpi=600, pil_kwargs={'compression': 'tiff_lzw'})
print('graphical abstract done')


In [ ]:
!python extra_graphical_abstract.py

## Step 18 – Method details and Method validation tables (Tables 2–5, S1–S4) and every quoted number with its source (results_numbers.md)

In [ ]:
%%writefile 14_report_numbers.py
"""Step 14 - assemble the Method details and Method validation chapters from the outputs.

Writes
  outputs/tables/Table2_classes.csv, Table3_parameters.csv, Table4_experiments.csv,
  outputs/tables/Table5_nasa_audit_card.csv, TableS1..S4 (*.csv)
  outputs/results_numbers.md   every number quoted in the Method validation chapter, by section (V1-V8),
                               each with the output file it comes from.
Run after scripts 00-13. Uses only files in outputs/."""
import json, os, numpy as np, pandas as pd

O = 'outputs/'; TD = O + 'tables/'; os.makedirs(TD, exist_ok=True)
rd = lambda f, **k: pd.read_csv(O + f, **k)
js = lambda f: json.load(open(O + f))
L = []                                    # markdown lines
def H(t): L.append(f'\n## {t}\n')
def item(txt, src): L.append(f'- {txt}  _(source: {src})_')
def rng(a, d=3): a = np.asarray(a, float); return f'{a.min():.{d}f}–{a.max():.{d}f}'

# ------------------------------------------------------------------ Method details tables (static definitions)
pd.DataFrame([
    ['D', 'Constant; affine in k or y; or deterministic in k (eta < 1e-3)', 'Not a predictor; may be a label.'],
    ['P', 'Restates the target: |r_res| >= 0.98 or restatement index zeta < 0.01', 'Co-computed with y; exclude and check provenance.'],
    ['I', 'Beyond-index time-shift test significant after Holm correction', 'Carries same-cycle health information that k does not.'],
    ['T', 'Co-trends with k; beyond-index test not significant', 'Not identifiable on one record; may still transfer across cells.'],
    ['N', 'No trend coupling; beyond-index test not significant', 'Coupling below the reported MDE80.'],
    ['N*', 'Target deterministic in k (eta(y) < 1e-3): beyond-index test undefined', 'No claim beyond k is possible.'],
], columns=['Class', 'Decision rule', 'Permitted claim']).to_csv(TD + 'Table2_classes.csv', index=False)
pd.DataFrame([
    ['Trend smoother', 'Cubic regression spline, min(12, n//15) quantile knots', 'knots 6-12 in V5'],
    ['Cross-fitting', 'Two folds, interleaved odd/even cycles', 'trend at a cycle never uses that cycle'],
    ['Determinism threshold', 'eta < 1e-3', 'swept 1e-4 to 1e-2 (V5)'],
    ['Proxy thresholds', '|r_res| >= 0.98; zeta < 0.01', 'swept 0.90-0.995 and 0.0025-0.05 (V5)'],
    ['Trend test', 'Permutation, B = 999; Ljung-Box (lag 10) gate', 'descriptive if the gate rejects'],
    ['Beyond-index statistic', 'max(|Spearman| with e_y, with diff(y)), lag 0', 'multi-lag maximum rejected (inflated size)'],
    ['Truncation radius r', 'min(max(n//4, ceil((p/alpha-1)/2)), floor(0.4 n)), p = candidate features', 'fixed before classification; m/(2r+1) <= alpha for tested m <= p'],
    ['Multiplicity', 'Holm within the feature family', 'family-wise error control'],
    ['Power', '80 %, coupling grid rho in [0, 0.9]', 'MDE80 per family'],
], columns=['Parameter', 'Default', 'Rationale; sensitivity']).to_csv(TD + 'Table3_parameters.csv', index=False)
pd.DataFrame([
    ['V1', 'Are known column classes recovered?', 'GMP published file'],
    ['V2', 'Are false positives controlled?', '1000 GMP nulls; AR(1) nulls n=3000; 1000 per NASA cell and phi'],
    ['V3', 'Does self-surrogate power match true power?', '500 replicates per rho (GMP); 400 per rho (NASA)'],
    ['V4', 'Are real signals and restatements found?', '4 NASA cells'],
    ['V5', 'Robust to settings, thresholds, p-resolution?', 'knots x radius; threshold sweeps on 378 cells; circular-shift re-test'],
    ['V6', 'What do the classes mean across many cells?', '374 public cells; within-dataset LOCO'],
    ['V7', 'Comparison with established tests', '10 planted types x 360 replicates on 12 targets; 6 competitors'],
    ['V8', 'Do the classes change modelling and explanation?', 'NASA LOCO, SHAP; GMP null band'],
], columns=['Exp.', 'Question', 'Data and replication']).to_csv(TD + 'Table4_experiments.csv', index=False)

# ------------------------------------------------------------------ V1 GMP
G = rd('out_audit_gmp.csv'); GA = js('out_audit_gmp_attrs.json')
H('V1. Recovery of the known structure of the GMP dataset')
item('Class D: ' + ', '.join(G[G['class'] == 'D'].feature), 'out_audit_gmp.csv')
item(f"Capacity determinism index eta = {GA['eta_target']:.2e}", 'out_audit_gmp_attrs.json')
item(f"Index-only ceiling R2 = {GA['ceiling']['R2_index']:.3f}, RMSE = {GA['ceiling']['RMSE_index']:.1e} Ah", 'out_audit_gmp_attrs.json')
item(f"Operating channels classed N*: {(G['class'] == 'N*').sum()} of {len(G) - (G['class'] == 'D').sum()}", 'out_audit_gmp.csv')
# index-only RUL: power law fitted to the first H cycles, end of life at 70% SOH (compared with the companion CMC study)
from scipy.optimize import curve_fit, brentq
_d = pd.read_csv('data/GMP.csv'); _k = _d.CycleNumber.values.astype(float); _y = _d['Capacity(Ah)'].values; _R = _d['RUL(cycles)'].values
_pw = lambda k, a, b, c: a - b * k ** c
for _h in [500, 1000, 1500]:
    _p, _ = curve_fit(_pw, _k[:_h], _y[:_h], p0=[_y[0], 1e-3, 1.2], maxfev=20000)
    _eol = brentq(lambda t: _pw(t, *_p) - 0.7 * _y[0], 1, 1e5)
    _rm = float(np.sqrt(np.mean((_R[_h:] - (_eol - _k[_h:])) ** 2)))
    item(f"Index-only RUL from the first {_h} cycles: predicted end of life {_eol:.1f}, RUL RMSE {_rm:.3f} cycles (CMC best LSTM: 25.59 cycles)", 'data/GMP.csv')

# ------------------------------------------------------------------ V2 false positives
V = js('out_validation.json'); TG = rd('out_typeI_gmp.csv'); AR = rd('out_typeI_ar.csv'); CN = rd('out_typeI_cond_nasa.csv')
fw = open(O + 'out_typeI_fwer.txt').read().split(','); LB = rd('out_lb_gate.csv'); D = js('out_diagnostics.json')
H('V2. False-positive control')
item(f"Trend test on 1000 GMP nulls: per-channel {rng(V['V2a_trend_FPR'])}; Holm FWER {V['V2a_trend_FWER']:.3f}", 'out_validation.json')
item(f"Time-shift test on GMP nulls: per-channel {rng(TG.FPR_TTS)}; FWER {float(fw[0]):.3f}", 'out_typeI_gmp.csv, out_typeI_fwer.txt')
item('AR(1), n = 3000: ' + '; '.join(f'phi {r.phi}: shift {r.FPR_TTS:.3f} vs naive {r.FPR_naive:.3f}' for _, r in AR.iterrows()), 'out_typeI_ar.csv')
item(f"NASA residual + AR(1) features: shift {rng(CN.FPR_TTS)}; naive at phi >= 0.9 {rng(CN[CN.phi >= 0.9].FPR_naive)}", 'out_typeI_cond_nasa.csv')
item('Ljung-Box gate rejection: ' + '; '.join(f'n={int(r.n)}, phi={r.phi}: {r.gate_rejects:.3f}' for _, r in LB.iterrows()), 'out_lb_gate.csv')
item(f"Modified Mann-Kendall size: {D['MK_HR_size_phi0.9']:.2f} (phi 0.9), {D['MK_HR_size_phi0.99']:.2f} (phi 0.99)", 'out_diagnostics.json')

# ------------------------------------------------------------------ V3 power
MS = rd('out_mde_surface.csv')
H('V3. Self-surrogate sensitivity')
item(f"Trend MDE80 family of 12: generator {V['MDE80_trend_generator_fam']:.3f}, self-surrogate {V['MDE80_trend_self-surrogate_fam']:.3f}; "
     f"per test {V['MDE80_trend_generator_test']:.3f} / {V['MDE80_trend_self-surrogate_test']:.3f}", 'out_validation.json')
item('NASA beyond-index MDE80 per test: ' + ', '.join(f"B{b}: {V[f'MDE80_cond_B{b}_test']:.2f}" for b in [5, 6, 7, 18]), 'out_validation.json')
for n in [150, 2000]:
    item(f"Simulated MDE80 at n = {n}: {rng(MS[MS.n == n].MDE80, 2)} (phi 0-0.95)", 'out_mde_surface.csv')
item(f"n = 100: MDE80 resolvable = {MS[MS.n == 100].MDE80.notna().any()}", 'out_mde_surface.csv')

# ------------------------------------------------------------------ V4 NASA
A = rd('out_audit_nasa.csv')
def cell(r):
    if r['class'] == 'P': return f"P ({abs(r.r_resid):.3f})"
    if r['class'] in ('I', 'T', 'N'): return f"{r['class']} ({r.S_cond:.3f}; {r.p_cond_holm:.3f})"
    return r['class']
card = A.assign(v=A.apply(cell, axis=1)).pivot(index='feature', columns='battery', values='v')
meta = A.groupby('battery').agg(n=('r', 'size'), r=('r', 'first'), eta=('eta_target', 'first'), R2=('R2_index', 'first'))
card.to_csv(TD + 'Table5_nasa_audit_card.csv')
H('V4. Measured cells: NASA (Table 5)')
L.append('\n' + card.to_markdown() + '\n')
item(f"Target eta {rng(meta.eta)}; index-only R2 {rng(meta.R2, 2)}; r per cell {', '.join(map(str, meta.r))}", 'out_audit_nasa.csv')
for c in ['P', 'I']:
    item(f"Class {c} counts per feature: " + ', '.join(f'{k} {v}/4' for k, v in A[A['class'] == c].feature.value_counts().items()), 'out_audit_nasa.csv')

# ------------------------------------------------------------------ V5 robustness
RB = rd('out_robust_nasa.csv'); TS = rd('out_threshold_sweep.csv'); ZS = rd('out_zeta_sweep.csv')
EM = rd('out_target_eta_margin.csv'); PF = rd('out_pfloor_nasa.csv'); E = rd('out_ext_audit.csv')
H('V5. Robustness')
res = RB[RB.r.astype(str) != '0.25']
item(f"Agreement with the default audit over knots x radius (excluding r = n/4): {res.agree.mean() * 100:.1f}% of {len(res)} verdicts; "
     f"D and P agreement {res[res.base.isin(['D', 'P'])].agree.mean() * 100:.0f}%; with r = n/4: {RB[RB.r.astype(str) == '0.25'].agree.mean() * 100:.1f}%", 'out_robust_nasa.csv')
item(f"Threshold sweep (eta x proxy) max share changed {TS.changed.max() * 100:.2f}%; within proxy 0.95-0.995 {TS[TS.proxy_thr >= 0.95].changed.max() * 100:.2f}%", 'out_threshold_sweep.csv')
item('zeta sweep: ' + '; '.join(f'{r.zeta_thr:g}: changed {r.changed * 100:.2f}%, P public {int(r.n_P_public)}' for _, r in ZS.iterrows()), 'out_zeta_sweep.csv')
item('Cells whose target would be deterministic: ' + '; '.join(f'{r.eta_det:g}: {int(r.cells_target_deterministic)}/{int(r.of)}' for _, r in EM.iterrows()), 'out_target_eta_margin.csv')
item(f"Smallest public target eta {E.drop_duplicates('cell').eta_target.min():.4f}; max |r_res| of a non-P public feature {E[E['class'] != 'P'].r_resid.abs().max():.3f}", 'out_ext_audit.csv')
I_ = PF[PF.cls == 'I']
item(f"p-floor: {int(I_.at_floor.sum())} of {len(I_)} class-I verdicts at the floor ({', '.join(f'{v:.3f}' for v in sorted(I_.p_floor_holm.unique()))}); "
     f"circular-shift Holm p {rng(I_.p_circ_holm)}; confirmed {int((I_.p_circ_holm <= 0.05).sum())}/{len(I_)}", 'out_pfloor_nasa.csv')
T_ = PF[PF.cls == 'T']
item(f"Class-T verdicts significant under the circular-shift test: {int((T_.p_circ_holm <= 0.05).sum())} of {len(T_)} ("
     + ', '.join(f'B{int(r.battery):04d} {r.feature}' for _, r in T_[T_.p_circ_holm <= 0.05].iterrows()) + ')', 'out_pfloor_nasa.csv')

# ------------------------------------------------------------------ V6 external
LX = rd('out_ext_loco_summary.csv')
H('V6. External validation on public cells (Fig. 7)')
item(f"{E.cell.nunique()} cells: " + ', '.join(f'{k} {v}' for k, v in E.groupby('dataset').cell.nunique().items()) +
     f"; cycles {int(E.n.min())}-{int(E.n.max())}; {len(E)} verdicts", 'out_ext_audit.csv')
item('Class shares: ' + ', '.join(f'{k} {v * 100:.1f}%' for k, v in E['class'].value_counts(normalize=True).items()), 'out_ext_audit.csv')
item(f"Cells with >= 1 class-I feature: {E.groupby('cell')['class'].apply(lambda c: (c == 'I').any()).mean() * 100:.0f}%", 'out_ext_audit.csv')
sh = E.pivot_table(index='feature', columns='class', values='cell', aggfunc='count', fill_value=0); sh = (sh.T / sh.sum(axis=1)).T
sh.round(3).to_csv(TD + 'Fig7a_class_share.csv')
item('Highest I shares: ' + ', '.join(f'{k} {v * 100:.0f}%' for k, v in sh['I'].sort_values(ascending=False).head(4).items()), 'out_ext_audit.csv')
piv = LX.pivot(index='dataset', columns='set', values='median_rmse_pct').round(2); piv.to_csv(TD + 'Fig7b_loco_rmse_pct.csv')
L.append('\n' + piv.to_markdown() + '\n')
fold = piv['index only'] / piv['index + T']
item(f"Error reduction index-only -> index + T: {fold.min():.1f}- to {fold.max():.0f}-fold", 'out_ext_loco_summary.csv')

# ------------------------------------------------------------------ V7 benchmark
BT = rd('out_bench2_table.csv', index_col=0); BS = js('out_bench2_summary.json'); SW = rd('out_bench2_sweep.csv'); PX = rd('out_bench2_proxy.csv')
H('V7. Benchmark (Fig. 8, Table S3)')
L.append('\n' + BT.round(3).to_markdown() + '\n')
NI = ['trend_hi', 'null_ar', 'null_trend']
for m in ['M1_corr', 'M3_detrended', 'M5_prewhiten', 'M6_dcca', 'M7_hac', 'M4v1_TRACE', 'M4_TRACE']:
    s = BS[m]; item(f"{m}: false credit on trend/null types {rng(BT.loc[NI, m], 3)}; precision {s['precision']:.3f}; recall design {s['recall_design']:.3f}, "
                    f"held-out {s['recall_heldout']:.3f}; proxy credited {BT.loc['proxy', m]:.2f}", 'out_bench2_table.csv, out_bench2_summary.json')
item(f"TRACE v2 leak detection {BS['TRACE_leak_detection']:.2f}; v1 {BS['TRACE_v1_leak_detection']:.2f}; replicates {BS['replicates']}", 'out_bench2_summary.json')
sw = SW.groupby(['feature', 'c_lag'])[['M5_prewhiten', 'M7_hac', 'M4v1_TRACE', 'M4_TRACE']].mean().round(2)
sw.to_csv(TD + 'Fig8c_strength_sweep.csv'); L.append('\n' + sw.to_markdown() + '\n')
px = pd.crosstab(PX.noise, PX.cls, normalize='index').round(2); px.to_csv(TD + 'proxy_noise_sweep.csv')
L.append('\nProxy-noise sweep (share of classes by noise level, fraction of SD(y)):\n\n' + px.to_markdown() + '\n')

# ------------------------------------------------------------------ V8 consequences
LO = rd('out_loco.csv'); C8 = js('out_consequence.json'); SN = rd('out_shap_nasa.csv')
H('V8. Consequences for modelling and explanation (Fig. 9, Table S1)')
m = LO[LO.model == 'GBR'].groupby('feature_set').RMSE.mean() * 1000
item('Mean GBR LOCO RMSE (mAh, % of 2 Ah): ' + '; '.join(f'{k} {v:.1f} ({v / 20:.1f}%)' for k, v in m.items()), 'out_loco.csv')
b7 = LO[(LO.model == 'GBR') & (LO.held_out == 'B0007')].set_index('feature_set').R2
item(f"B0007 R2: index only {b7['Index only']:.2f} -> TRACE-admitted {b7['TRACE-admitted (I)']:.2f}", 'out_loco.csv')
s = SN.groupby(['feature_set', 'feature']).mean_abs_shap.mean() * 1000
item('Mean |SHAP| (mAh), all features: ' + ', '.join(f'{k} {v:.1f}' for k, v in s['All features'].sort_values(ascending=False).head(3).items()), 'out_shap_nasa.csv')
item(f"Mean |SHAP| of V_600s after removing D and P: {s['TRACE-admitted (I)']['V_600s']:.1f} mAh", 'out_shap_nasa.csv')
item(f"GMP random split R2 {C8['gmp_r2_random_split']:.3f}; ops |SHAP| {C8['gmp_shap_ops_min']:.1e}-{C8['gmp_shap_ops_max']:.1e}; "
     f"Mann-Whitney p {C8['gmp_shap_mw_p']:.2f}; chronological ops-only R2 {C8['gmp_r2_chrono_ops_only']:.1f}", 'out_consequence.json')
LO.to_csv(TD + 'TableS1_nasa_loco.csv', index=False)
BT.to_csv(TD + 'TableS3_benchmark.csv'); TS.to_csv(TD + 'TableS4a_threshold_sweep.csv', index=False)
ZS.to_csv(TD + 'TableS4b_zeta_sweep.csv', index=False); EM.to_csv(TD + 'TableS4c_target_eta_margin.csv', index=False); PF.to_csv(TD + 'TableS4d_pfloor.csv', index=False)
pd.concat([LB.assign(test='Ljung-Box gate'), pd.DataFrame([dict(test='modified Mann-Kendall', n=3000, phi=0.9, gate_rejects=D['MK_HR_size_phi0.9']),
                                                            dict(test='modified Mann-Kendall', n=3000, phi=0.99, gate_rejects=D['MK_HR_size_phi0.99'])])]
          ).rename(columns={'gate_rejects': 'rejection_rate'}).to_csv(TD + 'TableS2_test_size.csv', index=False)

open(O + 'results_numbers.md', 'w').write('# TRACE - numbers quoted in the Method validation chapter\n' + '\n'.join(L) + '\n')
print('\n'.join(L)); print('\nwrote outputs/results_numbers.md and outputs/tables/')


In [ ]:
!python 14_report_numbers.py

## Step 19 – Supplementary Figures S1–S4

In [ ]:
%%writefile 15_supp_figures.py
"""Step 15 - Supplementary Figures S1-S4 (same visual style as the article figures).
  S1 threshold sensitivity (eta x proxy heatmap; zeta sweep)                 <- out_threshold_sweep.csv, out_zeta_sweep.csv
  S2 size of the Ljung-Box gate and of the modified Mann-Kendall test        <- out_lb_gate.csv, out_diagnostics.json
  S3 p-value floor: truncated time-shift vs circular-shift p (NASA class I)  <- out_pfloor_nasa.csv
  S4 class shares by public dataset and proxy-noise boundary of S1           <- out_ext_audit.csv, out_bench2_proxy.csv"""
import json, os, numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

os.makedirs('figures', exist_ok=True); O = 'outputs/'
BLUE, ORANGE, GREEN, AMBER, PINK = '#1f5fbf', '#d9541e', '#138a5e', '#c98a00', '#c2477d'
INK, GRID = '#111111', '#dddddd'
CLS = {'D': '#bdbdb5', 'P': '#f5ae8c', 'I': '#a9c6f0', 'T': '#f3d58c', 'N': '#d9d9d4'}
LIGHT = LinearSegmentedColormap.from_list('lightYlOrBr', plt.get_cmap('YlOrBr')(np.linspace(0.0, 0.55, 256)))
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 12, 'font.weight': 'bold', 'axes.titleweight': 'bold',
                     'axes.labelweight': 'bold', 'axes.edgecolor': INK, 'axes.labelcolor': INK, 'xtick.color': INK,
                     'ytick.color': INK, 'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True,
                     'grid.color': GRID, 'lines.linewidth': 2.4, 'legend.frameon': False, 'savefig.dpi': 300,
                     'savefig.bbox': 'tight'})
def ttl(a, t):
    lab = t.split(')')[0] + ')' if t.startswith('(') else ''
    if lab: a.set_title(lab, loc='left', fontsize=15, weight='bold')


def legible(fig, print_w=6.7, min_pt=7.0):
    import matplotlib.text as mtext
    need = min_pt * fig.get_figwidth() / print_w
    for t in fig.findobj(mtext.Text):
        if t.get_text() and t.get_fontsize() < need: t.set_fontsize(need)

# ---------------- S1 thresholds
S = pd.read_csv(O + 'out_threshold_sweep.csv'); Z = pd.read_csv(O + 'out_zeta_sweep.csv')
fig, ax = plt.subplots(1, 2, figsize=(13, 4.8), gridspec_kw=dict(width_ratios=[1.2, 1], wspace=0.35))
M = S.pivot(index='eta_det', columns='proxy_thr', values='changed') * 100
a = ax[0]; a.grid(False); a.imshow(M.values, cmap=LIGHT, vmin=0, vmax=max(0.5, M.values.max()), aspect='auto')
for i in range(M.shape[0]):
    for j in range(M.shape[1]): a.text(j, i, f'{M.values[i, j]:.2f}', ha='center', va='center', color=INK, fontsize=11)
a.set_xticks(range(M.shape[1])); a.set_xticklabels([f'{v:g}' for v in M.columns]); a.set_yticks(range(M.shape[0]))
a.set_yticklabels([f'{v:g}' for v in M.index]); a.set_xlabel('proxy threshold |r_res|'); a.set_ylabel('determinism threshold η')
ttl(a, '(a) % of 6016 verdicts changed vs defaults (10⁻³, 0.98)')
a = ax[1]; a.plot(Z.zeta_thr, Z.n_P_public, 'o-', color=ORANGE, label='public cells'); a.plot(Z.zeta_thr, Z.n_P_nasa, 's-', color=BLUE, label='NASA cells')
a.axvline(0.01, color=INK, ls='--', lw=1.3); a.set_xscale('log'); a.set_xticks(Z.zeta_thr); a.set_xticklabels([f'{v:g}' for v in Z.zeta_thr]); a.minorticks_off(); a.set_xlabel('restatement threshold ζ'); a.set_ylabel('features classed P')
a.legend(); ttl(a, '(b) Class P count vs ζ threshold (default 0.01)')
legible(fig); fig.savefig('figures/figS1_thresholds.png'); plt.close(fig)

# ---------------- S2 test size
LB = pd.read_csv(O + 'out_lb_gate.csv'); D = json.load(open(O + 'out_diagnostics.json'))
fig, ax = plt.subplots(figsize=(7.5, 4.6))
for n, c in [(168, ORANGE), (3000, BLUE)]:
    q = LB[LB.n == n]; ax.plot(q.phi, q.gate_rejects, 'o-', color=c, label=f'Ljung–Box gate, n = {n}')
    if 'chi2_reference_rejects' in q: ax.plot(q.phi, q.chi2_reference_rejects, 'x:', color=c, label=f'χ² reference, n = {n}')
ax.plot([0.9, 0.99], [D['MK_HR_size_phi0.9'], D['MK_HR_size_phi0.99']], 's--', color=PINK, label='modified Mann–Kendall, n = 3000')
ax.axhline(0.05, color=INK, ls=':', lw=1.3); ax.set_xlabel('AR(1) coefficient φ'); ax.set_ylabel('rejection rate under H₀'); ax.set_ylim(0, 1.05)
ax.legend(loc='center right'); ttl(ax, 'Size of the exchangeability gate and of the Mann–Kendall test')
legible(fig); fig.savefig('figures/figS2_test_size.png'); plt.close(fig)

# ---------------- S3 p floor
P = pd.read_csv(O + 'out_pfloor_nasa.csv')
fig, ax = plt.subplots(figsize=(7.5, 5))
for c, mk, col in [('I', 'o', BLUE), ('T', '^', AMBER), ('N', 'v', '#8c8c86')]:
    q = P[P.cls == c]
    if len(q): ax.scatter(q.p_tts_holm, q.p_circ_holm, s=70, marker=mk, color=col, edgecolor=INK, label=f'class {c} (default test)', zorder=3)
lim = [0, max(1.0, P[['p_tts_holm', 'p_circ_holm']].max().max())]; ax.plot(lim, lim, color=INK, lw=1)
for f_ in P.p_floor_holm.unique(): ax.axvline(f_, color=ORANGE, ls='--', lw=1.2)
ax.axhline(0.05, color=INK, ls=':', lw=1.2); ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('truncated time-shift p (Holm)'); ax.set_ylabel('circular-shift p, resolution 1/n (Holm)'); ax.legend(loc='lower right')
ttl(ax, 'p-value floor (dashed) and confirmation of NASA verdicts')
legible(fig); fig.savefig('figures/figS3_pfloor.png'); plt.close(fig)

# ---------------- S4 dataset shares and proxy boundary
E = pd.read_csv(O + 'out_ext_audit.csv'); PX = pd.read_csv(O + 'out_bench2_proxy.csv')
fig, ax = plt.subplots(1, 2, figsize=(13, 4.8), gridspec_kw=dict(wspace=0.3))
sh = pd.crosstab(E.dataset, E['class'], normalize='index')
for c in ['I', 'T', 'N', 'P']:
    if c not in sh: sh[c] = 0
a = ax[0]; bot = np.zeros(len(sh))
for c in ['I', 'T', 'N', 'P']:
    a.bar(sh.index, sh[c], bottom=bot, color=CLS[c], edgecolor='white', lw=2, label=c); bot += sh[c].values
for i, d in enumerate(sh.index): a.text(i, sh.loc[d, 'I'] / 2, f"{sh.loc[d, 'I']:.2f}", ha='center', va='center', color=INK)
a.set_ylabel('share of feature verdicts'); a.legend(ncol=4, loc='upper center', bbox_to_anchor=(0.5, -0.1)); ttl(a, '(a) Class shares by public dataset')
cx = pd.crosstab(PX.noise, PX.cls, normalize='index')
a = ax[1]
for c, col in [('P', ORANGE), ('I', BLUE), ('T', AMBER)]:
    if c in cx: a.plot(cx.index, cx[c], 'o-', color=col, label=f'classed {c}')
a.set_xscale('log'); a.set_xlabel('proxy noise (fraction of SD(y))'); a.set_ylabel('share of planted proxies'); a.legend()
ttl(a, '(b) Restatement boundary of S1')
legible(fig); fig.savefig('figures/figS4_external_proxy.png'); plt.close(fig)
print('supplementary figures done')


In [ ]:
!python 15_supp_figures.py

## Step 20 – Word files: main tables and Supplementary material

In [ ]:
%%writefile 16_tables_docx.py
"""Step 16 - write the result tables of the Method details / Method validation chapters and the
Supplementary Tables S1-S4 and Figures S1-S4 as Word files, from outputs/tables/*.csv (written by step 14).
  outputs/TRACE_main_tables.docx           Tables 2-5
  outputs/TRACE_Supplementary_Material.docx  Tables S1-S4 (S4a-d) and Figures S1-S4"""
import pandas as pd
from docx import Document
from docx.shared import Pt, Inches
import os
from docx.enum.section import WD_ORIENT

TD = 'outputs/tables/'


def fmt(v):
    if isinstance(v, float):
        return '' if pd.isna(v) else (f'{v:.3g}' if abs(v) < 1e-3 and v != 0 else f'{v:.3f}'.rstrip('0').rstrip('.') if v % 1 else f'{v:.0f}')
    return str(v)


def add_table(doc, caption, df, index=False):
    if index: df = df.reset_index()
    p = doc.add_paragraph(); r = p.add_run(caption); r.bold = True; r.font.size = Pt(10)
    t = doc.add_table(rows=1, cols=len(df.columns)); t.style = 'Table Grid'
    for j, c in enumerate(df.columns):
        t.rows[0].cells[j].text = str(c); t.rows[0].cells[j].paragraphs[0].runs[0].bold = True
    for _, row in df.iterrows():
        cells = t.add_row().cells
        for j, v in enumerate(row): cells[j].text = fmt(v)
    for row in t.rows:
        for c in row.cells:
            for par in c.paragraphs:
                for run in par.runs: run.font.size = Pt(8)
    doc.add_paragraph()


def new_doc(title):
    d = Document(); st = d.styles['Normal']; st.font.name = 'Times New Roman'; st.font.size = Pt(11)
    from docx.oxml.ns import qn
    z = d.settings.element.find(qn('w:zoom'))
    if z is not None and z.get(qn('w:percent')) is None: z.set(qn('w:percent'), '100')   # schema-valid settings.xml
    d.add_heading(title, level=1); return d


m = new_doc('TRACE - tables of the Method details and Method validation chapters')
add_table(m, 'Table 2. Audit classes, their decision rules and the claim each permits.', pd.read_csv(TD + 'Table2_classes.csv'))
add_table(m, 'Table 3. Default parameters and where each was stress-tested.', pd.read_csv(TD + 'Table3_parameters.csv'))
add_table(m, 'Table 4. Validation experiments.', pd.read_csv(TD + 'Table4_experiments.csv'))
add_table(m, 'Table 5. TRACE audit card for the four NASA cells: class with |r_res| for P, and (S0; Holm-adjusted p) for I and T.',
          pd.read_csv(TD + 'Table5_nasa_audit_card.csv'))
m.save('outputs/TRACE_main_tables.docx')

s = new_doc('Supplementary material: Tables S1-S4 and Figures S1-S4')
s.add_paragraph('All values are read from outputs/ and are regenerated by run_all.sh or the Colab notebook.')
lo = pd.read_csv(TD + 'TableS1_nasa_loco.csv'); lo['RMSE'] = lo['RMSE'] * 1000; lo = lo.rename(columns={'RMSE': 'RMSE (mAh)'})
add_table(s, 'Table S1. NASA leave-one-cell-out results for every feature set, model and held-out cell.', lo)
add_table(s, 'Table S2. Size of the Ljung-Box exchangeability gate under AR(1) nulls and of the modified Mann-Kendall test.', pd.read_csv(TD + 'TableS2_test_size.csv'))
add_table(s, 'Table S3. Benchmark: share of 360 replicates in which each rule credits each planted type; last column = share TRACE v2 flags as D or P.',
          pd.read_csv(TD + 'TableS3_benchmark.csv'))
add_table(s, 'Table S4a. Re-classification of 6016 measured-feature verdicts under alternative determinism and proxy thresholds.', pd.read_csv(TD + 'TableS4a_threshold_sweep.csv'))
add_table(s, 'Table S4b. Sensitivity to the restatement threshold zeta.', pd.read_csv(TD + 'TableS4b_zeta_sweep.csv'))
add_table(s, 'Table S4c. Margin of the target-determinism threshold over the 378 measured cells.', pd.read_csv(TD + 'TableS4c_target_eta_margin.csv'))
add_table(s, 'Table S4d. p-value floor of the time-shift test in the NASA cells and confirmation by a circular-shift test (resolution 1/n).',
          pd.read_csv(TD + 'TableS4d_pfloor.csv'))
s.add_heading('Supplementary Figures S1-S4', level=1)
for f, cap in [('figS1_thresholds', 'Fig. S1. Threshold sensitivity: (a) share of verdicts changed under alternative determinism and proxy thresholds; (b) number of features classed P as the restatement threshold zeta varies.'),
               ('figS2_test_size', 'Fig. S2. Rejection rate under AR(1) nulls of the Ljung-Box exchangeability gate (n = 168 and 3000) and of the modified Mann-Kendall test; dotted line = 0.05.'),
               ('figS3_pfloor', 'Fig. S3. Holm-adjusted p-values of the NASA features under the default truncated time-shift test and a circular-shift test with resolution 1/n; dashed lines = attainable floor of the default test.'),
               ('figS4_external_proxy', 'Fig. S4. (a) Class shares of the 16 charge features by public dataset; (b) classes assigned to planted proxies as their noise grows (restatement boundary of S1).')]:
    p = 'figures/' + f + '.png'
    if os.path.exists(p):
        s.add_picture(p, width=Inches(6.3)); c = s.add_paragraph(cap); c.runs[0].font.size = Pt(9)
s.save('outputs/TRACE_Supplementary_Material.docx')
print('wrote outputs/TRACE_main_tables.docx and outputs/TRACE_Supplementary_Material.docx')


In [ ]:
!python 16_tables_docx.py

## Step 21 – Results in Engineering article: index-only ceiling, feature atlas, family transfer (Figs. 2, 4, 6)

In [ ]:
%%writefile 17_rineng_analyses.py
"""Step 17 - analyses of the Results in Engineering research article (beyond the MethodsX validation).
(a) Index-attainable accuracy: within-record power-law extrapolation (S2 ceiling) for GMP, the 4 NASA cells and
    374 public cells; GMP remaining-useful-life comparison with the companion LSTM study.
(b) Feature atlas: share of class-I verdicts per charge feature and dataset, by record length and target smoothness.
(c) Transfer value of feature families: within-dataset leave-one-cell-out gradient boosting with
    index only / index + CC-phase / index + CV-phase / index + voltage statistics / index + current statistics.
Outputs: outputs/rineng_*.csv/json and figures/rineng_fig*.png."""
import json, os, sys, warnings, numpy as np, pandas as pd, matplotlib
from scipy import stats
from scipy.optimize import curve_fit, brentq
warnings.filterwarnings('ignore'); sys.path.insert(0, '.')
matplotlib.use('Agg'); import matplotlib.pyplot as plt
QUICK = os.environ.get('TRACE_QUICK') == '1'
O, FG = 'outputs/', 'figures/'
BLUE, ORANGE, GREEN, AMBER, PINK, GREY, INK = '#1f5fbf', '#d9541e', '#138a5e', '#c98a00', '#c2477d', '#8c8c86', '#111111'
DSC = {'XJTU': BLUE, 'TJU': ORANGE, 'HUST': GREEN, 'MIT': PINK}
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 12, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.color': '#dddddd', 'savefig.dpi': 300, 'savefig.bbox': 'tight', 'legend.frameon': False})
def lab(a, t): a.set_title(t, loc='left', fontsize=15, weight='bold')
def legible(fig, print_w=6.7, min_pt=7.0):
    import matplotlib.text as mt
    need = min_pt * fig.get_figwidth() / print_w
    for t in fig.findobj(mt.Text):
        if t.get_text() and t.get_fontsize() < need: t.set_fontsize(need)

E = pd.read_csv(O + 'out_ext_audit.csv'); A = pd.read_csv(O + 'out_audit_nasa.csv')
FAM = {'CC phase': ['CC Q', 'CC charge time'], 'CV phase': ['CV Q', 'CV charge time'],
       'voltage statistics': ['voltage mean', 'voltage std', 'voltage kurtosis', 'voltage skewness', 'voltage slope', 'voltage entropy'],
       'current statistics': ['current mean', 'current std', 'current kurtosis', 'current skewness', 'current slope', 'current entropy']}
fam_of = {f: k for k, v in FAM.items() for f in v}
R = {}

# ---------------- (a) index-attainable accuracy
d = pd.read_csv('data/GMP.csv'); k = d.CycleNumber.values.astype(float); y = d['Capacity(Ah)'].values; rul = d['RUL(cycles)'].values
pw = lambda k, a, b, c: a - b * k ** c
gmp = []
for h in [500, 1000, 1500]:
    p, _ = curve_fit(pw, k[:h], y[:h], p0=[y[0], 1e-3, 1.2], maxfev=20000)
    eol = brentq(lambda t: pw(t, *p) - 0.7 * y[0], 1, 1e5)
    gmp.append(dict(train_cycles=h, eol=eol, rul_rmse=float(np.sqrt(np.mean((rul[h:] - (eol - k[h:])) ** 2)))))
R['gmp_index_rul'] = gmp; R['cmc_best_lstm_rul_rmse'] = 25.59
C = E.drop_duplicates('cell')[['cell', 'dataset', 'n', 'eta_target', 'R2_index']].copy()
N = A.drop_duplicates('battery')[['battery', 'R2_index', 'eta_target']]
R['nasa_R2_index'] = dict(zip(N.battery, N.R2_index.round(2)))
R['public_R2_index'] = {ds: dict(median=float(g.R2_index.median()), q25=float(g.R2_index.quantile(.25)), q75=float(g.R2_index.quantile(.75)),
                                 share_ge_0_9=float((g.R2_index >= 0.9).mean()), share_lt_0=float((g.R2_index < 0).mean()), cells=len(g))
                        for ds, g in C.groupby('dataset')}
R['public_R2_index_all'] = dict(median=float(C.R2_index.median()), share_ge_0_9=float((C.R2_index >= 0.9).mean()), share_lt_0=float((C.R2_index < 0).mean()))

# ---------------- (b) feature atlas
E['I'] = (E['class'] == 'I').astype(int); E['family'] = E.feature.map(fam_of)
atlas = E.pivot_table(index='feature', columns='dataset', values='I', aggfunc='mean')
atlas['all'] = E.groupby('feature').I.mean(); atlas = atlas.sort_values('all', ascending=False); atlas.round(3).to_csv(O + 'rineng_atlas_feature_dataset.csv')
fam = E.pivot_table(index='family', columns='dataset', values='I', aggfunc='mean'); fam['all'] = E.groupby('family').I.mean(); fam.round(3).to_csv(O + 'rineng_atlas_family.csv')
bins = [0, 300, 600, 1200, 3000]; blab = ['<300', '300–600', '600–1200', '>1200']
E['nbin'] = pd.cut(E.n, bins, labels=blab)
byn = E.groupby(['dataset', 'nbin']).agg(I_share=('I', 'mean'), cells=('cell', 'nunique')).reset_index(); byn.to_csv(O + 'rineng_atlas_length.csv', index=False)
percell = E.groupby('cell').agg(nI=('I', 'sum'), n=('n', 'first'), eta=('eta_target', 'first'), dataset=('dataset', 'first'))
rho_n = stats.spearmanr(percell.n, percell.nI); rho_eta = stats.spearmanr(percell.eta, percell.nI)
R['atlas'] = dict(I_share_all=float(E.I.mean()), cells_with_I=float((percell.nI > 0).mean()),
                  spearman_nI_vs_length=[float(rho_n[0]), float(rho_n[1])], spearman_nI_vs_eta=[float(rho_eta[0]), float(rho_eta[1])],
                  top_features=atlas['all'].head(4).round(3).to_dict(), family_share=fam['all'].round(3).to_dict(),
                  share_by_length={str(b): float(E[E.nbin == b].I.mean()) for b in blab})

# ---------------- (c) transfer value of feature families (leave-one-cell-out within dataset)
from sklearn.ensemble import HistGradientBoostingRegressor
sys.path.insert(0, '.')
import importlib.util
spec = importlib.util.spec_from_file_location('ext', '11_external.py'); src = open('11_external.py').read()
ns = {}; exec(src.split("\nif __name__")[0].split("\nD = load()")[0], ns)   # reuse load() and F without running the audit
FIG_ONLY = os.environ.get('FIG_ONLY') == '1'
D = None if FIG_ONLY else ns['load']()
sets = {'index only': [], **{f'index + {k}': v for k, v in FAM.items()}}
rows = []
for ds, dd in ([] if FIG_ONLY else D[D.cell.isin(C.cell)].groupby('dataset')):
    cl = dd.cell.unique()
    if QUICK: cl = cl[:12]
    for te in cl:
        tr = dd[dd.cell.isin([c for c in cl if c != te])].iloc[::3]; ts = dd[dd.cell == te]; yv = ts.capacity.values
        for nm, fs in sets.items():
            X = ['k'] + fs
            p = HistGradientBoostingRegressor(max_iter=150, random_state=0).fit(tr[X], tr.capacity).predict(ts[X])
            rows.append(dict(dataset=ds, cell=te, set=nm, rmse_pct=float(100 * np.sqrt(np.mean((p - yv) ** 2)) / yv[:5].mean())))
    print(ds, 'family LOCO done', flush=True)
L = pd.read_csv(O + 'rineng_family_loco.csv') if FIG_ONLY else pd.DataFrame(rows)
if not FIG_ONLY: L.to_csv(O + 'rineng_family_loco.csv', index=False)
rng = np.random.default_rng(0); summ = []
for (ds, s), g in L.groupby(['dataset', 'set']):
    v = g.rmse_pct.values; bs = [np.median(rng.choice(v, len(v))) for _ in range(2000)]
    summ.append(dict(dataset=ds, set=s, cells=len(v), median=float(np.median(v)), lo=float(np.percentile(bs, 2.5)), hi=float(np.percentile(bs, 97.5))))
S = pd.DataFrame(summ); S.to_csv(O + 'rineng_family_loco_summary.csv', index=False); print(S.round(2).to_string(index=False))
R['family_loco'] = {ds: {r.set: round(r.median, 3) for r in g.itertuples()} for ds, g in S.groupby('dataset')}
json.dump(R, open(O + 'rineng_results.json', 'w'), indent=1, default=float)

# ---------------- figures
# Fig. 2: index-attainable accuracy
fig, ax = plt.subplots(1, 2, figsize=(13, 4.8), gridspec_kw=dict(width_ratios=[1, 1.25], wspace=0.3))
a = ax[0]; p, _ = curve_fit(pw, k[:500], y[:500], p0=[y[0], 1e-3, 1.2])
a.plot(k, y, color=INK, lw=3, label='GMP capacity (data)'); a.plot(k, pw(k, *p), color=ORANGE, ls='--', lw=2, label='power law, fit on cycles 1–500')
a.axvline(500, color=GREY, ls=':'); a.axhline(0.7 * y[0], color=GREY, ls=':'); a.set_xlabel('cycle'); a.set_ylabel('capacity (Ah)'); a.legend(loc='lower left')
a.text(0.04, 0.47, f"RUL error (cycles)\nindex only: {gmp[0]['rul_rmse']:.2f}\nbest LSTM [CMC]: 25.6", transform=a.transAxes, fontsize=10, va='top', ha='left', bbox=dict(fc='white', ec='0.8', pad=3))
lab(a, '(a)')
a = ax[1]; order = ['XJTU', 'TJU', 'HUST', 'MIT']
for i, ds in enumerate(order):
    v = C[C.dataset == ds].R2_index.clip(-2, 1).values
    a.boxplot(v, positions=[i], widths=0.55, showfliers=False, medianprops=dict(color=INK, lw=2))
    a.scatter(np.full(len(v), i) + np.random.default_rng(i).uniform(-0.18, 0.18, len(v)), v, s=10, color=DSC[ds], alpha=0.6, zorder=3)
a.scatter([4] * 4, N.R2_index.clip(-2, 1), s=40, color=GREY, zorder=3); a.scatter([5], [1.0], s=80, marker='*', color=ORANGE, zorder=3)
a.set_xticks(range(6)); a.set_xticklabels(order + ['NASA', 'GMP']); a.axhline(0, color=INK, lw=0.8)
a.set_ylabel('index-only R² (last 30% of cycles)'); lab(a, '(b)')
legible(fig); fig.savefig(FG + 'rineng_fig2_index_ceiling.png'); plt.close(fig)

# Fig. 3: NASA audit card
feats = ['ambient_T', 't_discharge', 't_Tmax', 'V_600s', 'V_mean', 'T_max', 'T_start', 'I_mean']
names = ['ambient temperature', 'discharge time', 'time to peak temperature', 'voltage after 600 s', 'mean voltage', 'peak temperature', 'start temperature', 'mean current']
CLS = {'D': '#bdbdb5', 'P': '#f5ae8c', 'I': '#a9c6f0', 'T': '#f3d58c', 'N': '#e5e5e0'}
bats = ['B0005', 'B0006', 'B0007', 'B0018']
fig, a = plt.subplots(figsize=(8.5, 5.2)); a.grid(False)
for i, f in enumerate(feats):
    for j, b in enumerate(bats):
        r = A[(A.feature == f) & (A.battery == b)].iloc[0]; c = r['class']
        a.add_patch(plt.Rectangle((j, i), 0.95, 0.9, fc=CLS[c], ec='white'))
        sub = f"|r|={abs(r.r_resid):.3f}" if c == 'P' else (f"S₀={r.S_cond:.2f}" if c in 'IT' else '')
        a.text(j + 0.47, i + 0.33, c, ha='center', va='center', fontsize=14, weight='bold'); a.text(j + 0.47, i + 0.68, sub, ha='center', va='center', fontsize=10)
a.set_xlim(0, 4); a.set_ylim(len(feats), 0); a.set_xticks(np.arange(4) + 0.47); a.set_xticklabels(bats); a.set_yticks(np.arange(len(feats)) + 0.45)
a.set_yticklabels(names); a.tick_params(length=0); [s.set_visible(False) for s in a.spines.values()]
legible(fig); fig.savefig(FG + 'rineng_fig3_nasa_card.png'); plt.close(fig)

# Fig. 4: feature atlas
fig, ax = plt.subplots(1, 2, figsize=(14, 6), gridspec_kw=dict(width_ratios=[1.2, 1], wspace=0.45))
a = ax[0]; a.grid(False); M = atlas[order].values
from matplotlib.colors import LinearSegmentedColormap
cm = LinearSegmentedColormap.from_list('b', ['#ffffff', '#a9c6f0', '#1f5fbf'])
a.imshow(M, cmap=cm, vmin=0, vmax=0.6, aspect='auto')
for i in range(M.shape[0]):
    for j in range(M.shape[1]): a.text(j, i, f'{M[i, j]:.2f}', ha='center', va='center', fontsize=10)
a.set_xticks(range(4)); a.set_xticklabels(order); a.set_yticks(range(len(atlas))); a.set_yticklabels(atlas.index); lab(a, '(a)')
a = ax[1]
for ds in order:
    q = byn[byn.dataset == ds]; q = q[q.cells >= 3]
    a.plot([blab.index(b) for b in q.nbin], q.I_share, marker='o', color=DSC[ds], lw=2.2, label=ds)
a.axhline(0.05, color=GREY, ls='--', lw=1.2); a.set_xticks(range(4)); a.set_xticklabels(blab); a.set_xlabel('record length (cycles)')
a.set_ylabel('share of features classed I'); a.legend(); lab(a, '(b)')
legible(fig); fig.savefig(FG + 'rineng_fig4_atlas.png'); plt.close(fig)

# Fig. 5: transfer value of feature families
fig, a = plt.subplots(figsize=(11, 4.8)); w = 0.16; sl = list(sets)
cols = [GREY, BLUE, GREEN, AMBER, PINK]
for i, s in enumerate(sl):
    q = S[S.set == s].set_index('dataset').reindex(order)
    a.bar(np.arange(4) + (i - 2) * w, q['median'], w, color=cols[i], label=s, yerr=[q['median'] - q.lo, q.hi - q['median']], capsize=2)
a.set_xticks(range(4)); a.set_xticklabels(order); a.set_ylabel('median LOCO RMSE (% of initial capacity)')
a.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=3)
legible(fig); fig.savefig(FG + 'rineng_fig5_family_transfer.png'); plt.close(fig)
print(json.dumps({k: v for k, v in R.items() if k != 'family_loco'}, indent=1, default=float)[:3000])


In [ ]:
!python 17_rineng_analyses.py

## Step 22 – Results in Engineering article: identifiability vs transfer (Fig. 7)

In [ ]:
%%writefile 19_rineng_rq4.py
"""Step 19 - RQ4 of the Results in Engineering article: does single-record identifiability (TRACE class-I share of a
feature family in a cell) predict that family's cross-cell transfer value (leave-one-cell-out error reduction)?
Needs outputs of step 11 (out_ext_audit.csv) and step 17 (rineng_family_loco*.csv).
Outputs: outputs/rineng_rq4.json, outputs/rineng_rq4_pairs.csv, figures/rineng_fig6_identifiability_vs_transfer.png."""
import json, numpy as np, pandas as pd, matplotlib, matplotlib.ticker
from scipy import stats
matplotlib.use('Agg'); import matplotlib.pyplot as plt
O, FG = 'outputs/', 'figures/'
DSC = {'XJTU': '#1f5fbf', 'TJU': '#d9541e', 'HUST': '#138a5e', 'MIT': '#c2477d'}
MK = {'CC phase': 'o', 'CV phase': 's', 'voltage statistics': '^', 'current statistics': 'D'}
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 12, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.color': '#dddddd', 'savefig.dpi': 300, 'savefig.bbox': 'tight', 'legend.frameon': False})
FAM = {'CC phase': ['CC Q', 'CC charge time'], 'CV phase': ['CV Q', 'CV charge time'],
       'voltage statistics': ['voltage mean', 'voltage std', 'voltage kurtosis', 'voltage skewness', 'voltage slope', 'voltage entropy'],
       'current statistics': ['current mean', 'current std', 'current kurtosis', 'current skewness', 'current slope', 'current entropy']}
fo = {f: k for k, v in FAM.items() for f in v}
E = pd.read_csv(O + 'out_ext_audit.csv'); L = pd.read_csv(O + 'rineng_family_loco.csv')
S = pd.read_csv(O + 'rineng_family_loco_summary.csv'); A = pd.read_csv(O + 'rineng_atlas_family.csv')
E['fam'] = E.feature.map(fo); E['I'] = (E['class'] == 'I').astype(int)
ci = E.groupby(['cell', 'fam']).agg(Ish=('I', 'mean')).reset_index()
base = L[L.set == 'index only'][['cell', 'rmse_pct']].rename(columns={'rmse_pct': 'b'})
X = L[L.set != 'index only'].copy(); X['fam'] = X.set.str.replace('index + ', '', regex=False)
X = X.merge(base, on='cell').merge(ci, on=['cell', 'fam']); X['log_gain'] = np.log(X.b / X.rmse_pct)
X.to_csv(O + 'rineng_rq4_pairs.csv', index=False)
rng = np.random.default_rng(0)
def rho_ci(g):
    r = stats.spearmanr(g.Ish, g.log_gain); bs = []
    cells = g.cell.unique()
    for _ in range(1000):
        s = rng.choice(cells, len(cells)); gg = pd.concat([g[g.cell == c] for c in s]) if False else g.set_index('cell').loc[s]
        bs.append(stats.spearmanr(gg.Ish, gg.log_gain)[0])
    return float(r[0]), float(r[1]), float(np.nanpercentile(bs, 2.5)), float(np.nanpercentile(bs, 97.5))
R = {'pairs': len(X), 'cells': int(X.cell.nunique()), 'pooled': rho_ci(X), 'per_dataset': {ds: rho_ci(g) for ds, g in X.groupby('dataset')}}
agg = []
for ds in DSC:
    b = S[(S.dataset == ds) & (S.set == 'index only')]['median'].item()
    for f in FAM:
        m = S[(S.dataset == ds) & (S.set == 'index + ' + f)]['median'].item()
        agg.append(dict(dataset=ds, family=f, I_share=A[A.family == f][ds].item(), fold=b / m,
                        improved=float((X[(X.dataset == ds) & (X.fam == f)].log_gain > 0).mean())))
G = pd.DataFrame(agg); r16 = stats.spearmanr(G.I_share, G.fold)
R['dataset_family'] = dict(rho=float(r16[0]), p=float(r16[1]), rows=G.round(3).to_dict('records'))
json.dump(R, open(O + 'rineng_rq4.json', 'w'), indent=1); print(json.dumps({k: R[k] for k in ['pairs', 'pooled', 'per_dataset']}, indent=1)); print(G.round(2))

fig, (a, b) = plt.subplots(1, 2, figsize=(12.5, 4.8), gridspec_kw=dict(width_ratios=[1.25, 1], wspace=0.3))
for r in G.itertuples():
    a.scatter(r.I_share, r.fold, s=90, color=DSC[r.dataset], marker=MK[r.family], edgecolor='white', lw=0.8, zorder=3)
a.set_yscale('log'); a.set_ylim(0.8, 9); a.set_yticks([1, 2, 4, 8]); a.set_yticklabels(['1×', '2×', '4×', '8×']); a.yaxis.set_minor_formatter(matplotlib.ticker.NullFormatter()); a.axhline(1, color='#555555', lw=1)
a.set_xlabel('share of family features classed I (single record)'); a.set_ylabel('cross-cell error reduction vs index only')
a.text(0.03, 0.97, f'Spearman ρ = {r16[0]:.2f} (p = {r16[1]:.2f})\n16 dataset–family pairs', transform=a.transAxes, ha='left', va='top', fontsize=11)
h1 = [plt.Line2D([], [], ls='', marker='o', color=c, ms=9, label=d) for d, c in DSC.items()]
h2 = [plt.Line2D([], [], ls='', marker=m, color='#555555', ms=9, label=f) for f, m in MK.items()]
a.legend(handles=h1 + h2, loc='upper right', bbox_to_anchor=(1.0, 0.64), fontsize=10, ncol=2); a.set_title('(a)', loc='left', fontsize=15, weight='bold')
names = list(DSC) + ['pooled']; vals = [R['per_dataset'][d] for d in DSC] + [R['pooled']]
for i, (n, v) in enumerate(zip(names, vals)):
    b.errorbar(i, v[0], yerr=[[v[0] - v[2]], [v[3] - v[0]]], fmt='o', ms=9, color=DSC.get(n, '#111111'), capsize=4, lw=2)
b.axhline(0, color='#555555', lw=1); b.set_xticks(range(len(names))); b.set_xticklabels(names)
b.set_ylabel('Spearman ρ, cell-level I share vs gain'); b.set_ylim(-0.6, 0.6); b.set_title('(b)', loc='left', fontsize=15, weight='bold')
fig.savefig(FG + 'rineng_fig6_identifiability_vs_transfer.png'); print('saved fig6')


In [ ]:
!python 19_rineng_rq4.py

## Step 23 – Results in Engineering article: robustness (training horizon, equal-size families, random forest)

In [ ]:
%%writefile 20_rineng_robustness.py
"""Step 20 - robustness analyses of the Results in Engineering article.
(a) Index-only ceiling: sensitivity of R2_index to the training fraction (60 / 70 / 80 % of cycles).
(b) Transfer: equal-size control (every family reduced to two features: CC Q + CC time, CV Q + CV time,
    voltage mean + std, current mean + std) and a second learner (random forest) for all input sets.
(c) NASA engineering reading: data for Fig. 3 (restatement of capacity by discharge time; beyond-index co-fluctuation of V600).
Outputs: outputs/rineng_robust_ceiling.csv, rineng_robust_transfer.csv, rineng_robust_transfer_summary.csv, rineng_robust.json."""
import json, os, sys, warnings, numpy as np, pandas as pd
warnings.filterwarnings('ignore'); sys.path.insert(0, '.')
import trace_audit as T
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from joblib import Parallel, delayed
O = 'outputs/'; QUICK = os.environ.get('TRACE_QUICK') == '1'
ns = {}; exec(open('11_external.py').read().split("\nif __name__")[0].split("\nD = load()")[0], ns)
E = pd.read_csv(O + 'out_ext_audit.csv'); cells = E.drop_duplicates('cell')[['cell', 'dataset']]
D = ns['load'](); D = D[D.cell.isin(cells.cell)]
R = {}

# (a) ceiling vs training fraction
rows = []
for c, g in D.groupby('cell'):
    k = g.k.values.astype(float); y = g.capacity.values
    for f in (0.6, 0.7, 0.8):
        rows.append(dict(cell=c, dataset=g.dataset.iloc[0], n=len(g), train_frac=f, R2_index=T.index_ceiling(k, y, train_frac=f)['R2_index']))
Cc = pd.DataFrame(rows); Cc.to_csv(O + 'rineng_robust_ceiling.csv', index=False)
summ = Cc.groupby(['train_frac', 'dataset']).R2_index.agg(median='median', ge09=lambda v: (v >= 0.9).mean(), lt0=lambda v: (v < 0).mean()).reset_index()
alls = Cc.groupby('train_frac').R2_index.agg(median='median', ge09=lambda v: (v >= 0.9).mean(), lt0=lambda v: (v < 0).mean()).reset_index(); alls['dataset'] = 'All'
R['ceiling_vs_frac'] = pd.concat([summ, alls]).round(3).to_dict('records')
w = Cc.pivot_table(index='cell', columns='train_frac', values='R2_index')
from scipy import stats
R['ceiling_rank_agreement'] = {'rho_60_70': float(stats.spearmanr(w[0.6], w[0.7])[0]), 'rho_70_80': float(stats.spearmanr(w[0.7], w[0.8])[0])}
print(pd.concat([summ, alls]).round(2).to_string(index=False))

# (b) equal-size control and second learner
SETS = {'index only': [], 'index + CC phase': ['CC Q', 'CC charge time'], 'index + CV phase': ['CV Q', 'CV charge time'],
        'index + voltage mean, std': ['voltage mean', 'voltage std'], 'index + current mean, std': ['current mean', 'current std'],
        'index + voltage statistics': ['voltage mean', 'voltage std', 'voltage kurtosis', 'voltage skewness', 'voltage slope', 'voltage entropy'],
        'index + current statistics': ['current mean', 'current std', 'current kurtosis', 'current skewness', 'current slope', 'current entropy']}
def one(ds, dd, te, cl):
    tr = dd[dd.cell.isin([c for c in cl if c != te])].iloc[::3]; ts = dd[dd.cell == te]; yv = ts.capacity.values; out = []
    for nm, fs in SETS.items():
        X = ['k'] + fs
        for lrn, m in (('GBR', HistGradientBoostingRegressor(max_iter=150, random_state=0)),
                       ('RF', RandomForestRegressor(n_estimators=60, min_samples_leaf=5, max_features=0.6, max_samples=0.5, n_jobs=1, random_state=0))):
            if lrn == 'GBR' and nm in ('index only', 'index + CC phase', 'index + CV phase', 'index + voltage statistics', 'index + current statistics'):
                continue   # already in rineng_family_loco.csv (identical settings)
            p = m.fit(tr[X], tr.capacity).predict(ts[X])
            out.append(dict(dataset=ds, cell=te, set=nm, learner=lrn, rmse_pct=float(100 * np.sqrt(np.mean((p - yv) ** 2)) / yv[:5].mean())))
    return out
jobs = []
for ds, dd in D.groupby('dataset'):
    cl = dd.cell.unique()
    if QUICK: cl = cl[:10]
    jobs += [(ds, dd, te, cl) for te in cl]
res = Parallel(n_jobs=-1, verbose=0)(delayed(one)(*j) for j in jobs)
B = pd.DataFrame([r for rr in res for r in rr])
L = pd.read_csv(O + 'rineng_family_loco.csv').assign(learner='GBR')
B = pd.concat([B, L[['dataset', 'cell', 'set', 'learner', 'rmse_pct']]], ignore_index=True); B.to_csv(O + 'rineng_robust_transfer.csv', index=False)
rng = np.random.default_rng(0); S = []
for (ds, s, l), g in B.groupby(['dataset', 'set', 'learner']):
    v = g.rmse_pct.values; bs = [np.median(rng.choice(v, len(v))) for _ in range(1000)]
    S.append(dict(dataset=ds, set=s, learner=l, cells=len(v), median=float(np.median(v)), lo=float(np.percentile(bs, 2.5)), hi=float(np.percentile(bs, 97.5))))
S = pd.DataFrame(S); S.to_csv(O + 'rineng_robust_transfer_summary.csv', index=False)
print(S.pivot_table(index=['learner', 'set'], columns='dataset', values='median').round(2))
# does the RF reproduce the GBR ranking of families? Spearman over the 4 families x 4 datasets of fold gains
fam = ['index + CC phase', 'index + CV phase', 'index + voltage statistics', 'index + current statistics']
P = S.pivot_table(index=['dataset', 'set'], columns='learner', values='median').reset_index()
base = P[P.set == 'index only'].set_index('dataset')
P = P[P.set.isin(fam)].copy(); P['gain_GBR'] = P.apply(lambda r: base.loc[r.dataset, 'GBR'] / r.GBR, axis=1); P['gain_RF'] = P.apply(lambda r: base.loc[r.dataset, 'RF'] / r.RF, axis=1)
R['learner_agreement'] = dict(rho=float(stats.spearmanr(P.gain_GBR, P.gain_RF)[0]), p=float(stats.spearmanr(P.gain_GBR, P.gain_RF)[1]),
                              best_family_RF={ds: g.loc[g.gain_RF.idxmax(), 'set'] for ds, g in P.groupby('dataset')})
# identifiability vs RF gain at the cell level (same test as step 19)
fo = {'CC Q': 'CC phase', 'CC charge time': 'CC phase', 'CV Q': 'CV phase', 'CV charge time': 'CV phase'}
for f in SETS['index + voltage statistics']: fo[f] = 'voltage statistics'
for f in SETS['index + current statistics']: fo[f] = 'current statistics'
E['fam'] = E.feature.map(fo); E['I'] = (E['class'] == 'I').astype(int); ci = E.groupby(['cell', 'fam']).I.mean().rename('Ish').reset_index()
rf = B[B.learner == 'RF']; b0 = rf[rf.set == 'index only'][['cell', 'rmse_pct']].rename(columns={'rmse_pct': 'b'})
X = rf[rf.set.isin(fam)].copy(); X['fam'] = X.set.str.replace('index + ', '', regex=False); X = X.merge(b0, on='cell').merge(ci, on=['cell', 'fam'])
X['lg'] = np.log(X.b / X.rmse_pct); r = stats.spearmanr(X.Ish, X.lg)
R['rf_identifiability_vs_gain'] = dict(rho=float(r[0]), p=float(r[1]), pairs=len(X))
json.dump(R, open(O + 'rineng_robust.json', 'w'), indent=1, default=float); print(json.dumps({k: R[k] for k in ['ceiling_rank_agreement', 'learner_agreement', 'rf_identifiability_vs_gain']}, indent=1))


In [ ]:
!python 20_rineng_robustness.py

## Step 24 – Results in Engineering article: NASA engineering reading (Fig. 3) and 3-D landscape (Fig. 5)

In [ ]:
%%writefile 21_rineng_extra_figures.py
"""Step 21 - additional figures of the Results in Engineering article.
Fig. 3  NASA engineering reading: (a) capacity vs charge counted from discharge time (restatement, class P);
        (b) cross-fitted residuals of V600 vs residuals of capacity (co-fluctuation beyond the index, class I);
        (c) cross-fitted residuals of mean current vs capacity (instrumentation check).
Fig. 5  3-D landscape of the 374 public cells: record length x index-only ceiling x number of class-I features.
Needs data/nasa_cycle_features.csv and outputs of steps 01, 11."""
import numpy as np, pandas as pd, matplotlib, sys
sys.path.insert(0, '.'); matplotlib.use('Agg')
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from scipy import stats
import trace_audit as T
FG, O = 'figures/', 'outputs/'
DSC = {'XJTU': '#1f5fbf', 'TJU': '#d9541e', 'HUST': '#138a5e', 'MIT': '#c2477d'}
BC = {5: '#1f5fbf', 6: '#d9541e', 7: '#138a5e', 18: '#c2477d'}
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 12, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.color': '#dddddd', 'savefig.dpi': 300, 'savefig.bbox': 'tight', 'legend.frameon': False})
lab = lambda a, t: a.set_title(t, loc='left', fontsize=15, weight='bold')

# ---------------- Fig. 3 NASA engineering reading
N = pd.read_csv('data/nasa_cycle_features.csv')
fig, ax = plt.subplots(1, 3, figsize=(15, 4.6), gridspec_kw=dict(wspace=0.32))
a = ax[0]
for b, g in N.groupby('battery'):
    q = np.abs(g.I_mean) * g.t_discharge / 3600
    a.scatter(q, g.capacity, s=12, color=BC[b], label=f'B{b:04d}', alpha=0.8)
lim = [1.25, 2.1]; a.plot(lim, lim, color='#111111', lw=1, ls='--'); a.set_xlim(lim); a.set_ylim(lim)
a.set_xlabel('|I| × discharge time (Ah)'); a.set_ylabel('measured capacity (Ah)'); a.legend(loc='upper left', fontsize=10)
r = stats.pearsonr(np.abs(N.I_mean) * N.t_discharge / 3600, N.capacity)[0]
a.text(0.97, 0.05, f'r = {r:.4f}\nclass P in 4/4 cells', transform=a.transAxes, ha='right', fontsize=11); lab(a, '(a)')
rows = []
for b, g in N.groupby('battery'):
    k = g.cycle.values.astype(float)
    ey = T.crossfit_trend(k, g.capacity.values)[1]
    for f in ['V_600s', 'I_mean']:
        ex = T.crossfit_trend(k, g[f].values)[1]
        rows.append(dict(battery=b, feature=f, ey=ey, ex=ex))
for i, (f, t, nm) in enumerate([('V_600s', '(b)', 'V$_{600}$ residual (z)'), ('I_mean', '(c)', 'mean-current residual (z)')]):
    a = ax[i + 1]
    for rr in rows:
        if rr['feature'] != f: continue
        zx = (rr['ex'] - rr['ex'].mean()) / rr['ex'].std(); zy = (rr['ey'] - rr['ey'].mean()) / rr['ey'].std()
        a.scatter(zx, zy, s=12, color=BC[rr['battery']], alpha=0.75)
        rows_rho = stats.spearmanr(zx, zy)[0]; rr['rho'] = rows_rho
    rhos = ', '.join(f"{rr['rho']:.2f}" for rr in rows if rr['feature'] == f)
    a.set_xlabel(nm); a.set_ylabel('capacity residual (z)'); a.set_xlim(-4.5, 4.5); a.set_ylim(-4.5, 4.5)
    a.text(0.03, 0.97, f'Spearman ρ by cell:\n{rhos}', transform=a.transAxes, va='top', fontsize=10.5); lab(a, t)
fig.savefig(FG + 'rineng_fig3_nasa_reading.png'); plt.close(fig)
pd.DataFrame([dict(battery=r['battery'], feature=r['feature'], rho=r['rho']) for r in rows]).to_csv(O + 'rineng_nasa_residual_rho.csv', index=False)
print('saved Fig. 3; residual rho:', [(r['battery'], r['feature'], round(r['rho'], 2)) for r in rows])

# ---------------- Fig. 5 3-D landscape
E = pd.read_csv(O + 'out_ext_audit.csv'); E['I'] = (E['class'] == 'I')
C = E.groupby('cell').agg(n=('n', 'first'), nI=('I', 'sum'), R2=('R2_index', 'first'), ds=('dataset', 'first')).reset_index()
C['R2c'] = C.R2.clip(-1, 1)
fig = plt.figure(figsize=(10.5, 8)); a = fig.add_subplot(111, projection='3d')
for ds in ['XJTU', 'TJU', 'HUST', 'MIT']:
    g = C[C.ds == ds]
    a.scatter(np.log10(g.n), g.R2c, g.nI, s=34, color=DSC[ds], label=f'{ds} ({len(g)} cells)', depthshade=True, edgecolor='white', linewidth=0.3)
# projected shadows on the floor
a.scatter(np.log10(C.n), C.R2c, np.zeros(len(C)) - 0.3, s=6, color='#bbbbbb', alpha=0.5, depthshade=False)
zt = C.nI.max() + 1
xx, yy = np.meshgrid([np.log10(100.0), np.log10(300.0)], [-1, 1]); a.plot_surface(xx, yy, np.full_like(xx, -0.3), color='#d9541e', alpha=0.18, linewidth=0)
a.text2D(0.03, 0.70, 'shaded floor: records < 300 cycles,\nno class-I verdict in any of 79 cells', transform=a.transAxes, fontsize=11, color='#8a2e0b')
a.set_xticks(np.log10([100, 300, 1000, 3000])); a.set_xticklabels(['100', '300', '1000', '3000'])
a.set_xlabel('record length (cycles)', labelpad=10); a.set_ylabel('index-only R² (clipped to ±1)', labelpad=10); a.set_zlabel('class-I features per cell', labelpad=8)
a.set_zlim(-0.3, zt); a.view_init(elev=22, azim=-58); a.legend(loc='upper left', fontsize=11, bbox_to_anchor=(0.02, 0.92))
for p in (a.xaxis.pane, a.yaxis.pane, a.zaxis.pane): p.set_facecolor((0.97, 0.97, 0.97, 1)); p.set_edgecolor('#cccccc')
fig.savefig(FG + 'rineng_fig5_landscape3d.png'); plt.close(fig)
r1 = stats.spearmanr(C.n, C.nI); r2 = stats.spearmanr(C.R2, C.nI); r3 = stats.spearmanr(C.n, C.R2)
print('3D: rho(n,nI)=%.2f rho(R2,nI)=%.2f p=%.3g rho(n,R2)=%.2f p=%.3g' % (r1[0], r2[0], r2[1], r3[0], r3[1]))
pd.Series(dict(rho_n_nI=r1[0], rho_R2_nI=r2[0], p_R2_nI=r2[1], rho_n_R2=r3[0], p_n_R2=r3[1])).to_csv(O + 'rineng_landscape_rho.csv')


In [ ]:
!python 21_rineng_extra_figures.py

## Step 25 – Results in Engineering article: SHAP explainability and explanation reliability (Fig. 8)

In [ ]:
%%writefile 22_rineng_explain.py
"""Step 22 - explainability and explanation reliability (Results in Engineering article).
(A) Single-record explanations: for each of the 374 public cells a model is fitted within the record with the common
    random 80/20 row split (index k + 16 charge features -> capacity), and explained with TreeSHAP.
    Measured: random-split R2 (the score such a study would report), share of mean |SHAP| given to the index and to
    features of each TRACE class, whether the top-ranked feature is one the audit can credit (class I),
    and explanation stability (Spearman agreement of the feature ranking between two random splits).
(B) Cross-cell explanations: within each dataset, leave-one-cell-out models on index + 16 features, TreeSHAP on each
    held-out cell; reliability of the feature ranking across folds (Kendall's W) and its agreement with the
    single-record class-I share of each feature.
Outputs: outputs/rineng_xai_within.csv, rineng_xai_cross.csv, rineng_xai.json; figures/rineng_fig8_explainability.png."""
import json, os, sys, warnings, numpy as np, pandas as pd, matplotlib, shap
warnings.filterwarnings('ignore'); sys.path.insert(0, '.'); matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import r2_score
from joblib import Parallel, delayed
O, FG = 'outputs/', 'figures/'; QUICK = os.environ.get('TRACE_QUICK') == '1'
ns = {}; exec(open('11_external.py').read().split("\nif __name__")[0].split("\nD = load()")[0], ns); F = ns['F']
E = pd.read_csv(O + 'out_ext_audit.csv'); D = ns['load'](); D = D[D.cell.isin(E.cell.unique())]
CLS = E.set_index(['cell', 'feature'])['class']
DSC = {'XJTU': '#1f5fbf', 'TJU': '#d9541e', 'HUST': '#138a5e', 'MIT': '#c2477d'}
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 12, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.color': '#dddddd', 'savefig.dpi': 300, 'savefig.bbox': 'tight', 'legend.frameon': False})
X_COLS = ['k'] + F


def within(cell, g):
    out = {}
    imps = []
    for seed in (0, 1):
        rng = np.random.default_rng(seed); idx = rng.permutation(len(g)); cut = int(0.8 * len(g))
        tr, te = g.iloc[idx[:cut]], g.iloc[idx[cut:]]
        m = HistGradientBoostingRegressor(max_iter=150, random_state=0).fit(tr[X_COLS], tr.capacity)
        sv = shap.TreeExplainer(m).shap_values(te[X_COLS]); imp = np.abs(sv).mean(0); imps.append(imp)
        if seed == 0: out['R2_random_split'] = r2_score(te.capacity, m.predict(te[X_COLS]))
    imp = imps[0]; tot = imp.sum()
    cls = np.array(['index'] + [CLS[(cell, f)] for f in F])
    for c in ['index', 'I', 'T', 'N', 'P']: out[f'share_{c}'] = float(imp[cls == c].sum() / tot)
    fi = imp[1:]; top = F[int(np.argmax(fi))]
    out.update(cell=cell, dataset=g.dataset.iloc[0], n=len(g), top_feature=top, top_class=CLS[(cell, top)],
               n_I=int((cls[1:] == 'I').sum()), stability=float(stats.spearmanr(imps[0][1:], imps[1][1:])[0]),
               index_rank=int((imp > imp[0]).sum()) + 1)
    return out


cells = list(D.groupby('cell'))
if QUICK: cells = cells[:30]
FIG_ONLY = os.environ.get('FIG_ONLY') == '1'
if FIG_ONLY: W = pd.read_csv(O + 'rineng_xai_within.csv')
else: W = pd.DataFrame(Parallel(n_jobs=-1)(delayed(within)(c, g) for c, g in cells)); W.to_csv(O + 'rineng_xai_within.csv', index=False)


def cross_fold(ds, dd, te, cl):
    tr = dd[dd.cell.isin([c for c in cl if c != te])].iloc[::3]; ts = dd[dd.cell == te]
    if len(ts) > 300: ts = ts.iloc[np.linspace(0, len(ts) - 1, 300).astype(int)]
    m = HistGradientBoostingRegressor(max_iter=150, random_state=0).fit(tr[X_COLS], tr.capacity)
    sv = np.abs(shap.TreeExplainer(m).shap_values(ts[X_COLS])).mean(0)
    return dict(dataset=ds, cell=te, **{c: float(v) for c, v in zip(X_COLS, sv)})


jobs = []
for ds, dd in D.groupby('dataset'):
    cl = dd.cell.unique()
    if QUICK: cl = cl[:8]
    jobs += [(ds, dd, te, cl) for te in cl]
if FIG_ONLY: X = pd.read_csv(O + 'rineng_xai_cross.csv')
else: X = pd.DataFrame(Parallel(n_jobs=-1)(delayed(cross_fold)(*j) for j in jobs)); X.to_csv(O + 'rineng_xai_cross.csv', index=False)


def kendall_w(M):  # rows = folds (raters), cols = features
    R = np.apply_along_axis(stats.rankdata, 1, M); m, n = R.shape
    S = ((R.sum(0) - m * (n + 1) / 2) ** 2).sum(); return float(12 * S / (m ** 2 * (n ** 3 - n)))


Ish = E.assign(I=(E['class'] == 'I')).groupby(['dataset', 'feature']).I.mean()
res = {'within': {}, 'cross': {}}
for ds, g in W.groupby('dataset'):
    res['within'][ds] = dict(cells=len(g), R2_random_median=float(g.R2_random_split.median()), share_index=float(g.share_index.median()),
                             share_I=float(g.share_I.mean()), share_T=float(g.share_T.mean()), share_N=float(g.share_N.mean()),
                             top_is_I=float((g.top_class == 'I').mean()), top_is_T=float((g.top_class == 'T').mean()),
                             stability_median=float(g.stability.median()))
res['within']['All'] = dict(cells=len(W), R2_random_median=float(W.R2_random_split.median()), share_index=float(W.share_index.median()),
                            share_I=float(W.share_I.mean()), share_T=float(W.share_T.mean()), share_N=float(W.share_N.mean()),
                            top_is_I=float((W.top_class == 'I').mean()), top_is_T=float((W.top_class == 'T').mean()),
                            stability_median=float(W.stability.median()), R2_ge_0_99=float((W.R2_random_split >= 0.99).mean()),
                            stability_vs_topI=float(stats.mannwhitneyu(W[W.top_class == 'I'].stability, W[W.top_class != 'I'].stability)[1]))
for ds, g in X.groupby('dataset'):
    M = g[F].values; mean_imp = g[F].mean(); rk = mean_imp.sort_values(ascending=False)
    r = stats.spearmanr(mean_imp.values, Ish.loc[ds].reindex(F).values)
    res['cross'][ds] = dict(folds=len(g), kendall_W=kendall_w(M), top3=list(rk.index[:3]), index_share=float((g['k'] / g[X_COLS].sum(1)).median()),
                            rho_importance_vs_Ishare=float(r[0]), p=float(r[1]))
json.dump(res, open(O + 'rineng_xai.json', 'w'), indent=1); print(json.dumps(res, indent=1))

# ---------------- figure
fig, ax = plt.subplots(1, 3, figsize=(17, 5.2), gridspec_kw=dict(width_ratios=[1.15, 1, 1.25], wspace=0.38))
a = ax[0]; order = ['XJTU', 'TJU', 'HUST', 'MIT']; cols = {'index': '#111111', 'I': '#138a5e', 'T': '#8c8c86', 'N': '#c98a00'}
bottom = np.zeros(4)
for c in ['index', 'I', 'T', 'N']:
    v = np.array([W[W.dataset == d][f'share_{c}'].mean() for d in order]); a.bar(order, 100 * v, bottom=100 * bottom, color=cols[c], label=c if c != 'index' else 'cycle index', width=0.6)
    bottom += v
a.set_ylabel('share of mean |SHAP| (%)'); a.set_ylim(0, 118); a.legend(ncol=4, loc='upper center', fontsize=10.5, bbox_to_anchor=(0.5, 1.02), handlelength=1)
a.set_title('(a)', loc='left', fontsize=15, weight='bold')
a = ax[1]
for d in order:
    g = W[W.dataset == d]; a.scatter(g.stability, g.R2_random_split.clip(0.9, 1), s=16, color=DSC[d], alpha=0.75, label=d)
a.set_xlabel('explanation stability between two splits (ρ)'); a.set_ylabel('random-split R² (clipped at 0.9)'); a.legend(fontsize=10, loc='lower left')
a.set_title('(b)', loc='left', fontsize=15, weight='bold')
a = ax[2]
for d in order:
    g = X[X.dataset == d]; mi = g[F].mean(); a.scatter(Ish.loc[d].reindex(F).values, mi.values / mi.max(), s=40, color=DSC[d], label=f"{d}: ρ = {res['cross'][d]['rho_importance_vs_Ishare']:.2f}, W = {res['cross'][d]['kendall_W']:.2f}")
a.set_xlabel('single-record class-I share of feature'); a.set_ylabel('cross-cell mean |SHAP| (relative)'); a.legend(fontsize=9.5, loc='upper right', bbox_to_anchor=(1.0, 0.88))
a.set_title('(c)', loc='left', fontsize=15, weight='bold')
fig.savefig(FG + 'rineng_fig8_explainability.png'); print('saved fig8')


In [ ]:
!python 22_rineng_explain.py

## Step 26 – Results in Engineering article: methodology figure (Fig. 1)

In [ ]:
%%writefile 18_rineng_fig1.py
"""Step 18 - Fig. 1 of the Results in Engineering article: methodology / study design.
Four zones built from the real data and results: (1) data, (2) audit of each record (TRACE, co-submitted),
(3) analysis steps, (4) verified outcomes and the screening checklist. Run after steps 11, 17 and 19."""
import json, sys, numpy as np, pandas as pd, matplotlib
sys.path.insert(0, '.'); matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
import trace_audit as T
O, FG = 'outputs/', 'figures/'
BLUE, ORANGE, GREEN, PINK, GREY, INK, AMBER, PURPLE = '#1f5fbf', '#d9541e', '#138a5e', '#c2477d', '#8c8c86', '#111111', '#c98a00', '#6a3d9a'
DSC = {'XJTU': BLUE, 'TJU': ORANGE, 'HUST': GREEN, 'MIT': PINK}
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 9, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.color': '#e6e6e6', 'savefig.dpi': 300, 'legend.frameon': False})
W, H = 18, 10.2
fig = plt.figure(figsize=(W, H))
bg = fig.add_axes([0, 0, 1, 1]); bg.set_xlim(0, W); bg.set_ylim(0, H); bg.axis('off')


def zone(x, w, col, title):
    bg.add_patch(FancyBboxPatch((x, 0.75), w, 8.25, boxstyle='round,pad=0.02,rounding_size=0.18', fc='white', ec=col, lw=2.5))
    bg.text(x + w / 2, 8.68, title, ha='center', va='center', fontsize=14, weight='bold', color=INK)


def ax_at(x, y, w, h, **kw):
    return fig.add_axes([x / W, y / H, w / W, h / H], **kw)


def tag(a, t, fs=9.5):
    a.set_title(t, loc='left', fontsize=fs, weight='bold')


def arrow(x0, x1, y, txt):
    bg.add_patch(FancyArrowPatch((x0, y), (x1, y), arrowstyle='-|>', mutation_scale=22, lw=2.2, color=GREY))
    bg.text((x0 + x1) / 2, y + 0.18, txt, ha='center', fontsize=9, color='#555555')


# banner
bg.add_patch(FancyBboxPatch((0.2, 9.25), W - 0.4, 0.75, boxstyle='round,pad=0.02,rounding_size=0.15', fc='#fbf1e4', ec=AMBER, lw=1.5))
bg.text(W / 2, 9.62, 'What can one battery ageing record support?  Index ceiling · provenance · identifiable features · cross-cell transfer · explanation reliability',
        ha='center', va='center', fontsize=14, weight='bold')
zx = [(0.2, 3.7, GREEN, '1  Data (Step 1)'), (4.35, 4.15, BLUE, '2  Audit of each record (Step 2)'),
      (8.95, 4.6, PURPLE, '3  Analyses (Steps 3–8)'), (14.0, 3.8, ORANGE, '4  Verified outcomes')]
for z in zx: zone(*z)
arrow(3.92, 4.33, 4.9, ''); arrow(8.52, 8.93, 4.9, ''); arrow(13.57, 13.98, 4.9, '')
bg.text(W / 2, 0.38, 'one row per cycle  →  remove restatements  →  report the index ceiling  →  check record length  →  test transfer on other cells'
        '     |     reproducible Python / Google Colab code', ha='center', fontsize=11, color=PURPLE, weight='bold')

# ---------- zone 1 data
g = pd.read_csv('data/GMP.csv'); a = ax_at(0.75, 6.25, 2.95, 1.75)
a.plot(g.CycleNumber, g['Capacity(Ah)'], color=BLUE, lw=2); a.set_xlabel('cycle', labelpad=1); a.set_ylabel('capacity (Ah)')
tag(a, '(a) GMP LiFePO$_4$ pack, 25.6 V 40 Ah'); a.text(0.04, 0.08, '1 record, 3000 cycles\ngenerator-reproducible', transform=a.transAxes, fontsize=8.5)
N = pd.read_csv('data/nasa_cycle_features.csv'); a = ax_at(0.75, 3.55, 2.95, 1.75)
for b, gg in N.groupby('battery'): a.plot(gg.cycle, gg.capacity, lw=1.3, label=f'B{b:04d}')
a.legend(fontsize=7.5, ncol=2, loc='upper right'); a.set_xlabel('cycle', labelpad=1); a.set_ylabel('capacity (Ah)')
tag(a, '(b) NASA cells: measured, noisy'); a.set_ylim(1.2, 2.25)
ns = {}; exec(open('11_external.py').read().split("\nif __name__")[0].split("\nD = load()")[0], ns)
E = pd.read_csv(O + 'out_ext_audit.csv'); D = ns['load'](); D = D[D.cell.isin(E.cell.unique())]
a = ax_at(0.75, 1.15, 2.95, 1.6); rng = np.random.default_rng(1)
for ds, col in DSC.items():
    cs = D[D.dataset == ds].cell.unique(); pick = rng.choice(cs, min(8, len(cs)), replace=False)
    for c in pick:
        gg = D[D.cell == c]; a.plot(gg.k, gg.capacity / gg.capacity.iloc[:5].mean(), color=col, lw=0.8, alpha=0.8)
a.set_xscale('log'); a.set_xticks([1, 10, 100, 1000]); a.set_xticklabels(['1', '10', '100', '1000']); a.set_xlabel('cycle', labelpad=1); a.set_ylabel('relative capacity'); a.set_ylim(0.6, 1.08)
tag(a, '(c) 374 public cells, 16 charge features')
for i, (ds, col) in enumerate(DSC.items()): a.text(0.03 + 0.24 * i, 0.06, ds, color=col, transform=a.transAxes, fontsize=8.5, weight='bold')

# ---------- zone 2 audit
b5 = N[N.battery == 5]; k = b5.cycle.values.astype(float)
ey = T.crossfit_trend(k, b5.capacity.values)[1]; ev = T.crossfit_trend(k, b5.V_600s.values)[1]
a = ax_at(4.85, 6.25, 3.4, 1.75)
a.plot(k, (ey - ey.mean()) / ey.std(), color=BLUE, lw=1.2, label='capacity residual'); a.plot(k, (ev - ev.mean()) / ev.std(), color=ORANGE, lw=1.0, label='V$_{600}$ residual')
a.legend(fontsize=7.5, loc='upper right'); a.set_xlabel('cycle', labelpad=1); a.set_ylabel('z')
tag(a, '(d) S3: co-fluctuation beyond the index (B0005)')
bg.text(4.6, 5.62, 'S1  provenance: D (index-derived), P (restated target)\nS2  index-only ceiling: power law, R²$_{index}$\n'
        'S3  time-shift test, Holm: I / T / N\nS4  self-surrogate power: MDE$_{80}$', fontsize=9.5, va='top', linespacing=1.5)
sh = E['class'].value_counts(normalize=True).reindex(['T', 'I', 'N', 'P']).fillna(0)
a = ax_at(4.85, 1.05, 3.4, 3.3)
cols = {'T': GREY, 'I': GREEN, 'N': AMBER, 'P': PINK}
a.bar(sh.index, 100 * sh.values, color=[cols[c] for c in sh.index], width=0.6)
for i, v in enumerate(sh.values): a.text(i, 100 * v + 1.5, f'{100 * v:.1f}%', ha='center', fontsize=9)
a.set_ylabel('% of 5984 verdicts'); a.set_ylim(0, 85); tag(a, '(e) audit card classes, public cells')
a.text(0.98, 0.62, 'method and validation:\nco-submitted MethodsX article', transform=a.transAxes, ha='right', fontsize=8.5, style='italic')

# ---------- zone 3 analyses
from scipy.optimize import curve_fit
kk = g.CycleNumber.values.astype(float); yy = g['Capacity(Ah)'].values
p, _ = curve_fit(lambda k, a_, b_, c_: a_ - b_ * k ** c_, kk[:500], yy[:500], p0=[yy[0], 1e-3, 1.2], maxfev=20000)
a = ax_at(9.45, 6.25, 1.75, 1.75); a.plot(kk, yy, color=INK, lw=2.5); a.plot(kk, p[0] - p[1] * kk ** p[2], color=ORANGE, ls='--', lw=1.5)
a.axvline(500, color=GREY, ls=':'); a.set_xticks([0, 1500, 3000]); tag(a, '(f) Step 3: index ceiling'); a.set_xlabel('cycle', labelpad=1)
a.text(0.05, 0.08, 'RUL error\n0.03 cycles', transform=a.transAxes, fontsize=8.5)
fam = pd.read_csv(O + 'rineng_atlas_family.csv').set_index('family').loc[['CC phase', 'CV phase', 'voltage statistics', 'current statistics'], ['XJTU', 'TJU', 'HUST', 'MIT']]
a = ax_at(11.85, 6.25, 1.5, 1.75); a.imshow(fam.values, cmap='Blues', vmin=0, vmax=0.7, aspect='auto'); a.grid(False)
a.set_xticks(range(4)); a.set_xticklabels(['XJ', 'TJ', 'HU', 'MI'], fontsize=8); a.set_yticks(range(4)); a.set_yticklabels(['CC', 'CV', 'V stats', 'I stats'], fontsize=8)
for i in range(4):
    for j in range(4): a.text(j, i, f'{fam.values[i, j]:.2f}', ha='center', va='center', fontsize=7, color='white' if fam.values[i, j] > 0.4 else INK)
tag(a, '(g) Step 5: atlas', 9.5)
C = E.groupby('cell').agg(n=('n', 'first'), nI=('class', lambda s: (s == 'I').sum()), R2=('R2_index', 'first'), ds=('dataset', 'first'))
a = ax_at(9.05, 2.55, 2.75, 3.1, projection='3d')
for ds, col in DSC.items():
    cc = C[C.ds == ds]; a.scatter(np.log10(cc.n), cc.R2.clip(-1, 1), cc.nI, s=5, color=col, depthshade=True)
a.set_xticks(np.log10([100, 1000])); a.set_xticklabels(['100', '1000'], fontsize=7); a.tick_params(labelsize=7, pad=0)
a.set_xlabel('cycles', fontsize=7.5, labelpad=-6); a.set_ylabel('R²$_{index}$', fontsize=7.5, labelpad=-6); a.set_zlabel('# I', fontsize=7.5, labelpad=-6)
a.set_zticks([0, 10]); a.set_yticks([-1, 0, 1]); a.view_init(elev=20, azim=-58); a.set_title('(h) Step 5: 3-D cell landscape', fontsize=9.5, weight='bold', loc='left')
S = pd.read_csv(O + 'rineng_family_loco_summary.csv'); a = ax_at(12.5, 3.0, 0.95, 2.25)
dss = ['XJTU', 'TJU', 'HUST', 'MIT']
for i, ds in enumerate(dss):
    v0 = S[(S.dataset == ds) & (S.set == 'index only')]['median'].item(); v1 = S[(S.dataset == ds) & (S.set == 'index + voltage statistics')]['median'].item()
    a.bar(i - 0.2, v0, 0.4, color=GREY); a.bar(i + 0.2, v1, 0.4, color=AMBER)
a.set_ylim(0, 4.8); a.set_xticks(range(4)); a.set_xticklabels(['XJ', 'TJ', 'HU', 'MI'], fontsize=8); a.set_ylabel('LOCO RMSE (%)', fontsize=8, labelpad=1)
tag(a, '(i) Step 6: transfer', 9.5); a.text(0.98, 0.92, 'index only', color=GREY, transform=a.transAxes, ha='right', fontsize=7.5, weight='bold')
a.text(0.98, 0.80, '+ voltage stats', color=AMBER, transform=a.transAxes, ha='right', fontsize=7.5, weight='bold')
bg.text(9.3, 2.2, 'Step 4: NASA verdicts → measurement mechanism\nStep 6: leave-one-cell-out GBR / RF, 4 families\n'
        'Step 7: identifiability vs transfer (1496 pairs)\nStep 8: SHAP explanations and their reliability', fontsize=9.3, va='top', linespacing=1.5)

# ---------- zone 4 outcomes
rq4 = json.load(open(O + 'rineng_rq4.json')); R = json.load(open(O + 'rineng_results.json')); X = json.load(open(O + 'rineng_xai.json'))['within']['All']
XAI_TXT = f"stable SHAP (ρ = {X['stability_median']:.2f}), yet top feature\nnot creditable in {100 * (1 - X['top_is_I']):.0f}% of records"
items = [(ORANGE, 'Index ceiling', f"GMP RUL error 0.03 cycles (LSTM 25.6)\nR²$_{{index}}$ ≥ 0.9 in {100 * R['public_R2_index_all']['share_ge_0_9']:.1f}% of cells"),
         (PINK, 'Restatements', 't$_{dis}$, t$_{Tmax}$ = class P in 4/4 NASA cells\nV$_{600}$ = class I in 4/4 cells'),
         (GREEN, 'Identifiable features', f"{100 * R['atlas']['I_share_all']:.1f}% of verdicts; CV phase highest\nnone in 79 records < 300 cycles"),
         (BLUE, 'Identifiability ≠ transfer', f"ρ = {rq4['pooled'][0]:.2f} (1496 pairs)\nvoltage stats: up to 6.2-fold gain"),
         (PURPLE, 'Explanations ≠ evidence', XAI_TXT)]
for i, (col, t, b) in enumerate(items):
    y0 = 7.5 - i * 1.13
    bg.add_patch(FancyBboxPatch((14.25, y0), 3.3, 0.98, boxstyle='round,pad=0.02,rounding_size=0.1', fc='white', ec=col, lw=1.8))
    bg.text(14.4, y0 + 0.76, t, fontsize=10.5, weight='bold', color=col, va='center')
    bg.text(14.4, y0 + 0.52, b, fontsize=8.5, va='top', linespacing=1.35)
bg.add_patch(FancyBboxPatch((14.25, 0.95), 3.3, 1.85, boxstyle='round,pad=0.02,rounding_size=0.1', fc='#f3f0f8', ec=PURPLE, lw=1.8))
bg.text(14.4, 2.6, 'Screening checklist (Section 5)', fontsize=10.5, weight='bold', color=PURPLE, va='center')
bg.text(14.4, 2.38, '1  remove restated columns\n2  report R²$_{index}$ beside every score\n3  require a measured target\n'
        '4  ≥ 300 cycles; report MDE$_{80}$\n5  test transfer on other cells', fontsize=9, va='top', linespacing=1.4)
fig.savefig(FG + 'rineng_fig1_study_design.png'); print('saved figures/rineng_fig1_study_design.png')


In [ ]:
!python 18_rineng_fig1.py

## Step 27 – view the figures and tables and key outputs

In [ ]:
from IPython.display import Image, display
import glob, json, pandas as pd
for f in sorted(glob.glob('figures/*.png')): print(f); display(Image(f, width=900))
print(open('outputs/results_numbers.md').read())
display(pd.read_csv('outputs/out_audit_nasa.csv')[['battery','feature','r_resid','S_cond','p_cond_holm','class']])

In [ ]:
!zip -qr TRACE_outputs.zip outputs figures && echo 'download TRACE_outputs.zip from the Files pane'